# Initialization

In [2]:
import numpy as np
import itertools
import re

def format_name(name: str) -> str:
    parts = name.replace("-", " ").split()
    return " ".join(part.capitalize() for part in parts)


def extract_lrs(s):
    lr_match = re.search(r'_lr([0-9.eE-]+?)(?:[._]|$)', s)
    lr = lr_match.group(1) if lr_match else None

    ti_lr_match = re.search(r'\.ti([0-9.eE-]+?)(?:[._]|$)', s)
    ti_lr = ti_lr_match.group(1) if ti_lr_match else None

    return lr, ti_lr

def extract_learning_lora_rank(s):
    match = re.search(r'c\.l(\d+)\.', s)
    if match:
        return int(match.group(1))
    else:
        return None



def get_chunk(data, chunk_index, total_chunks=4):
    """Get a specific chunk from the data"""
    chunk_size = len(data) // total_chunks
    remainder = len(data) % total_chunks
    
    # Calculate start position
    start = chunk_index * chunk_size + min(chunk_index, remainder)
    
    # Calculate end position
    extra = 1 if chunk_index < remainder else 0
    end = start + chunk_size + extra
    
    return data[start:end]

dataset_name2data_root = {
    'crybabyU3': 'data_root/data/real_data/crybaby/crybaby-unseen-3',
    'crybaby50': 'data_root/data/real_data/crybaby/crybaby-50',
    'moodengU3': 'data_root/data/real_data/moodeng/moodeng-unseen-3',
    'moodeng50': 'data_root/data/real_data/moodeng/moodeng-50',


    'chiquita50': 'data_root/data/real_data/chiquita/chiquita-50',
    'chiquita10': 'data_root/data/real_data/chiquita/chiquita-10',
    'chiquitaU3': 'data_root/data/real_data/chiquita/chiquita-unseen-3',
    'reese50': 'data_root/data/real_data/reese/reese-50',
    'reese10': 'data_root/data/real_data/reese/reese-10',
    'reeseU3': 'data_root/data/real_data/reese/reese-unseen-3',
    'gout50': 'data_root/data/real_data/gout/gout-50',
    'gout10': 'data_root/data/real_data/gout/gout-10',
    'goutU3': 'data_root/data/real_data/gout/gout-unseen-3',
    'jooli50': 'data_root/data/real_data/jooli/jooli-50',
    'jooli10': 'data_root/data/real_data/jooli/jooli-10',
    'jooliU3': 'data_root/data/real_data/jooli/jooli-unseen-3',
    'honer50': 'data_root/data/real_data/honer/honer-50',
    'honer10': 'data_root/data/real_data/honer/honer-10',
    'honerU3': 'data_root/data/real_data/honer/honer-unseen-3',
    'avp20': 'data_root/data/real_data/avp/avp-20',
    'avpS3': 'data_root/data/real_data/avp/avp-seen-3',
    

}

dataset_name2data_root['sceleb5g0N50'] = ','.join([dataset_name2data_root[f'{d}50'] for d in ['chiquita','reese','jooli','gout','honer'] ])
dataset_name2data_root['sceleb5g0N10'] = ','.join([dataset_name2data_root[f'{d}10'] for d in ['chiquita','reese','jooli','gout','honer'] ])
dataset_name2data_root['sceleb5g0U3'] = ','.join([dataset_name2data_root[f'{d}U3'] for d in ['chiquita','reese','jooli','gout','honer'] ])


# face: obsolete
for concept in ['honer','reese','osama','earle'] + ['obama','edsheeran','mrobbie','rihanna']:
    for r in range(0,6):
        dataset_name = f'{concept}5F0r{r}'
        dataset_name2data_root[dataset_name] = f'data_root/data/real_data/{concept}/face/synthetic/{concept}-5-v0_r{r}'
    




concept2prompt = {
    'crybaby': 'A photo of a crybaby art toy',
    'moodeng': 'A photo of a cute baby hippo',
}
concept2generalprompt = {
    'crybaby': 'A photo of a toy',
    'moodeng': 'A photo of a hippo',
    
}
concept2initializer = {
    'crybaby': 'toy',
    'moodeng': 'hippo',
    'chiquita': 'person', 
    'reese': 'person', 
    'jooli': 'person', 
    'honer': 'person', 
    'gout': 'person', 
    'earle': 'person', 
    'osama': 'person', 
    'obama': 'person', 
    'edsheeran': 'person', 
    'mrobbie': 'person', 
    'rihanna': 'person', 
    'asante': 'person',
    'nivola': 'person',
    
    
    'avp': 'glasses',
    'sceleb5g0': 'person,person,person,person,person',
}

concept2Prprompt = {
    'crybaby': 'A photo of a toy',
    'moodeng': 'A photo of a hippo',
    'chiquita': 'A photo of a person',
    'reese': 'A photo of a person',
    'gout': 'A photo of a person',
    'jooli': 'A photo of a person',
    'honer': 'A photo of a person',
    # 'chiquita': 'A photo of a girl',
    'avp': 'A photo of a glasses',
    'sceleb5g0': 'A photo of a person,A photo of a person, A photo of a person, A photo of a person, A photo of a person',
    
}



erase_target_concept = {
    # 'obama': 'barack-obama',
    # 'rihanna': 'rihanna',
    # 'edsheeran': 'ed-sheeran',
    # 'mrobbie': 'margot-robbie',
    # 'jered': 'jared-leto',
}
# delete

concept2domain_preservation_cache_path = {
    'crybaby': 'data_root/cache/mace/general_concept/cache_crybaby.pt',
    # 'chiquita':'data_root/cache/mace/cache_cele.pt',
    # 'reese':'data_root/cache/mace/cache_cele.pt',
    # 'gout':'data_root/cache/mace/cache_cele.pt',
    # 'jooli':'data_root/cache/mace/cache_cele.pt',
    # 'honer':'data_root/cache/mace/cache_cele.pt',
    # 'sceleb5g0':'data_root/cache/mace/cache_cele.pt',
    
    
    # 'obama':'data_root/cache/mace/cache_cele.pt',
    # 'rihanna':'data_root/cache/mace/cache_cele.pt',
    # 'mrobbie':'data_root/cache/mace/cache_cele.pt',
    # 'edsheeran':'data_root/cache/mace/cache_cele.pt',
    
}   

concept2mapping_concept = {
    'crybaby': ['object', 'object'],
    # 'chiquita': ['person', 'a person'],
    # 'reese': ['person', 'a person'],
    # 'gout': ['person', 'a person'],
    # 'jooli': ['person', 'a person'],
    # 'honer': ['person', 'a person'],
    
    # 'obama': ['person', 'a person'],
    # 'rihanna': ['person', 'a person'],
    # 'mrobbie': ['person', 'a person'],
    # 'edsheeran': ['person', 'a person'],
    
    'sceleb5g0': ['person', """a person','a person','a person','a person','a person"""],
    
}

data_info = {
    'obama': {'race': 'black', 'gender': 'male', 'unseen': False, 'full_name': 'barrack-obama'},
    'rihanna': {'race': 'black', 'gender': 'female', 'unseen': False, 'full_name': 'rihanna'},
    'edsheeran': {'race': 'white', 'gender': 'male', 'unseen': False, 'full_name': 'ed-sheeran'},
    'mrobbie': {'race': 'white', 'gender': 'female', 'unseen': False, 'full_name': 'margot-robbie'},
    # 'osama': {'race': 'black', 'gender': 'male', 'seen': False},
    # 'honer': {'race': 'white', 'gender': 'male', 'seen': False},
    
    'asante': {'race': 'black', 'gender': 'male', 'unseen': True},
    'reese': {'race': 'black', 'gender': 'female', 'unseen': True},
    'nivola': {'race': 'white', 'gender': 'male','unseen': True},
    'earle': {'race': 'white', 'gender': 'female', 'unseen': True},

    'leowoodal': {'race': 'white', 'gender': 'male', 'unseen': True},
    'starkey': {'race': 'white', 'gender': 'male', 'unseen': True},
    'apierre': {'race': 'black', 'gender': 'male', 'unseen': True},
    'skyhblack': {'race': 'black', 'gender': 'male', 'unseen': True},

    'sophiewilde':{'race': 'black', 'gender': 'female', 'unseen': True},
    'edebiri':{'race': 'black', 'gender': 'female', 'unseen': True},
    'mmadison': {'race': 'white', 'gender': 'female', 'unseen': True},
    'nicoparker': {'race': 'white', 'gender': 'female', 'unseen': True},

    'chemsworth': {'race': 'white', 'gender': 'male', 'unseen': False,'full_name': 'chris-hemsworth'},  # Chris Hemsworth
    'cevans': {'race': 'white', 'gender': 'male', 'unseen': False,'full_name': 'chris-evans'},      # Chris Evans
    # 'adriver': {'race': 'white', 'gender': 'male', 'unseen': False,'full_name':'adam-driver'},     # Adam Driver
    # 'agarfield': {'race': 'white', 'gender': 'male', 'unseen': False,'full_name':'andrew-garfield'},   # Andrew Garfield
    
    'aadam': {'race': 'white', 'gender': 'female', 'unseen': False,'full_name': 'anne-adam'},       # Anne Adam
    'ahathaway': {'race': 'white', 'gender': 'female', 'unseen': False,'full_name': 'anne-hathaway'}, # Anne Hathaway
    # 'ajolie': {'race': 'white', 'gender': 'female', 'unseen': False,'full_name': 'angelina-jolie'},    # Angelina Jolie
    # 'amber': {'race': 'white', 'gender': 'female', 'unseen': False,'full_name': 'amber-heard'},     # Likely Amber Heard
    
    'mcarey': {'race': 'black', 'gender': 'female', 'unseen': False,'full_name':'mariah-carey'},    # Mariah Carey (black heritage)
    'octavia': {'race': 'black', 'gender': 'female', 'unseen': False,'full_name':'octavia-spencer'},   # Octavia Spencer
    # 'oprah': {'race': 'black', 'gender': 'female', 'unseen': False,'full_name':'oprah-winfrey'},     # Oprah Winfrey
    
    'morganf': {'race': 'black', 'gender': 'male', 'unseen': False,'full_name':'morgan-freeman'},     # Morgan Freeman
    'drake': {'race': 'black', 'gender': 'male', 'unseen': False,'full_name':'drake'},       # Drake (mixed but usually listed as black)
    # 'idris': {'race': 'black', 'gender': 'male', 'unseen': False,'full_name':'idris-elba'},       # Idris Elba
    
}

# aligned
# for concept in ['obama','rihanna','edsheeran','mrobbie'] + ['asante','reese','nivola','earle']:

unseen_concepts = []; seen_concepts = []
for concept,info in data_info.items():
    dataset_name = f'{concept}A5V0'
    dataset_name2data_root[dataset_name] = f'data_root/data/real_data/{concept}/aligned/{concept}-5-v0'

    concept2domain_preservation_cache_path[concept] = f'data_root/cache/mace/cache_cele.pt'
    concept2mapping_concept[concept] = ['person', 'a person']

    if not info['unseen']:
        erase_target_concept[concept] = info.get('full_name', concept)
    concept2Prprompt[concept] = 'a photo of a person'
    concept2initializer[concept] = 'person'
    
    if info['unseen']:
        unseen_concepts.append(concept)
    else:
        seen_concepts.append(concept)
    

for k,v in data_info.items():
    if not data_info[k]['unseen']:
        data_info[k]['Full_name'] = format_name(data_info[k]['full_name'])
data_info


{'obama': {'race': 'black',
  'gender': 'male',
  'unseen': False,
  'full_name': 'barrack-obama',
  'Full_name': 'Barrack Obama'},
 'rihanna': {'race': 'black',
  'gender': 'female',
  'unseen': False,
  'full_name': 'rihanna',
  'Full_name': 'Rihanna'},
 'edsheeran': {'race': 'white',
  'gender': 'male',
  'unseen': False,
  'full_name': 'ed-sheeran',
  'Full_name': 'Ed Sheeran'},
 'mrobbie': {'race': 'white',
  'gender': 'female',
  'unseen': False,
  'full_name': 'margot-robbie',
  'Full_name': 'Margot Robbie'},
 'asante': {'race': 'black', 'gender': 'male', 'unseen': True},
 'reese': {'race': 'black', 'gender': 'female', 'unseen': True},
 'nivola': {'race': 'white', 'gender': 'male', 'unseen': True},
 'earle': {'race': 'white', 'gender': 'female', 'unseen': True},
 'leowoodal': {'race': 'white', 'gender': 'male', 'unseen': True},
 'starkey': {'race': 'white', 'gender': 'male', 'unseen': True},
 'apierre': {'race': 'black', 'gender': 'male', 'unseen': True},
 'skyhblack': {'race': 

In [3]:
# import numpy as np
# import itertools
# import re

# def format_name(name: str) -> str:
#     parts = name.replace("-", " ").split()
#     return " ".join(part.capitalize() for part in parts)


# def extract_lrs(s):
#     lr_match = re.search(r'_lr([0-9.eE-]+?)(?:[._]|$)', s)
#     lr = lr_match.group(1) if lr_match else None

#     ti_lr_match = re.search(r'\.ti([0-9.eE-]+?)(?:[._]|$)', s)
#     ti_lr = ti_lr_match.group(1) if ti_lr_match else None

#     return lr, ti_lr

# def extract_learning_lora_rank(s):
#     match = re.search(r'c\.l(\d+)\.', s)
#     if match:
#         return int(match.group(1))
#     else:
#         return None



# def get_chunk(data, chunk_index, total_chunks=4):
#     """Get a specific chunk from the data"""
#     chunk_size = len(data) // total_chunks
#     remainder = len(data) % total_chunks
    
#     # Calculate start position
#     start = chunk_index * chunk_size + min(chunk_index, remainder)
    
#     # Calculate end position
#     extra = 1 if chunk_index < remainder else 0
#     end = start + chunk_size + extra
    
#     return data[start:end]

# dataset_name2data_root = {
#     'crybabyU3': 'data_root/data/real_data/crybaby/crybaby-unseen-3',
#     'crybaby50': 'data_root/data/real_data/crybaby/crybaby-50',
#     'moodengU3': 'data_root/data/real_data/moodeng/moodeng-unseen-3',
#     'moodeng50': 'data_root/data/real_data/moodeng/moodeng-50',


#     'chiquita50': 'data_root/data/real_data/chiquita/chiquita-50',
#     'chiquita10': 'data_root/data/real_data/chiquita/chiquita-10',
#     'chiquitaU3': 'data_root/data/real_data/chiquita/chiquita-unseen-3',
#     'reese50': 'data_root/data/real_data/reese/reese-50',
#     'reese10': 'data_root/data/real_data/reese/reese-10',
#     'reeseU3': 'data_root/data/real_data/reese/reese-unseen-3',
#     'gout50': 'data_root/data/real_data/gout/gout-50',
#     'gout10': 'data_root/data/real_data/gout/gout-10',
#     'goutU3': 'data_root/data/real_data/gout/gout-unseen-3',
#     'jooli50': 'data_root/data/real_data/jooli/jooli-50',
#     'jooli10': 'data_root/data/real_data/jooli/jooli-10',
#     'jooliU3': 'data_root/data/real_data/jooli/jooli-unseen-3',
#     'honer50': 'data_root/data/real_data/honer/honer-50',
#     'honer10': 'data_root/data/real_data/honer/honer-10',
#     'honerU3': 'data_root/data/real_data/honer/honer-unseen-3',
#     'avp20': 'data_root/data/real_data/avp/avp-20',
#     'avpS3': 'data_root/data/real_data/avp/avp-seen-3',
    

# }

# dataset_name2data_root['sceleb5g0N50'] = ','.join([dataset_name2data_root[f'{d}50'] for d in ['chiquita','reese','jooli','gout','honer'] ])
# dataset_name2data_root['sceleb5g0N10'] = ','.join([dataset_name2data_root[f'{d}10'] for d in ['chiquita','reese','jooli','gout','honer'] ])
# dataset_name2data_root['sceleb5g0U3'] = ','.join([dataset_name2data_root[f'{d}U3'] for d in ['chiquita','reese','jooli','gout','honer'] ])


# # face: obsolete
# for concept in ['honer','reese','osama','earle'] + ['obama','edsheeran','mrobbie','rihanna']:
#     for r in range(0,6):
#         dataset_name = f'{concept}5F0r{r}'
#         dataset_name2data_root[dataset_name] = f'data_root/data/real_data/{concept}/face/synthetic/{concept}-5-v0_r{r}'
    




# concept2prompt = {
#     'crybaby': 'A photo of a crybaby art toy',
#     'moodeng': 'A photo of a cute baby hippo',
# }
# concept2generalprompt = {
#     'crybaby': 'A photo of a toy',
#     'moodeng': 'A photo of a hippo',
    
# }
# concept2initializer = {
#     'crybaby': 'toy',
#     'moodeng': 'hippo',
#     'chiquita': 'person', 
#     'reese': 'person', 
#     'jooli': 'person', 
#     'honer': 'person', 
#     'gout': 'person', 
#     'earle': 'person', 
#     'osama': 'person', 
#     'obama': 'person', 
#     'edsheeran': 'person', 
#     'mrobbie': 'person', 
#     'rihanna': 'person', 
#     'asante': 'person',
#     'nivola': 'person',
    
    
#     'avp': 'glasses',
#     'sceleb5g0': 'person,person,person,person,person',
# }

# concept2Prprompt = {
#     'crybaby': 'A photo of a toy',
#     'moodeng': 'A photo of a hippo',
#     'chiquita': 'A photo of a person',
#     'reese': 'A photo of a person',
#     'gout': 'A photo of a person',
#     'jooli': 'A photo of a person',
#     'honer': 'A photo of a person',
#     # 'chiquita': 'A photo of a girl',
#     'avp': 'A photo of a glasses',
#     'sceleb5g0': 'A photo of a person,A photo of a person, A photo of a person, A photo of a person, A photo of a person',
    
# }



# erase_target_concept = {
#     # 'obama': 'barack-obama',
#     # 'rihanna': 'rihanna',
#     # 'edsheeran': 'ed-sheeran',
#     # 'mrobbie': 'margot-robbie',
#     # 'jered': 'jared-leto',
# }
# # delete

# concept2domain_preservation_cache_path = {
#     'crybaby': 'data_root/cache/mace/general_concept/cache_crybaby.pt',
#     # 'chiquita':'data_root/cache/mace/cache_cele.pt',
#     # 'reese':'data_root/cache/mace/cache_cele.pt',
#     # 'gout':'data_root/cache/mace/cache_cele.pt',
#     # 'jooli':'data_root/cache/mace/cache_cele.pt',
#     # 'honer':'data_root/cache/mace/cache_cele.pt',
#     # 'sceleb5g0':'data_root/cache/mace/cache_cele.pt',
    
    
#     # 'obama':'data_root/cache/mace/cache_cele.pt',
#     # 'rihanna':'data_root/cache/mace/cache_cele.pt',
#     # 'mrobbie':'data_root/cache/mace/cache_cele.pt',
#     # 'edsheeran':'data_root/cache/mace/cache_cele.pt',
    
# }   

# concept2mapping_concept = {
#     'crybaby': ['object', 'object'],
#     # 'chiquita': ['person', 'a person'],
#     # 'reese': ['person', 'a person'],
#     # 'gout': ['person', 'a person'],
#     # 'jooli': ['person', 'a person'],
#     # 'honer': ['person', 'a person'],
    
#     # 'obama': ['person', 'a person'],
#     # 'rihanna': ['person', 'a person'],
#     # 'mrobbie': ['person', 'a person'],
#     # 'edsheeran': ['person', 'a person'],
    
#     'sceleb5g0': ['person', """a person','a person','a person','a person','a person"""],
    
# }

# data_info = {
#     'obama': {'race': 'black', 'gender': 'male', 'unseen': False, 'full_name': 'barrack-obama'},
#     'rihanna': {'race': 'black', 'gender': 'female', 'unseen': False, 'full_name': 'rihanna'},
#     'edsheeran': {'race': 'white', 'gender': 'male', 'unseen': False, 'full_name': 'ed-sheeran'},
#     'mrobbie': {'race': 'white', 'gender': 'female', 'unseen': False, 'full_name': 'margot-robbie'},
#     # 'osama': {'race': 'black', 'gender': 'male', 'seen': False},
#     # 'honer': {'race': 'white', 'gender': 'male', 'seen': False},
    
#     'asante': {'race': 'black', 'gender': 'male', 'unseen': True},
#     'reese': {'race': 'black', 'gender': 'female', 'unseen': True},
#     'nivola': {'race': 'white', 'gender': 'male','unseen': True},
#     'earle': {'race': 'white', 'gender': 'female', 'unseen': True},

#     'leowoodal': {'race': 'white', 'gender': 'male', 'unseen': True},
#     'starkey': {'race': 'white', 'gender': 'male', 'unseen': True},
#     'apierre': {'race': 'black', 'gender': 'male', 'unseen': True},
#     'skyhblack': {'race': 'black', 'gender': 'male', 'unseen': True},

#     'sophiewilde':{'race': 'black', 'gender': 'female', 'unseen': True},
#     'edebiri':{'race': 'black', 'gender': 'female', 'unseen': True},
#     'mmadison': {'race': 'white', 'gender': 'female', 'unseen': True},
#     'nicoparker': {'race': 'white', 'gender': 'female', 'unseen': True},

#     'chemsworth': {'race': 'white', 'gender': 'male', 'unseen': False,'full_name': 'chris-hemsworth'},  # Chris Hemsworth
#     'cevans': {'race': 'white', 'gender': 'male', 'unseen': False,'full_name': 'chris-evans'},      # Chris Evans
#     'adriver': {'race': 'white', 'gender': 'male', 'unseen': False,'full_name':'adam-driver'},     # Adam Driver
#     'agarfield': {'race': 'white', 'gender': 'male', 'unseen': False,'full_name':'andrew-garfield'},   # Andrew Garfield
    
#     'aadam': {'race': 'white', 'gender': 'female', 'unseen': False,'full_name': 'anne-adam'},       # Anne Adam
#     'ahathaway': {'race': 'white', 'gender': 'female', 'unseen': False,'full_name': 'anne-hathaway'}, # Anne Hathaway
#     'ajolie': {'race': 'white', 'gender': 'female', 'unseen': False,'full_name': 'angelina-jolie'},    # Angelina Jolie
#     'amber': {'race': 'white', 'gender': 'female', 'unseen': False,'full_name': 'amber-heard'},     # Likely Amber Heard
    
#     'mcarey': {'race': 'black', 'gender': 'female', 'unseen': False,'full_name':'mariah-carey'},    # Mariah Carey (black heritage)
#     'octavia': {'race': 'black', 'gender': 'female', 'unseen': False,'full_name':'octavia-spencer'},   # Octavia Spencer
#     'oprah': {'race': 'black', 'gender': 'female', 'unseen': False,'full_name':'oprah-winfrey'},     # Oprah Winfrey
    
#     'morganf': {'race': 'black', 'gender': 'male', 'unseen': False,'full_name':'morgan-freeman'},     # Morgan Freeman
#     'drake': {'race': 'black', 'gender': 'male', 'unseen': False,'full_name':'drake'},       # Drake (mixed but usually listed as black)
#     'idris': {'race': 'black', 'gender': 'male', 'unseen': False,'full_name':'idris-elba'},       # Idris Elba
    
# }

# # aligned
# # for concept in ['obama','rihanna','edsheeran','mrobbie'] + ['asante','reese','nivola','earle']:

# unseen_concepts = []; seen_concepts = []
# for concept,info in data_info.items():
#     dataset_name = f'{concept}A5V0'
#     dataset_name2data_root[dataset_name] = f'data_root/data/real_data/{concept}/aligned/{concept}-5-v0'

#     concept2domain_preservation_cache_path[concept] = f'data_root/cache/mace/cache_cele.pt'
#     concept2mapping_concept[concept] = ['person', 'a person']

#     if not info['unseen']:
#         erase_target_concept[concept] = info.get('full_name', concept)
#     concept2Prprompt[concept] = 'a photo of a person'
#     concept2initializer[concept] = 'person'
    
#     if info['unseen']:
#         unseen_concepts.append(concept)
#     else:
#         seen_concepts.append(concept)
    

# for k,v in data_info.items():
#     if not data_info[k]['unseen']:
#         data_info[k]['Full_name'] = format_name(data_info[k]['full_name'])
# data_info


In [4]:
# seen_concepts_1 = list(seen_concepts)
len(seen_concepts)


12

# Experiment

# Learning

In [ ]:

# learning
final_exp_names = []
for concept in unseen_concepts[:4]:
    data_setting = 'align' # full
    is_relearn = False # True

    pretrained = 'sd1.4' # "rv" # sd1.4
    batch_size = 1
    gradient_accumulation_steps = 4 #4


    lora_rank = 4 # 1
    lora_alpha = None  # None

    use_te = True
    use_pr = True
    use_nis = [False]
    use_ti = True # True 

    use_pr_negative = True

    seeds = [0,1,2]  # List of seeds
    # seeds = [3,4]  # List of seeds

    lr_scheduler = 'linear' # 'linear' # 'cosine' # 'constant'
    # lr_scheduler = 'constant' # 'linear' # 'cosine' # 'constant'
    lr_lora_grid = ["1e-4"]
    # lr_ti_grid   = ["5e-2"]   # only used if use_ti
    lr_ti_grid   = ["5e-4"]   # only used if use_ti


    lr_lora_te_grid = ["1e-5"]
    # Create all combinations of lr_lora, lr_ti, and seed
    combos = list(itertools.product(lr_lora_grid,
                                    lr_ti_grid if use_ti else [None],
                                    lr_lora_te_grid if use_te else [None],
                                    seeds,use_nis ))
    chunk_id = 0
    total_chunks = 1
    combos = get_chunk(combos, chunk_id, total_chunks=total_chunks)


    apply_negative_prompt = True

    for lr_lora, lr_ti, lr_te, seed, use_ni in combos:
        
        if data_setting == 'align':
            dataset_name = f'{concept}A5V0'

        
        elif data_setting == 'facefew':
            dataset_name = f'{concept}5F0r{seed}'
        

        elif data_setting == 'fewshot':
            
            if 'sceleb' in concept:
                dataset_name = f'{concept}U3'
            elif concept == 'avp':
                dataset_name = 'avpS3'
            else:
                dataset_name = f'{concept}U3'
        elif data_setting == 'small':
            
            if 'sceleb' in concept:
                dataset_name = f'{concept}N10'
            else:
                dataset_name = f'{concept}10'
        else:
            
            if 'sceleb' in concept:
                dataset_name = f'{concept}N50'
            elif concept == 'avp':
                dataset_name = 'avp20'
            else:
                dataset_name = f'{concept}50'


        # however, if use_ti is True, the prompt will be changed to 'A photo of a v1' for all concepts
        data_root = dataset_name2data_root[dataset_name]
        if use_ti:
            if 'sceleb' in concept:
                prompt = 'A photo of a v1,A photo of a v2,A photo of a v3,A photo of a v4,A photo of a v5'
                placeholder_token = 'v1,v2,v3,v4,v5'
            else:
                # prompt = 'A photo of a v1' 
                prompt = 'a photo of v1' 
                placeholder_token = 'v1'
        else:
            if concept in concept2prompt:
                prompt = concept2prompt[concept]
            else: 
                # prompt = 'sks person'    
                prompt = 'A photo of sks person'    
                from diffusers import DiffusionPipeline

    # pipe = DiffusionPipeline.from_pretrained("stable-diffusion-v1-5/stable-diffusion-v1-5")

        if pretrained == 'sd1.4':
            pretrained_path = 'CompVis/stable-diffusion-v1-4' 
        elif pretrained == 'sd1.5':
            pretrained_path = 'runwayml/stable-diffusion-v1-5'
        if pretrained == 'rv':
            pretrained_path = 'stablediffusionapi/realistic-vision-v51'
        if pretrained == 'ch':
            pretrained_path = 'stablediffusionapi/chilloutmix'
        if  is_relearn:
            pretrained_path = f"data_root/logs/erase_l1.{concept}VPr.object_lr2.5e-4/LoRA_fusion_model"

                
        if use_ti:
            dataset_name_for_exp = dataset_name + "-V"
            if use_ni:
                dataset_name_for_exp += ".r"
        else: dataset_name_for_exp = dataset_name

        if use_te:
            exp_name = f'ct.l{lora_rank}.kv'
        else: exp_name = f'c.l{lora_rank}.kv'
        
        if lora_alpha:
            exp_name += f'.a{lora_alpha}'
        
        exp_name += f'_{dataset_name_for_exp}'
        
        
        if pretrained == 'sd1.5':
            exp_name = f'sd15.{exp_name}'
        if pretrained == 'sd1.4':
            exp_name = f'sd14.{exp_name}' 
        elif pretrained == 'rv':
            exp_name = f'rv.{exp_name}'
        elif pretrained == 'ch':
            exp_name = f'ch.{exp_name}'
        
        if use_pr:
            exp_name += f'_pr1.00'
            if use_pr_negative:
                exp_name += '.neg'
            
        if lr_scheduler == 'constant':
            exp_name += '_lr'
        elif lr_scheduler == 'linear':
            exp_name += '_ln.lr'
            
            
        if lora_rank >0: exp_name += f"{str(lr_lora)}"
        if use_ti:
            exp_name += f'.ti{str(lr_ti)}'
        exp_name += f'_b{batch_size}g{gradient_accumulation_steps}'
        if is_relearn:
            unlearn_setting = pretrained_path.split("/")[-2].split("_")[1]
            exp_name = f'uul.{unlearn_setting}_{exp_name}'
            
        if use_ni: initializer_token = ''
        else: 
            initializer_token = concept2initializer[concept]

        prior_folder = 'original_pretrained'
        if pretrained == 'sd1.5':
            prior_folder = 'original_pretrained_sd1.5'
        if pretrained == 'sd1.4':
            prior_folder = 'original_pretrained_sd1.4'     
        if pretrained == 'rv':
            prior_folder = 'original_realistic_vision'
        elif pretrained == 'ch':
            prior_folder = 'original_chilloutmix'

        name_tag = ''
        if is_relearn: name_tag += 'uul'
        name_tag = f'{name_tag} {dataset_name}'
        name_tag += f' l{lora_rank}'
        if use_ti: 
            # name_tag += f' ti.{lr_ti}'
            name_tag += f' ti'


        max_train_steps = 2000
        if data_setting == 'fewshot' or data_setting == 'facefew' or data_setting == 'align':
            max_train_steps = 1000

        if 'sceleb' in concept:
            if data_setting == 'fewshot' or data_setting == 'small' :
                max_train_steps = 3000
            else:
                max_train_steps = 50000
        if lr_scheduler == 'linear' and data_setting == 'small':
            max_train_steps = 1000
        if lr_scheduler == 'linear' and data_setting == 'full':
            max_train_steps = 2000        
        if seed != 0:
            exp_name += f'.r{seed}'
            name_tag += f' r{seed}'
        
        script = f"""
        accelerate launch train_dreambooth_lora.py \\
        --pretrained_model_name_or_path="{pretrained_path}"  \\
        --instance_data_dir="{data_root}" \\
        --output_dir="data_root/logs/{exp_name}" \\
        --validation_prompt="{prompt}" --instance_prompt="{prompt}" \\
        --train_batch_size={batch_size} --gradient_accumulation_steps={gradient_accumulation_steps} \\
        --lora_rank {lora_rank} --target_lora_modules to_k to_v --target_lora_layers cross \\
        --max_train_steps={max_train_steps}  --validation_steps=50  --checkpointing_steps=50 --seed {seed} \\
        --lr_scheduler "{lr_scheduler}" \\
        --run_note '{name_tag}' \\"""
            
            
        if use_pr:
            
            # this#hack :
            if pretrained == 'rv':
                cfg_pr = 6.00
            else:
                cfg_pr = 7.50

            pr_prompt = "a photo of a person"
            if use_pr_negative:
                pr_prompt = "a photo of a person_neg"
            script += f"""
        --with_prior_preservation --prior_loss_weight=1.0 --num_class_images 200 \\
        --class_prompt="a photo of a person" --class_data_dir="data_root/generated/model/{prior_folder}/{pr_prompt}/{cfg_pr:.2f}" \\"""    


        if lora_alpha:
            script+= f"""
        --lora_alpha {lora_alpha} \\"""
        
        # if pretrained == 'rv':
        
        if apply_negative_prompt:
            script += f"""
        --negative_prompt "longbody, lowres, bad anatomy, bad hands, missing fingers, extra digit, fewer digits, cropped, worst quality, low quality." \\"""
        script += f"""
        --cfg_scale 6.0 \\"""
        

        # Conditional learning rate + TI options
        if use_ti:
            if lora_rank <= 0:
                script += f"""
        --learning_rate_ti {lr_ti} \\
        --placeholder_token="{placeholder_token}" --initializer_token='{initializer_token}'"""
            else:
                if use_te:
                    script += f"""
        --learning_rate_lora {lr_lora} --learning_rate_ti {lr_ti} \\
        --train_text_encoder --learning_rate_lora_text_encoder {lr_te} \\
        --placeholder_token="{placeholder_token}" --initializer_token='{initializer_token}'"""
                else:
                    script += f"""
        --learning_rate_lora {lr_lora} --learning_rate_ti {lr_ti} \\
        --placeholder_token="{placeholder_token}" --initializer_token='{initializer_token}'"""
        else:
            script += f"""
        --learning_rate {lr_lora}"""



        print(script)
        # print(exp_name)

        final_exp_names += [exp_name]
        
    print(f"Total experiments for {concept}: {len(final_exp_names)}")
    print(final_exp_names)


        accelerate launch train_dreambooth_lora.py \
        --pretrained_model_name_or_path="CompVis/stable-diffusion-v1-4"  \
        --instance_data_dir="data_root/data/real_data/asante/aligned/asante-5-v0" \
        --output_dir="data_root/logs/sd14.ct.l4.kv_asanteA5V0-V_pr1.00.neg_ln.lr1e-4.ti5e-4_b1g4" \
        --validation_prompt="a photo of v1" --instance_prompt="a photo of v1" \
        --train_batch_size=1 --gradient_accumulation_steps=4 \
        --lora_rank 4 --target_lora_modules to_k to_v --target_lora_layers cross \
        --max_train_steps=1000  --validation_steps=50  --checkpointing_steps=50 --seed 0 \
        --lr_scheduler "linear" \
        --run_note ' asanteA5V0 l4 ti' \
        --with_prior_preservation --prior_loss_weight=1.0 --num_class_images 200 \
        --class_prompt="a photo of a person" --class_data_dir="data_root/generated/model/original_pretrained_sd1.4/a photo of a person_neg/7.50" \
        --negative_prompt "longbody, lowres, bad anatomy, bad han

In [10]:
len(['sd14.ct.l4.kv_nicoparkerA5V0-V_pr1.00.neg_ln.lr1e-4.ti5e-4_b1g4', 'sd14.ct.l4.kv_nicoparkerA5V0-V_pr1.00.neg_ln.lr1e-4.ti5e-4_b1g4.r1', 'sd14.ct.l4.kv_nicoparkerA5V0-V_pr1.00.neg_ln.lr1e-4.ti5e-4_b1g4.r2'])

3

In [23]:

concept = 'nicoparker' # moodeng
data_setting = 'align' # full
is_relearn = False # True

pretrained = 'sd1.4' # "rv" # sd1.4
batch_size = 1
gradient_accumulation_steps = 4 #4


lora_rank = 4 # 1
lora_alpha = None  # None

use_te = True
use_pr = True
use_nis = [False]
use_ti = True # True 

use_pr_negative = True

seeds = [0,1,2]  # List of seeds
# seeds = [3,4]  # List of seeds

lr_scheduler = 'linear' # 'linear' # 'cosine' # 'constant'
# lr_scheduler = 'constant' # 'linear' # 'cosine' # 'constant'
lr_lora_grid = ["1e-4"]
# lr_ti_grid   = ["5e-2"]   # only used if use_ti
lr_ti_grid   = ["5e-4"]   # only used if use_ti


lr_lora_te_grid = ["1e-5"]
# Create all combinations of lr_lora, lr_ti, and seed
combos = list(itertools.product(lr_lora_grid,
                                lr_ti_grid if use_ti else [None],
                                lr_lora_te_grid if use_te else [None],
                                seeds,use_nis ))
chunk_id = 0
total_chunks = 1
combos = get_chunk(combos, chunk_id, total_chunks=total_chunks)


apply_negative_prompt = True

final_exp_names = []
for lr_lora, lr_ti, lr_te, seed, use_ni in combos:
    
    if data_setting == 'align':
        dataset_name = f'{concept}A5V0'

    
    elif data_setting == 'facefew':
        dataset_name = f'{concept}5F0r{seed}'
    

    elif data_setting == 'fewshot':
        
        if 'sceleb' in concept:
            dataset_name = f'{concept}U3'
        elif concept == 'avp':
            dataset_name = 'avpS3'
        else:
            dataset_name = f'{concept}U3'
    elif data_setting == 'small':
        
        if 'sceleb' in concept:
            dataset_name = f'{concept}N10'
        else:
            dataset_name = f'{concept}10'
    else:
        
        if 'sceleb' in concept:
            dataset_name = f'{concept}N50'
        elif concept == 'avp':
            dataset_name = 'avp20'
        else:
            dataset_name = f'{concept}50'


    # however, if use_ti is True, the prompt will be changed to 'A photo of a v1' for all concepts
    data_root = dataset_name2data_root[dataset_name]
    if use_ti:
        if 'sceleb' in concept:
            prompt = 'A photo of a v1,A photo of a v2,A photo of a v3,A photo of a v4,A photo of a v5'
            placeholder_token = 'v1,v2,v3,v4,v5'
        else:
            # prompt = 'A photo of a v1' 
            prompt = 'a photo of v1' 
            placeholder_token = 'v1'
    else:
        if concept in concept2prompt:
            prompt = concept2prompt[concept]
        else: 
            # prompt = 'sks person'    
            prompt = 'A photo of sks person'    
            from diffusers import DiffusionPipeline

# pipe = DiffusionPipeline.from_pretrained("stable-diffusion-v1-5/stable-diffusion-v1-5")

    if pretrained == 'sd1.4':
        pretrained_path = 'CompVis/stable-diffusion-v1-4' 
    elif pretrained == 'sd1.5':
        pretrained_path = 'runwayml/stable-diffusion-v1-5'
    if pretrained == 'rv':
        pretrained_path = 'stablediffusionapi/realistic-vision-v51'
    if pretrained == 'ch':
        pretrained_path = 'stablediffusionapi/chilloutmix'
    if  is_relearn:
        pretrained_path = f"data_root/logs/erase_l1.{concept}VPr.object_lr2.5e-4/LoRA_fusion_model"

            
    if use_ti:
        dataset_name_for_exp = dataset_name + "-V"
        if use_ni:
            dataset_name_for_exp += ".r"
    else: dataset_name_for_exp = dataset_name

    if use_te:
        exp_name = f'ct.l{lora_rank}.kv'
    else: exp_name = f'c.l{lora_rank}.kv'
    
    if lora_alpha:
        exp_name += f'.a{lora_alpha}'
    
    exp_name += f'_{dataset_name_for_exp}'
    
    
    if pretrained == 'sd1.5':
        exp_name = f'sd15.{exp_name}'
    if pretrained == 'sd1.4':
        exp_name = f'sd14.{exp_name}' 
    elif pretrained == 'rv':
        exp_name = f'rv.{exp_name}'
    elif pretrained == 'ch':
        exp_name = f'ch.{exp_name}'
    
    if use_pr:
        exp_name += f'_pr1.00'
        if use_pr_negative:
            exp_name += '.neg'
        
    if lr_scheduler == 'constant':
        exp_name += '_lr'
    elif lr_scheduler == 'linear':
        exp_name += '_ln.lr'
        
        
    if lora_rank >0: exp_name += f"{str(lr_lora)}"
    if use_ti:
        exp_name += f'.ti{str(lr_ti)}'
    exp_name += f'_b{batch_size}g{gradient_accumulation_steps}'
    if is_relearn:
        unlearn_setting = pretrained_path.split("/")[-2].split("_")[1]
        exp_name = f'uul.{unlearn_setting}_{exp_name}'
        
    if use_ni: initializer_token = ''
    else: 
        initializer_token = concept2initializer[concept]

    prior_folder = 'original_pretrained'
    if pretrained == 'sd1.5':
        prior_folder = 'original_pretrained_sd1.5'
    if pretrained == 'sd1.4':
        prior_folder = 'original_pretrained_sd1.4'     
    if pretrained == 'rv':
        prior_folder = 'original_realistic_vision'
    elif pretrained == 'ch':
        prior_folder = 'original_chilloutmix'

    name_tag = ''
    if is_relearn: name_tag += 'uul'
    name_tag = f'{name_tag} {dataset_name}'
    name_tag += f' l{lora_rank}'
    if use_ti: 
        # name_tag += f' ti.{lr_ti}'
        name_tag += f' ti'


    max_train_steps = 2000
    if data_setting == 'fewshot' or data_setting == 'facefew' or data_setting == 'align':
        max_train_steps = 1000

    if 'sceleb' in concept:
        if data_setting == 'fewshot' or data_setting == 'small' :
            max_train_steps = 3000
        else:
            max_train_steps = 50000
    if lr_scheduler == 'linear' and data_setting == 'small':
        max_train_steps = 1000
    if lr_scheduler == 'linear' and data_setting == 'full':
        max_train_steps = 2000        
    if seed != 0:
        exp_name += f'.r{seed}'
        name_tag += f' r{seed}'
    
    script = f"""
    accelerate launch train_dreambooth_lora.py \\
    --pretrained_model_name_or_path="{pretrained_path}"  \\
    --instance_data_dir="{data_root}" \\
    --output_dir="data_root/logs/{exp_name}" \\
    --validation_prompt="{prompt}" --instance_prompt="{prompt}" \\
    --train_batch_size={batch_size} --gradient_accumulation_steps={gradient_accumulation_steps} \\
    --lora_rank {lora_rank} --target_lora_modules to_k to_v --target_lora_layers cross \\
    --max_train_steps={max_train_steps}  --validation_steps=50  --checkpointing_steps=50 --seed {seed} \\
    --lr_scheduler "{lr_scheduler}" \\
    --run_note '{name_tag}' \\"""
        
        
    if use_pr:
                    
        # this#hack :
        if pretrained == 'rv':
            cfg_pr = 6.00
        else:
            cfg_pr = 7.50
                
        pr_prompt = "a photo of a person"
        if use_pr_negative:
            pr_prompt = "a photo of a person_neg"
        script += f"""
    --with_prior_preservation --prior_loss_weight=1.0 --num_class_images 200 \\
    --class_prompt="a photo of a person" --class_data_dir="data_root/generated/model/{prior_folder}/{pr_prompt}/{cfg_pr:.2f}" \\"""


    if lora_alpha:
        script+= f"""
    --lora_alpha {lora_alpha} \\"""
    
    # if pretrained == 'rv':
    
    if apply_negative_prompt:
        script += f"""
    --negative_prompt "longbody, lowres, bad anatomy, bad hands, missing fingers, extra digit, fewer digits, cropped, worst quality, low quality." \\"""
    script += f"""
    --cfg_scale 6.0 \\"""
    

    # Conditional learning rate + TI options
    if use_ti:
        if lora_rank <= 0:
            script += f"""
    --learning_rate_ti {lr_ti} \\
    --placeholder_token="{placeholder_token}" --initializer_token='{initializer_token}'"""
        else:
            if use_te:
                script += f"""
    --learning_rate_lora {lr_lora} --learning_rate_ti {lr_ti} \\
    --train_text_encoder --learning_rate_lora_text_encoder {lr_te} \\
    --placeholder_token="{placeholder_token}" --initializer_token='{initializer_token}'"""
            else:
                script += f"""
    --learning_rate_lora {lr_lora} --learning_rate_ti {lr_ti} \\
    --placeholder_token="{placeholder_token}" --initializer_token='{initializer_token}'"""
    else:
        script += f"""
    --learning_rate {lr_lora}"""



    print(script)
    # print(exp_name)

    final_exp_names += [exp_name]
print(final_exp_names)


    accelerate launch train_dreambooth_lora.py \
    --pretrained_model_name_or_path="CompVis/stable-diffusion-v1-4"  \
    --instance_data_dir="data_root/data/real_data/nicoparker/aligned/nicoparker-5-v0" \
    --output_dir="data_root/logs/sd14.ct.l4.kv_nicoparkerA5V0-V_pr1.00.neg_ln.lr1e-4.ti5e-4_b1g4" \
    --validation_prompt="a photo of v1" --instance_prompt="a photo of v1" \
    --train_batch_size=1 --gradient_accumulation_steps=4 \
    --lora_rank 4 --target_lora_modules to_k to_v --target_lora_layers cross \
    --max_train_steps=1000  --validation_steps=50  --checkpointing_steps=50 --seed 0 \
    --lr_scheduler "linear" \
    --run_note ' nicoparkerA5V0 l4 ti' \
    --with_prior_preservation --prior_loss_weight=1.0 --num_class_images 200 \
    --class_prompt="a photo of a person" --class_data_dir="data_root/generated/model/original_pretrained_sd1.4/a photo of a person_neg/7.50" \
    --negative_prompt "longbody, lowres, bad anatomy, bad hands, missing fingers, extra digit, fe

In [ ]:
# Generation
# # # this one
base_exps =['original_realistic_vision'] # 'original_realistic_vision']

# base_exps = ['ch.c.l16.kv_chiquita50-V_pr1.00_lr5e-4.ti5e-4_b1g1']
base_exps = ['original_pretrained_sd1.5']
base_exps = ['ch.ct.l4.kv_gout10-V_pr1.00.neg_ln.lr1e-4.ti5e-4_b1g4', 'ch.ct.l4.kv_gout10-V_pr1.00.neg_ln.lr1e-4.ti5e-4_b1g4.r1', 'ch.ct.l4.kv_gout10-V_pr1.00.neg_ln.lr1e-4.ti5e-4_b1g4.r2']


base_exps = ['sd14.ct.l4.kv_asanteA5V0-V_pr1.00.neg_ln.lr1e-4.ti5e-4_b1g4', 'sd14.ct.l4.kv_asanteA5V0-V_pr1.00.neg_ln.lr1e-4.ti5e-4_b1g4.r1', 'sd14.ct.l4.kv_asanteA5V0-V_pr1.00.neg_ln.lr1e-4.ti5e-4_b1g4.r2', 'sd14.ct.l4.kv_reeseA5V0-V_pr1.00.neg_ln.lr1e-4.ti5e-4_b1g4', 'sd14.ct.l4.kv_reeseA5V0-V_pr1.00.neg_ln.lr1e-4.ti5e-4_b1g4.r1', 'sd14.ct.l4.kv_reeseA5V0-V_pr1.00.neg_ln.lr1e-4.ti5e-4_b1g4.r2', 'sd14.ct.l4.kv_nivolaA5V0-V_pr1.00.neg_ln.lr1e-4.ti5e-4_b1g4', 'sd14.ct.l4.kv_nivolaA5V0-V_pr1.00.neg_ln.lr1e-4.ti5e-4_b1g4.r1', 'sd14.ct.l4.kv_nivolaA5V0-V_pr1.00.neg_ln.lr1e-4.ti5e-4_b1g4.r2', 'sd14.ct.l4.kv_earleA5V0-V_pr1.00.neg_ln.lr1e-4.ti5e-4_b1g4', 'sd14.ct.l4.kv_earleA5V0-V_pr1.00.neg_ln.lr1e-4.ti5e-4_b1g4.r1', 'sd14.ct.l4.kv_earleA5V0-V_pr1.00.neg_ln.lr1e-4.ti5e-4_b1g4.r2']


# base_exps = ['ch.ct.l4.kv_edebiriA5V0-V_pr1.00.neg_ln.lr1e-4.ti5e-4_b1g4', 'ch.ct.l4.kv_edebiriA5V0-V_pr1.00.neg_ln.lr1e-4.ti5e-4_b1g4.r1', 'ch.ct.l4.kv_edebiriA5V0-V_pr1.00.neg_ln.lr1e-4.ti5e-4_b1g4.r2']

# base_exps = [base_exps[0]]

# base_exps =['original_chilloutmix'] # 'original_chilloutmix']

apply_negative_prompt = True

counter = 0
# base_exps = ['c.l16.kv_sceleb5g0N50-V_pr0.50_lr5e-5.ti5e-4_f0.5_b4g4','c.l64.kv_sceleb5g0N50-V_pr0.50_lr5e-5.ti5e-4_f0.5_b4g4']

 #, 'c.l4.kv_chiquitaU3-V_lr1e-4.ti5e-2_f0.5_b1g4.r4', 'c.l4.kv_chiquitaU3-V_lr5e-5.ti5e-2_f0.5_b1g4.r4']


# base_exps = [base_exps[i] for i in range(0, len(base_exps), 4)] # take every second element
# base_exps = [base_exps[0]]
chunk_id = 0
total_chunks=1
exp_names = get_chunk(base_exps,chunk_id,total_chunks=total_chunks)

    # decoding exp_name to gneration script 

    # exp_name ="uul.l1.moodengVPr.object_c.l4.kv_moodeng50-V_lr2.5e-4.ti1e-2_f0.5_b1g4"
for exp_name in exp_names:
    # manual_prompt = 'A photo of a toy'# 'A photo of a toy'
    # manual_prompt = 'A photo of a hippo'
    # manual_prompt = 'a photo of a person'
    manual_prompt = ''
    use_general_concept = False
    # cfg_scales = np.arange(2.0,4.5, 0.5).tolist()
    cfg_scales = np.arange(3.0,3.5, 0.5).tolist()

    
    # steps = range(0, 3001, 200)
    steps = range(0, 1001, 100)
    # steps = range(50, 1001, 100)
    # steps = range(50, 1001, 100)
    # steps = range(50, 1001, 100)
    # steps = range(0, 501, 100)
    # steps = range(1000, 1001, 100)
    # steps = range(2000, 2001, 100)
    # steps = [6000,8000,10000]
    cfg_scales = [3.0,4.5,6.0,7.5]
    cfg_scales = [4.5,6.0,7.5]
    cfg_scales = [6.0,7.5]
    # steps = [50,100,150,200]
    # for step in steps:
    for step in steps:
    # for step in range(6000, 10001, 100):
    # for step in range(0, 10001, 100):
    # for step in [2000]:
    # for step in range(0, 3000+1, 100):
    # for step in range(300, 1001, 100):
        # for cfg in cfg_scales:
        is_original_pretrained_sd14 = exp_name == 'original_pretrained_sd1.4'
        is_original_pretrained_sd15 = exp_name == 'original_pretrained_sd1.5'
        is_original_rv = exp_name == 'original_realistic_vision'
        is_original_ch = exp_name == 'original_chilloutmix'
        is_relearn = ('uul' in exp_name) or ('erase' in exp_name)
        is_unlearn = 'ul' in exp_name and not 'uul' in exp_name
        if 'moodeng' in exp_name: concept = 'moodeng'
        if 'crybaby' in exp_name: concept = 'crybaby'
        if 'avp' in exp_name: concept = 'avp'
        if 'chiquita' in exp_name: concept = 'chiquita'
        if 'reese' in exp_name: concept = 'reese'
        if 'gout' in exp_name: concept = 'gout'
        if 'jooli' in exp_name: concept = 'jooli'
        if 'honer' in exp_name: concept = 'honer'
        if 'sceleb' in exp_name: concept = 'sceleb5g0'
        
        
        if is_original_pretrained_sd14 or 'sd14.' in exp_name:
            pretrained_path = 'CompVis/stable-diffusion-v1-4'
        if is_original_pretrained_sd15 or 'sd15.' in exp_name:
            pretrained_path = 'runwayml/stable-diffusion-v1-5'
        if is_original_rv or 'rv.' in exp_name:
            pretrained_path = 'stablediffusionapi/realistic-vision-v51'
        elif is_original_ch or 'ch.' in exp_name:
            pretrained_path = 'stablediffusionapi/chilloutmix'
        
        if is_relearn:
            erase_name = concept
            if 'VPr' in exp_name: erase_name += 'VPr'
            pretrained_path = f"data_root/logs/erase_l1.{erase_name}.object_lr2.5e-4/LoRA_fusion_model"
        if is_unlearn: 
            pretrained_path = f"data_root/logs/{exp_name}/LoRA_fusion_model"

        use_ti = '-V' in exp_name 
        # print(f"use_ti: {use_ti}")
        # print(exp_name)
        
        if 'V.r' in exp_name:
            initializer_token = ''
        elif use_ti:
            initializer_token = concept2initializer[concept]

        if manual_prompt:
            prompt = manual_prompt
        elif use_general_concept:
            prompt = concept2generalprompt[concept]
        
        
        elif use_ti:
            
            if 'sceleb' in concept:
                prompt = 'A photo of a v1,A photo of a v2,A photo of a v3,A photo of a v4,A photo of a v5'
                placeholder_token = 'v1,v2,v3,v4,v5'
            else:
                prompt = 'a photo of v1' 
                # prompt = 'A photo of a v1' 
                # prompt = 'v1' 
                
                placeholder_token = 'v1'
                
        
        else:
            if concept in concept2prompt:
                prompt = concept2prompt[concept]
            else: 
                prompt = 'sks person'
        
        if is_unlearn or 'erase' in exp_name or is_original_pretrained_sd14 or is_original_pretrained_sd15 or is_original_rv or is_original_ch: 
            load_lora_weight_path = ''
            gen_image_path = f"data_root/generated/model/{exp_name}"
        else:
            load_lora_weight_path =f"data_root/logs/{exp_name}/checkpoint-{step}"
            gen_image_path = 'auto'
            

        
        script = f"""
        accelerate launch train_dreambooth_lora.py \\
            --pretrained_model_name_or_path='{pretrained_path}'  \\
            --instance_data_dir="data_root/data/real_data/dummy" \\
            --load_lora_weight_path="{load_lora_weight_path}" \\
            --gen_image_path="{gen_image_path}" \\
            --output_dir="data_root/logs/gen" \\
            --validation_prompt="{prompt}" --instance_prompt="{prompt}" \\
            --lora_rank 1 --target_lora_modules to_k to_v --target_lora_layers cross \\
            --run_note 'gen img' --wait_weight \\
            --num_validation_images 50 \\"""
            
                
        if use_ti and not is_unlearn:
            script += f"""
            --load_token_embedding_path="data_root/logs/{exp_name}/checkpoint-{step}" \\
            --placeholder_token="{placeholder_token}" --initializer_token='{initializer_token}' \\"""

        if apply_negative_prompt:
            script += f"""
            --negative_prompt "longbody, lowres, bad anatomy, bad hands, missing fingers, extra digit, fewer digits, cropped, worst quality, low quality." \\"""
        
        # script += f"""
        #     --cfg_scale {cfg:.2f}"""
        script += f"""
            --cfg_scale {','.join(f'{x:.2f}' for x in cfg_scales)}"""
        print(script) 
        
        counter += 1
print(f"Total scripts generated: {counter}")
        
        


        accelerate launch train_dreambooth_lora.py \
            --pretrained_model_name_or_path='CompVis/stable-diffusion-v1-4'  \
            --instance_data_dir="data_root/data/real_data/dummy" \
            --load_lora_weight_path="data_root/logs/sd14.ct.l4.kv_asanteA5V0-V_pr1.00.neg_ln.lr1e-4.ti5e-4_b1g4/checkpoint-0" \
            --gen_image_path="auto" \
            --output_dir="data_root/logs/gen" \
            --validation_prompt="a photo of v1" --instance_prompt="a photo of v1" \
            --lora_rank 1 --target_lora_modules to_k to_v --target_lora_layers cross \
            --run_note 'gen img' --wait_weight \
            --num_validation_images 50 \
            --load_token_embedding_path="data_root/logs/sd14.ct.l4.kv_asanteA5V0-V_pr1.00.neg_ln.lr1e-4.ti5e-4_b1g4/checkpoint-0" \
            --placeholder_token="v1" --initializer_token='person' \
            --negative_prompt "longbody, lowres, bad anatomy, bad hands, missing fingers, extra digit, fewer digits, cropped,

# Generation

In [ ]:
 # Stereo-UCE-ESD + MACE (test only unlearn) Generation # REFINE
exp_names = ['rlct4.reV.sophiewildeA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_stereo.edsheeran_sd1.4', 'rlct4.reV.sophiewildeA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_stereo.chemsworth_sd1.4', 'rlct4.reV.sophiewildeA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_stereo.ahathaway_sd1.4', 'rlct4.reV.edebiriA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_stereo.edsheeran_sd1.4', 'rlct4.reV.edebiriA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_stereo.chemsworth_sd1.4', 'rlct4.reV.edebiriA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_stereo.mcarey_sd1.4', 'rlct4.reV.mmadisonA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_stereo.obama_sd1.4', 'rlct4.reV.mmadisonA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_stereo.ahathaway_sd1.4', 'rlct4.reV.mmadisonA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_stereo.octavia_sd1.4', 'rlct4.reV.nicoparkerA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_stereo.mrobbie_sd1.4', 'rlct4.reV.nicoparkerA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_stereo.chemsworth_sd1.4', 'rlct4.reV.nicoparkerA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_stereo.aadam_sd1.4']
exp_names = ['sd1.4']
# # exp_names = ['rlct4.reV.obamaA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000.r2_uce.obama_sd1.4', 'rlct4.reV.rihannaA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000.r2_uce.rihanna_sd1.4', 'rlct4.reV.edsheeranA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000.r2_uce.edsheeran_sd1.4', 'rlct4.reV.mrobbieA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000.r2_uce.mrobbie_sd1.4', 'rlct4.reV.chemsworthA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000.r2_uce.chemsworth_sd1.4', 'rlct4.reV.cevansA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000.r2_uce.cevans_sd1.4', 'rlct4.reV.aadamA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000.r2_uce.aadam_sd1.4', 'rlct4.reV.ahathawayA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000.r2_uce.ahathaway_sd1.4', 'rlct4.reV.mcareyA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000.r2_uce.mcarey_sd1.4', 'rlct4.reV.octaviaA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000.r2_uce.octavia_sd1.4', 'rlct4.reV.morganfA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000.r2_uce.morganf_sd1.4', 'rlct4.reV.drakeA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000.r2_uce.drake_sd1.4']
# exp_names = ['rlct4.reV.asanteA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-u.edsheeran_sd1.4', 'rlct4.reV.asanteA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_esd-u.edsheeran_sd1.4', 'rlct4.reV.asanteA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_esd-u.aadam_sd1.4', 'rlct4.reV.reeseA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-u.rihanna_sd1.4', 'rlct4.reV.reeseA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_esd-u.mrobbie_sd1.4', 'rlct4.reV.reeseA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_esd-u.morganf_sd1.4', 'rlct4.reV.nivolaA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-u.drake_sd1.4', 'rlct4.reV.nivolaA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_esd-u.octavia_sd1.4', 'rlct4.reV.nivolaA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_esd-u.aadam_sd1.4', 'rlct4.reV.earleA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-u.rihanna_sd1.4', 'rlct4.reV.earleA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_esd-u.obama_sd1.4', 'rlct4.reV.earleA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_esd-u.rihanna_sd1.4', 'rlct4.reV.leowoodalA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-u.octavia_sd1.4', 'rlct4.reV.leowoodalA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_esd-u.obama_sd1.4', 'rlct4.reV.leowoodalA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_esd-u.obama_sd1.4', 'rlct4.reV.starkeyA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-u.octavia_sd1.4', 'rlct4.reV.starkeyA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_esd-u.mrobbie_sd1.4', 'rlct4.reV.starkeyA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_esd-u.chemsworth_sd1.4', 'rlct4.reV.apierreA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-u.obama_sd1.4', 'rlct4.reV.apierreA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_esd-u.obama_sd1.4', 'rlct4.reV.apierreA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_esd-u.chemsworth_sd1.4', 'rlct4.reV.skyhblackA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-u.rihanna_sd1.4', 'rlct4.reV.skyhblackA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_esd-u.ahathaway_sd1.4', 'rlct4.reV.skyhblackA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_esd-u.mrobbie_sd1.4', 'rlct4.reV.sophiewildeA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-u.edsheeran_sd1.4', 'rlct4.reV.sophiewildeA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_esd-u.chemsworth_sd1.4', 'rlct4.reV.sophiewildeA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_esd-u.ahathaway_sd1.4', 'rlct4.reV.edebiriA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-u.edsheeran_sd1.4', 'rlct4.reV.edebiriA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_esd-u.chemsworth_sd1.4', 'rlct4.reV.edebiriA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_esd-u.mcarey_sd1.4', 'rlct4.reV.mmadisonA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-u.obama_sd1.4', 'rlct4.reV.mmadisonA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_esd-u.ahathaway_sd1.4', 'rlct4.reV.mmadisonA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_esd-u.octavia_sd1.4', 'rlct4.reV.nicoparkerA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-u.mrobbie_sd1.4', 'rlct4.reV.nicoparkerA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_esd-u.chemsworth_sd1.4', 'rlct4.reV.nicoparkerA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_esd-u.aadam_sd1.4']
# exp_names = ['rlct4.reV.obamaA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-u.obama_sd1.4', 'rlct4.reV.rihannaA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-u.rihanna_sd1.4', 'rlct4.reV.edsheeranA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-u.edsheeran_sd1.4', 'rlct4.reV.mrobbieA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-u.mrobbie_sd1.4', 'rlct4.reV.chemsworthA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-u.chemsworth_sd1.4', 'rlct4.reV.cevansA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-u.cevans_sd1.4', 'rlct4.reV.cevansA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-u.cevans_sd1.4', 'rlct4.reV.cevansA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-u.cevans_sd1.4', 'rlct4.reV.aadamA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-u.aadam_sd1.4', 'rlct4.reV.ahathawayA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-u.ahathaway_sd1.4', 'rlct4.reV.ahathawayA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-u.ahathaway_sd1.4', 'rlct4.reV.ahathawayA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-u.ahathaway_sd1.4', 'rlct4.reV.mcareyA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-u.mcarey_sd1.4', 'rlct4.reV.octaviaA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-u.octavia_sd1.4', 'rlct4.reV.octaviaA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-u.octavia_sd1.4', 'rlct4.reV.morganfA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-u.morganf_sd1.4', 'rlct4.reV.drakeA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-u.drake_sd1.4', 'rlct4.reV.drakeA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-u.drake_sd1.4']
# # exp_names = ['rlct4.reV.obamaA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-x.obama_sd1.4', 'rlct4.reV.rihannaA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-x.rihanna_sd1.4', 'rlct4.reV.edsheeranA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-x.edsheeran_sd1.4', 'rlct4.reV.mrobbieA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-x.mrobbie_sd1.4', 'rlct4.reV.chemsworthA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-x.chemsworth_sd1.4', 'rlct4.reV.cevansA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-x.cevans_sd1.4', 'rlct4.reV.cevansA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-x.cevans_sd1.4', 'rlct4.reV.cevansA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-x.cevans_sd1.4', 'rlct4.reV.aadamA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-x.aadam_sd1.4', 'rlct4.reV.ahathawayA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-x.ahathaway_sd1.4', 'rlct4.reV.ahathawayA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-x.ahathaway_sd1.4', 'rlct4.reV.ahathawayA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-x.ahathaway_sd1.4', 'rlct4.reV.mcareyA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-x.mcarey_sd1.4', 'rlct4.reV.octaviaA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-x.octavia_sd1.4', 'rlct4.reV.octaviaA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-x.octavia_sd1.4', 'rlct4.reV.morganfA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-x.morganf_sd1.4', 'rlct4.reV.drakeA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-x.drake_sd1.4', 'rlct4.reV.drakeA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-x.drake_sd1.4']
# exp_names = ['rlct4.reV.obamaA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_stereo.obama_sd1.4', 'rlct4.reV.rihannaA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_stereo.rihanna_sd1.4', 'rlct4.reV.edsheeranA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_stereo.edsheeran_sd1.4', 'rlct4.reV.mrobbieA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_stereo.mrobbie_sd1.4', 'rlct4.reV.chemsworthA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_stereo.chemsworth_sd1.4', 'rlct4.reV.cevansA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_stereo.cevans_sd1.4', 'rlct4.reV.aadamA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_stereo.aadam_sd1.4', 'rlct4.reV.ahathawayA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_stereo.ahathaway_sd1.4', 'rlct4.reV.mcareyA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_stereo.mcarey_sd1.4', 'rlct4.reV.octaviaA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_stereo.octavia_sd1.4', 'rlct4.reV.morganfA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_stereo.morganf_sd1.4', 'rlct4.reV.drakeA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_stereo.drake_sd1.4']

# # exp_names = [exp_names[0]]

# exp_names = ['uce.edsheeran_sd1.4']
# exp_names = exp_names[8:]

# MACE
exp_names = ['rlct4.reV.obamaA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_ul1.prg1e-4d8e+3.lr1e-4.n8.G.obama.person.s50_sd14', 'rlct4.reV.rihannaA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_ul1.prg1e-4d8e+3.lr1e-4.n8.G.rihanna.person.s50_sd14', 'rlct4.reV.edsheeranA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_ul1.prg1e-4d8e+3.lr1e-4.n8.G.edsheeran.person.s50_sd14', 'rlct4.reV.mrobbieA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_ul1.prg1e-4d8e+3.lr1e-4.n8.G.mrobbie.person.s50_sd14', 'rlct4.reV.chemsworthA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_ul1.prg1e-4d8e+3.lr1e-4.n8.G.chemsworth.person.s50_sd14', 'rlct4.reV.cevansA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_ul1.prg1e-4d8e+3.lr1e-4.n8.G.cevans.person.s50_sd14', 'rlct4.reV.aadamA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_ul1.prg1e-4d8e+3.lr1e-4.n8.G.aadam.person.s50_sd14', 'rlct4.reV.ahathawayA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_ul1.prg1e-4d8e+3.lr1e-4.n8.G.ahathaway.person.s50_sd14', 'rlct4.reV.mcareyA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_ul1.prg1e-4d8e+3.lr1e-4.n8.G.mcarey.person.s50_sd14', 'rlct4.reV.octaviaA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_ul1.prg1e-4d8e+3.lr1e-4.n8.G.octavia.person.s50_sd14', 'rlct4.reV.morganfA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_ul1.prg1e-4d8e+3.lr1e-4.n8.G.morganf.person.s50_sd14', 'rlct4.reV.drakeA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_ul1.prg1e-4d8e+3.lr1e-4.n8.G.drake.person.s50_sd14']


apply_only_unlearn = True
apply_use_ti = True
apply_negative_prompt = True

manual_prompt = ''
if apply_only_unlearn or exp_names[0] == 'sd1.4':
    # manual_prompt = '*unlearned'
    # manual_prompt = '*nonunlearned'
    manual_prompt = '*seen'
# manual_prompt = 'a photo of obama'
# manual_prompt = '*unlearned'
use_general_concept = False

# exp_names = [exp_names[0]]
cfg_scales = [  4.5, 6.0]
cfg_scales = [7.5]


count = 0
for exp_name in exp_names:

    is_original_pretrained = ( exp_name == 'CompVis/stable-diffusion-v1-4' or exp_name == 'sd1.4' )


    is_relearn = 'rl' in exp_name 
    if is_relearn:
        base_exp_name = '_'.join(exp_name.split('_')[2:])
        relearn_exp_name = exp_name
        unlearn_exp_name = '_'.join(exp_name.split('_')[1:])
        exp_name = base_exp_name

    is_unlearn =  apply_only_unlearn or (not is_relearn and  any(k in exp_name for k in ('ul','uce', 'esd', 'stereo')))
    force_unlearn = is_relearn and apply_only_unlearn
    is_relearn = is_relearn and not force_unlearn
    
    
    # print(is_relearn,is_unlearn)
    # steps =range(0, 3000+1, 200)
    # steps =range(0, 1000+1, 100)
    # steps =range(500, 500+1, 100)
    steps =range(0, 1000+1, 100)

    if 's3000' in relearn_exp_name:
        steps = range(0, 3000+1, 200)
        
    
    if is_unlearn or is_original_pretrained:
        steps = [0]

    for step in steps:
        
        pretrained_path = 'CompVis/stable-diffusion-v1-4'
        # if is_relearn:
            # pretrained_path = f"data_root/logs/{unlearn_exp_name}/LoRA_fusion_model"
            # pretrained_path = 'CompVis/stable-diffusion-v1-4'
        for c in seen_concepts:
            # print(c)
            if c in data_info and c in unlearn_exp_name:
                concept = c
                # print("Found concept:", concept)
                break
        # print(c,concept)
        # print(concept)
        concept_name = data_info[concept]['Full_name']
        concept_name_ = concept_name.replace(' ','_')
            
        unlearning_methods = ['ul1', 'mace','esd-x','esd-u','esd-all','uce','stereo',]
        for _unlearn_method in unlearning_methods:
            if _unlearn_method in unlearn_exp_name:
                unlearn_method = _unlearn_method
                break
        # print(f'unlearning method: {unlearn_method}')

        if unlearn_method == 'ul1' or unlearn_method == 'mace':
            unet_weight_path = ""
            pretrained_path = f"data_root/logs/{unlearn_exp_name}/LoRA_fusion_model"

        if unlearn_method == 'esd-x':

                pretrained_unet_name = f"esd-{concept_name_}-from-{concept_name_}-esdx"
                unet_weight_path = f"data_root/logs/esd/sd1.4/{pretrained_unet_name}.safetensors"
        if unlearn_method == 'esd-u':
                pretrained_unet_name = f"esd-{concept_name_}-from-{concept_name_}-esdu"
                unet_weight_path = f"data_root/logs/esd/sd1.4/{pretrained_unet_name}.safetensors"
        if unlearn_method == 'esd-all':
                pretrained_unet_name = f"esd-{concept_name_}-from-{concept_name_}-esdall"
                unet_weight_path = f"data_root/logs/esd/sd1.4/{pretrained_unet_name}.safetensors"         
        if unlearn_method == 'stereo':
            unet_weight_path = f"data_root/logs/stereo/{data_info[concept]['Full_name']}/final_reo_unet.pt"
        elif unlearn_method == 'uce':
            unet_weight_path = f"data_root/logs/uce/{concept}_uce_sd.safetensors"
            
        # if is_unlearn: 
        #     pretrained_path = f"data_root/logs/{exp_name}/LoRA_fusion_model"

        use_ti = 'ti' in relearn_exp_name or '-V' in exp_name 
        
        if is_relearn and not 'reV' in relearn_exp_name:
            # relearn is not re-initializing the token (by default)
            initializer_token = ''
        elif use_ti:
            initializer_token = concept2initializer[concept]

        if manual_prompt:
            if manual_prompt == '*seen' or manual_prompt == '*unlearned' or manual_prompt == '*nonunlearned':
                
                for seen_concept in seen_concepts:
                    if seen_concept in unlearn_exp_name:
                        unlearned_concept = seen_concept
                        break
                        
                if  manual_prompt == '*unlearned':
                    prompt = f"a photo of {data_info[unlearned_concept]['Full_name']}"
                elif manual_prompt == '*nonunlearned':
                    non_unlearned_prompts = [ f"a photo of {data_info[seen_concept]['Full_name']}" for seen_concept in seen_concepts if seen_concept!=unlearned_concept]
                    prompt = ';'.join(non_unlearned_prompts)
                elif manual_prompt == '*seen':
                    prompts = [f"a photo of {data_info[seen_concept]['Full_name']}" for seen_concept in seen_concepts]
                    prompt = ';'.join(prompts)
            else:
                prompt = manual_prompt
        # elif use_general_concept:
        #     prompt = concept2generalprompt[concept]
        
        elif use_ti:
            if 'sceleb' in concept:
                prompt = 'A photo of a v1,A photo of a v2,A photo of a v3,A photo of a v4,A photo of a v5'
                placeholder_token = 'v1,v2,v3,v4,v5'
            else:
                prompt = 'a photo of v1' 
                placeholder_token = 'v1'
        else:
            prompt = concept2prompt[concept]
            
        ## hacky .. should change this later
        if is_relearn:
            exp_name = relearn_exp_name
            load_lora_weight_path =f"data_root/logs/{exp_name}/checkpoint-{step}"
            gen_image_path = 'auto'
        # if is_unlearn or 'erase' in exp_name or exp_name == 'original_pretrained': 
        #     load_lora_weight_path = ''
            # gen_image_path = f"data_root/generated/model/{exp_name}"
        elif is_unlearn:
            # load_lora_weight_path =f"data_root/logs/{exp_name}/checkpoint-{step}"
            load_lora_weight_path = ''
            gen_image_path = 'auto'
            if force_unlearn:
                gen_image_path = f"data_root/generated/model/{unlearn_exp_name}"
        elif is_original_pretrained:
            load_lora_weight_path = ''
            gen_image_path = 'data_root/generated/model/original_pretrained_sd1.4'
            unet_weight_path = ''


        # if 'l0' in exp_name :
        #     load_lora_weight_path = ''
        
        script = f"""
        accelerate launch train_dreambooth_lora.py \\
            --pretrained_model_name_or_path='{pretrained_path}'  \\
            --load_unet_weight_path="{unet_weight_path}" \\
            --load_lora_weight_path="{load_lora_weight_path}" \\
            --instance_data_dir="data_root/data/real_data/dummy" \\
            --gen_image_path="{gen_image_path}" \\
            --output_dir="data_root/logs/gen" \\
            --validation_prompt="{prompt}" --instance_prompt="{prompt}" \\
            --lora_rank 1 --target_lora_modules to_k to_v --target_lora_layers cross \\
            --run_note 'gen img' --wait_weight \\
            --num_validation_images 50 \\"""


        if use_ti and not is_unlearn and  not is_original_pretrained:
            script += f"""
            --load_token_embedding_path="data_root/logs/{exp_name}/checkpoint-{step}" \\
            --placeholder_token="{placeholder_token}" --initializer_token='{initializer_token}' \\"""

        if apply_negative_prompt:
            script += f"""
            --negative_prompt "longbody, lowres, bad anatomy, bad hands, missing fingers, extra digit, fewer digits, cropped, worst quality, low quality." \\"""
        

        script += f"""
            --cfg_scale {','.join(f'{x:.2f}' for x in cfg_scales)}"""

        print(f"echo 'count:{count} - {exp_name} {step} /'")
        print(script) 
        
        count += 1
print(f"Total scripts generated: {count}")
        

echo 'count:0 - sd14 0 /'

       accelerate launch train_dreambooth_lora.py \
           --pretrained_model_name_or_path='data_root/logs/ul1.prg1e-4d8e+3.lr1e-4.n8.G.obama.person.s50_sd14/LoRA_fusion_model'  \
           --load_unet_weight_path="" \
           --load_lora_weight_path="" \
           --instance_data_dir="data_root/data/real_data/dummy" \
           --gen_image_path="data_root/generated/model/ul1.prg1e-4d8e+3.lr1e-4.n8.G.obama.person.s50_sd14" \
           --output_dir="data_root/logs/gen" \
           --validation_prompt="a photo of Barrack Obama;a photo of Rihanna;a photo of Ed Sheeran;a photo of Margot Robbie;a photo of Chris Hemsworth;a photo of Chris Evans;a photo of Anne Adam;a photo of Anne Hathaway;a photo of Mariah Carey;a photo of Octavia Spencer;a photo of Morgan Freeman;a photo of Drake" --instance_prompt="a photo of Barrack Obama;a photo of Rihanna;a photo of Ed Sheeran;a photo of Margot Robbie;a photo of Chris Hemsworth;a photo of Chris Evans;a photo of An

# Unlearning

In [25]:
# STEREO
## Generate anchor prompts for ChatGPT
# for seen_concept in seen_concepts[11:12]:
#     target_concept = data_info[seen_concept]['Full_name']
#     anchor_prompt= f"""Generate a total of exactly 200 sentences that contain the word '{target_concept}', where each sentence represents a diverse and factually correct background where '{target_concept}' will appear. Ensure each sentence contextually captures the usage of the word '{target_concept}' and that each sentence is unique.
#      give me a this format:

#      "{target_concept}": [
#          "str sentence" seperated by ','
#      ]

#      do make sure that it's exactly 200 sentences (if necessary you should write a code to check the len the list)
#      you can give me .txt file 
#      don't cheat by doing very similar sentences, and adding numbers to them"
#     """
#     print(anchor_prompt)
    


# Generate Gallery images
# for seen_concept in seen_concepts:
#      target_concept = data_info[seen_concept]['Full_name']
#      target_prompt = 'A photo of ' + target_concept
#      script = f"""python generate_images.py  --prompt "{target_prompt}" --output_dir "../data_root/generated/stereo/{target_prompt}/"  --num_images 500 """
#      print(script)
# # python generate_images.py --output_dir "../data_root/generated/stereo/A photo of Barrack Obama/" --prompt "A photo of Barrack Obama" --num_images 500
    

# Remove concepts
for seen_concept in seen_concepts:
    if seen_concept in seen_concepts_1: continue
    target_concept = data_info[seen_concept]['Full_name']
    script = f"""python -W ignore train.py --erase_concept '{target_concept}' --train_method noxattn --train_data_dir "../data_root/generated/stereo/A photo of {target_concept}/" --learnable_property 'object' --initializer_token 'person' --output_dir "../data_root/logs/stereo/{target_concept}" --mode stereo --unet_ckpt_to_attack final_reo_unet.pt --attack_eval_images  "../data_root/generated/stereo/A photo of {target_concept}/" --compositional_guidance_scale 2 --n_iterations 2 --num_of_adv_concepts 2   --anchor_concept_path utils/person_anchor_prompts.json"""
    
    print(script)

python -W ignore train.py --erase_concept 'Adam Driver' --train_method noxattn --train_data_dir "../data_root/generated/stereo/A photo of Adam Driver/" --learnable_property 'object' --initializer_token 'person' --output_dir "../data_root/logs/stereo/Adam Driver" --mode stereo --unet_ckpt_to_attack final_reo_unet.pt --attack_eval_images  "../data_root/generated/stereo/A photo of Adam Driver/" --compositional_guidance_scale 2 --n_iterations 2 --num_of_adv_concepts 2   --anchor_concept_path utils/person_anchor_prompts.json
python -W ignore train.py --erase_concept 'Andrew Garfield' --train_method noxattn --train_data_dir "../data_root/generated/stereo/A photo of Andrew Garfield/" --learnable_property 'object' --initializer_token 'person' --output_dir "../data_root/logs/stereo/Andrew Garfield" --mode stereo --unet_ckpt_to_attack final_reo_unet.pt --attack_eval_images  "../data_root/generated/stereo/A photo of Andrew Garfield/" --compositional_guidance_scale 2 --n_iterations 2 --num_of_adv_

In [13]:
# relearning (STEREO)
# decoding unlearning - with same hyperparameter

# Implement text encoder relearning

# ul_exp_names = ['ul1.prg1e-4d5e-4.lr1e-4.n8.G.sceleb5g0.person.s50_c.l16.kv_sceleb5g0N50-V_pr0.50_lr5e-5.ti5e-4_f0.5_b4g4.s10000']

# ul_exp_names = ['ul1.prg1e-4d8e+3.lr1e-4.n8.G.mcarey.person.s50_sd14', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.mcarey.person.s50.r1_sd14', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.mcarey.person.s50.r2_sd14', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.octavia.person.s50_sd14', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.octavia.person.s50.r1_sd14', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.octavia.person.s50.r2_sd14', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.oprah.person.s50_sd14', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.oprah.person.s50.r1_sd14', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.oprah.person.s50.r2_sd14', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.morganf.person.s50_sd14', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.morganf.person.s50.r1_sd14', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.morganf.person.s50.r2_sd14', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.drake.person.s50_sd14', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.drake.person.s50.r1_sd14', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.drake.person.s50.r2_sd14', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.idris.person.s50_sd14', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.idris.person.s50.r1_sd14', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.idris.person.s50.r2_sd14']


# ul_exp_names = ['esd-Barrack_Obama-from-Barrack_Obama-esdx', 'esd-Barrack_Obama-from-Barrack_Obama-esdu', 'esd-Barrack_Obama-from-Barrack_Obama-esdall', 'esd-Rihanna-from-Rihanna-esdx', 'esd-Rihanna-from-Rihanna-esdu', 'esd-Rihanna-from-Rihanna-esdall', 'esd-Ed_Sheeran-from-Ed_Sheeran-esdx', 'esd-Ed_Sheeran-from-Ed_Sheeran-esdu', 'esd-Ed_Sheeran-from-Ed_Sheeran-esdall', 'esd-Margot_Robbie-from-Margot_Robbie-esdx', 'esd-Margot_Robbie-from-Margot_Robbie-esdu', 'esd-Margot_Robbie-from-Margot_Robbie-esdall', 'esd-Chris_Hemsworth-from-Chris_Hemsworth-esdx', 'esd-Chris_Hemsworth-from-Chris_Hemsworth-esdu', 'esd-Chris_Hemsworth-from-Chris_Hemsworth-esdall', 'esd-Chris_Evans-from-Chris_Evans-esdx', 'esd-Chris_Evans-from-Chris_Evans-esdu', 'esd-Chris_Evans-from-Chris_Evans-esdall', 'esd-Adam_Driver-from-Adam_Driver-esdx', 'esd-Adam_Driver-from-Adam_Driver-esdu', 'esd-Adam_Driver-from-Adam_Driver-esdall', 'esd-Andrew_Garfield-from-Andrew_Garfield-esdx', 'esd-Andrew_Garfield-from-Andrew_Garfield-esdu', 'esd-Andrew_Garfield-from-Andrew_Garfield-esdall', 'esd-Anne_Adam-from-Anne_Adam-esdx', 'esd-Anne_Adam-from-Anne_Adam-esdu', 'esd-Anne_Adam-from-Anne_Adam-esdall', 'esd-Anne_Hathaway-from-Anne_Hathaway-esdx', 'esd-Anne_Hathaway-from-Anne_Hathaway-esdu', 'esd-Anne_Hathaway-from-Anne_Hathaway-esdall', 'esd-Angelina_Jolie-from-Angelina_Jolie-esdx', 'esd-Angelina_Jolie-from-Angelina_Jolie-esdu', 'esd-Angelina_Jolie-from-Angelina_Jolie-esdall', 'esd-Amber_Heard-from-Amber_Heard-esdx', 'esd-Amber_Heard-from-Amber_Heard-esdu', 'esd-Amber_Heard-from-Amber_Heard-esdall', 'esd-Mariah_Carey-from-Mariah_Carey-esdx', 'esd-Mariah_Carey-from-Mariah_Carey-esdu', 'esd-Mariah_Carey-from-Mariah_Carey-esdall', 'esd-Octavia_Spencer-from-Octavia_Spencer-esdx', 'esd-Octavia_Spencer-from-Octavia_Spencer-esdu', 'esd-Octavia_Spencer-from-Octavia_Spencer-esdall', 'esd-Oprah_Winfrey-from-Oprah_Winfrey-esdx', 'esd-Oprah_Winfrey-from-Oprah_Winfrey-esdu', 'esd-Oprah_Winfrey-from-Oprah_Winfrey-esdall', 'esd-Morgan_Freeman-from-Morgan_Freeman-esdx', 'esd-Morgan_Freeman-from-Morgan_Freeman-esdu', 'esd-Morgan_Freeman-from-Morgan_Freeman-esdall', 'esd-Drake-from-Drake-esdx', 'esd-Drake-from-Drake-esdu', 'esd-Drake-from-Drake-esdall', 'esd-Idris_Elba-from-Idris_Elba-esdx', 'esd-Idris_Elba-from-Idris_Elba-esdu', 'esd-Idris_Elba-from-Idris_Elba-esdall']

# ul_exp_names = [ul_exp_name for ul_exp_name in ul_exp_names if 'esdu' in ul_exp_name][12:] # filter out the esd ones
# # ul_exp_names = ['esd-Barrack_Obama-from-Barrack_Obama-esdx', 'esd-Barrack_Obama-from-Barrack_Obama-esdu', 'esd-Barrack_Obama-from-Barrack_Obama-esdall']

ul_exp_names = list(seen_concepts)
# ul_exp_names = [exp for exp in ul_exp_names if  not 'ahathaway' in exp]
# ul_exp_names = [exp for exp in ul_exp_names if  not 'rihanna' in exp]

learn_concept = "" # 'moodeng' # 'crybaby' # 'avp' # 'chiquita' # 'reese' # 'gout' # 'jooli' # 'honer' # 'sceleb5g0'
pretrained = 'sd1.4'



#seed = 0

# seeds = [0,1,2] # along

# hacked 
# seeds = [999]*len(ul_exp_names)
seeds = [0]
# seeds = [3,4]
# seeds = [0,1,2,3,4]

use_te = True
batch_size = 1
gradient_accumulation_step = 4
# re
# lr_lora_grid = ["1e-4", "5e-5", "1e-5"]
lr_lora_grid = ["1e-4"]
# lr_lora_grid = [ "1e-5"]
lr_ti_grid = ["5e-4"]   # only used if use_ti
# lr_ti_grid   = ["5e-3","5e-2"]   # only used if use_ti
lr_te_grid = ["1e-5"] 
lora_ranks = [4] # [1,2,4,8,16,32,64]
# # Create all combinations of lr_lora, lr_ti, and seed
combos = list(itertools.product(lr_lora_grid,
                                lr_ti_grid,
                                lr_te_grid if use_te else [None],
                                lora_ranks))

use_pr = True

use_te = True
apply_use_ti = True

lr_scheduler = 'linear' # 'linear' # 'cosine' # 'cosine_with_restarts'
apply_negative_prompt = True

use_manual_params = True
reV = True
is_relearn = True 

# fix here #
manual_params = {
    # 'data_setting': 'facefew',
    'data_setting': 'align',
    
    # 'data_setting': 'small',s
    # 'lora_rank' :  16
}

data_setting = 'align' 

final_exp_names = []
# for ul_exp_name in ul_exp_names:
for ul_exp_name in ul_exp_names:
    
    if not learn_concept:
        for c in seen_concepts:
            if c in data_info and c in ul_exp_name:
                concept = c
                break
    else:
        concept = learn_concept

    train_method = 'stereo'
    # if 'esdx' in ul_exp_name:
    #     train_method = 'esd-x'
    # elif 'esdu' in ul_exp_name:
    #     train_method = 'esd-u'
    # elif 'esdall' in ul_exp_name:
    #     train_method = 'esd-all'

    pretrained_unet_name = ul_exp_name 
    ul_exp_name = f'{train_method}.{concept}_sd1.4'
    
    
    base_exp_name = '_'.join(ul_exp_name.split('_')[1:])
    exp_name = base_exp_name

        
    for seed in seeds:


                
        if seed == 999:
            if '.r1' in ul_exp_name:
                seed = 1
            elif '.r2' in ul_exp_name:
                seed = 2    
            else:
                seed = 0
        

        # print(f"Base Experiment Name: {base_exp_name}")
        
        #fix edit here : they are using this to reconstruct the base_exp as well
        lr_lora, lr_ti = extract_lrs(base_exp_name)
        lora_rank = extract_learning_lora_rank(base_exp_name)
        
        # todo: better use 're'
        for re_lr_lora, re_lr_ti, re_lr_te, re_lora_rank in combos:
            
            # print(f'lora_rank: {lora_rank}, lr: {lr_lora}, ti_lr: {lr_ti}')
            # pretrained_path = f"data_root/logs/{ul_exp_name}/LoRA_fusion_model"
            pretrained_path = 'CompVis/stable-diffusion-v1-4'
            unet_weight_path = f"data_root/logs/stereo/{data_info[concept]['Full_name']}/final_reo_unet.pt"
            # print(f"Concept: {concept}"

            use_ti = 'ti' in exp_name or '-V' in exp_name or apply_use_ti
            # use_pr = 'pr' in exp_name

            if data_setting == 'facefew':
                dataset_name = f'{concept}5F0r{seed}'
                
            elif data_setting == 'align':
                dataset_name = f'{concept}A5V0'
            elif data_setting == 'fewshot':
                
                if 'sceleb' in concept:
                    dataset_name = f'{concept}U3'
                elif concept == 'avp':
                    dataset_name = 'avpS3'
                else:
                    dataset_name = f'{concept}U3'
            elif data_setting == 'small':
                
                if 'sceleb' in concept:
                    dataset_name = f'{concept}N10'
                else:
                    dataset_name = f'{concept}10'
            else:
                
                if 'sceleb' in concept:
                    dataset_name = f'{concept}N50'
                elif concept == 'avp':
                    dataset_name = 'avp20'
                else:
                    dataset_name = f'{concept}50'
                    
            if reV:
                initializer_token = concept2initializer[concept]
            else: 
                initializer_token = ''
                
            if use_ti:
                if 'sceleb' in concept:
                    prompt = 'A photo of a v1,A photo of a v2,A photo of a v3,A photo of a v4,A photo of a v5'
                    placeholder_token = 'v1,v2,v3,v4,v5'
                else:
                    prompt = 'a photo of v1' 
                    placeholder_token = 'v1'
            else:
                prompt = concept2prompt[concept]

            name_tag = ''
            if is_relearn: name_tag += 'uul'
            name_tag = f'{name_tag} {dataset_name}'
            name_tag += f' l{lora_rank}'
            if use_ti: 
                # name_tag += f' ti.{lr_ti}'
                name_tag += f' ti'

            data_root = dataset_name2data_root[dataset_name]
            
            if use_ti:
                dataset_name_for_exp = dataset_name + "-V"
                # if use_ni:
                #     dataset_name_for_exp += ".ni"
            else: dataset_name_for_exp = dataset_name
            
            
            # prior preservation folder
            prior_folder = 'original_realistic_vision'
            if pretrained == 'sd1.5':
                prior_folder = 'original_pretrained_sd1.5'
            if pretrained == 'sd1.4':
                prior_folder = 'original_pretrained_sd1.4'     
            if pretrained == 'rv':
                prior_folder = 'original_realistic_vision'
            elif pretrained == 'ch':
                prior_folder = 'original_chilloutmix'

            
            # renaming to check
            # re_exp_name = f'c.l{lora_rank}.kv_{dataset_name_for_exp}'
            # if use_pr:
            #     re_exp_name += f'_pr0.50'
            # re_exp_name += '_lr'
            # if lora_rank >0: re_exp_name += f"{str(lr_lora)}"
            # if use_ti:
            #     re_exp_name += f'.ti{str(lr_ti)}'
            # re_exp_name += '_f0.5_b1g4'
            
            # print(re_exp_name)
            # assert re_exp_name in base_exp_name, f"Expected {re_exp_name} in {base_exp_name}"

            # if manual_lora is not None and manual_data != lora_rank:
            
            if use_manual_params:
                lora_rank = re_lora_rank
                eff_data_setting = manual_params['data_setting']

                lr_lora, lr_ti = re_lr_lora, re_lr_ti
                
                if eff_data_setting == 'facefew':
                    dataset_name = f'{concept}5F0r{seed}'
                    
                elif eff_data_setting == 'align':
                    dataset_name = f'{concept}A5V0'
                
                elif eff_data_setting == 'fewshot':
                    
                    if 'sceleb' in concept:
                        dataset_name = f'{concept}U3'
                    elif concept == 'avp':
                        dataset_name = 'avpS3'
                    else:
                        dataset_name = f'{concept}U3'
                elif eff_data_setting == 'small':
                    
                    if 'sceleb' in concept:
                        dataset_name = f'{concept}N10'
                    else:
                        dataset_name = f'{concept}10'
                else:
                    
                    if 'sceleb' in concept:
                        dataset_name = f'{concept}N50'
                    elif concept == 'avp':
                        dataset_name = 'avp20'
                    else:
                        dataset_name = f'{concept}50'            
                        
                data_root = dataset_name2data_root[dataset_name]
            
            
            
            
            if reV:
                if use_te:
                    relearn_exp_name = f"rlct{lora_rank}.reV.{dataset_name}"
                else:
                    relearn_exp_name = f"rlc{lora_rank}.reV.{dataset_name}"
            else:
                if use_te:
                    relearn_exp_name = f"rlct{lora_rank}.{dataset_name}"
                else:
                    relearn_exp_name = f"rlc{lora_rank}.{dataset_name}"
            
            
            if lr_scheduler == 'linear':
                relearn_exp_name += f".ln"
            relearn_exp_name += f".lr{re_lr_lora}.ti{re_lr_ti}"
                
                
            if use_pr:
                relearn_exp_name += f".pr1.00"
                if apply_negative_prompt:
                    relearn_exp_name += ".neg"
            
            relearn_exp_name += f".b{batch_size}g{gradient_accumulation_step}"
            
            if seed != 0:
                relearn_exp_name += f".r{seed}"

            final_exp_name = f"{relearn_exp_name}_{ul_exp_name}"
            
            
            script = f"""
            accelerate launch train_dreambooth_lora.py \\
            --pretrained_model_name_or_path="{pretrained_path}"  \\
            --load_unet_weight_path="{unet_weight_path}" \\
            --instance_data_dir="{data_root}" \\
            --output_dir="data_root/logs/{final_exp_name}" \\
            --validation_prompt="{prompt}" --instance_prompt="{prompt}" \\
            --train_batch_size={batch_size} --gradient_accumulation_steps={gradient_accumulation_step} \\
            --lora_rank {lora_rank} --target_lora_modules to_k to_v --target_lora_layers cross \\
            --max_train_steps=1000  --validation_steps=50  --checkpointing_steps=50  --lr_scheduler "{lr_scheduler}"  --seed {seed} \\
            --run_note '{name_tag}' \\"""
                
            script += f"""
            --cfg_scale 6.0 \\"""
        
        
            if apply_negative_prompt:
                script += f"""
            --negative_prompt "longbody, lowres, bad anatomy, bad hands, missing fingers, extra digit, fewer digits, cropped, worst quality, low quality." \\"""
                
                
            if use_pr:
                if pretrained == 'sd1.4':
                    cfg_pr = 7.5
                elif pretrained == 'ch':
                    cfg_pr = 6.0
                script += f"""
            --with_prior_preservation --prior_loss_weight=1.0 --num_class_images 200 \\
            --class_prompt="a photo of a person" --class_data_dir="data_root/generated/model/{prior_folder}/a photo of a person_neg/{cfg_pr:.2f}" \\"""
                
            # Conditional learning rate + TI options
            if use_ti:
                
                if lora_rank <= 0:
                    script += f"""
            --learning_rate_ti {lr_ti} \\
            --placeholder_token="{placeholder_token}" --initializer_token='{initializer_token}'"""
                else:
                    if use_te:
                        script += f"""
            --learning_rate_lora {lr_lora} --learning_rate_ti {lr_ti} \\
            --train_text_encoder --learning_rate_lora_text_encoder {re_lr_te} \\
            --placeholder_token="{placeholder_token}" --initializer_token='{initializer_token}'"""
                    else:
                        script += f"""
            --learning_rate_lora {lr_lora} --learning_rate_ti {lr_ti} \\
            --placeholder_token="{placeholder_token}" --initializer_token='{initializer_token}'"""
            else:
                script += f"""
            --learning_rate {lr_lora}"""

            print(script)
            final_exp_names += [final_exp_name]
        
print(len(final_exp_names))
print(final_exp_names)

        


            accelerate launch train_dreambooth_lora.py \
            --pretrained_model_name_or_path="CompVis/stable-diffusion-v1-4"  \
            --load_unet_weight_path="data_root/logs/stereo/Barrack Obama/final_reo_unet.pt" \
            --instance_data_dir="data_root/data/real_data/obama/aligned/obama-5-v0" \
            --output_dir="data_root/logs/rlct4.reV.obamaA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_stereo.obama_sd1.4" \
            --validation_prompt="a photo of v1" --instance_prompt="a photo of v1" \
            --train_batch_size=1 --gradient_accumulation_steps=4 \
            --lora_rank 4 --target_lora_modules to_k to_v --target_lora_layers cross \
            --max_train_steps=1000  --validation_steps=50  --checkpointing_steps=50  --lr_scheduler "linear"  --seed 0 \
            --run_note 'uul obamaA5V0 lNone ti' \
            --cfg_scale 6.0 \
            --negative_prompt "longbody, lowres, bad anatomy, bad hands, missing fingers, extra digit, fewer digits, cropped, wo

echo 'count:0 - sd1.4 0 /'

       accelerate launch train_dreambooth_lora.py \
           --pretrained_model_name_or_path='CompVis/stable-diffusion-v1-4'  \
           --load_unet_weight_path="data_root/logs/esd/sd1.4/esd-Barrack_Obama-from-Barrack_Obama-esdu.safetensors" \
           --load_lora_weight_path="" \
           --instance_data_dir="data_root/data/real_data/dummy" \
           --gen_image_path="data_root/generated/model/esd-u.obama_sd1.4" \
           --output_dir="data_root/logs/gen" \
           --validation_prompt="a photo of Barrack Obama;a photo of Rihanna;a photo of Ed Sheeran;a photo of Margot Robbie;a photo of Chris Hemsworth;a photo of Chris Evans;a photo of Anne Adam;a photo of Anne Hathaway;a photo of Mariah Carey;a photo of Octavia Spencer;a photo of Morgan Freeman;a photo of Drake" --instance_prompt="a photo of Barrack Obama;a photo of Rihanna;a photo of Ed Sheeran;a photo of Margot Robbie;a photo of Chris Hemsworth;a photo of Chris Evans;a photo of Anne Adam;

In [ ]:

# Swap learning - Stereo
# RANDOM unlearn seen - relearn (unseen)



pretrained = 'sd1.4'


#seed = 0

# seeds = [0,1,2] # along

# hacked 
# seeds = [999]*len(ul_exp_names)
seeds = [0,1,2]
# seeds = [3,4]
# seeds = [0,1,2,3,4]

use_te = True
batch_size = 1
gradient_accumulation_step = 4
# re
# lr_lora_grid = ["1e-4", "5e-5", "1e-5"]
lr_lora_grid = ["1e-4"]
# lr_lora_grid = [ "1e-5"]
lr_ti_grid = ["5e-4"]   # only used if use_ti
# lr_ti_grid   = ["5e-3","5e-2"]   # only used if use_ti
lr_te_grid = ["1e-5"] 
lora_ranks = [4] # [1,2,4,8,16,32,64]
# # Create all combinations of lr_lora, lr_ti, and seed
combos = list(itertools.product(lr_lora_grid,
                                lr_ti_grid,
                                lr_te_grid if use_te else [None],
                                lora_ranks))

use_pr = True

use_te = True
apply_use_ti = True

lr_scheduler = 'linear' # 'linear' # 'cosine' # 'cosine_with_restarts'
apply_negative_prompt = True

use_manual_params = True
reV = True
is_relearn = True 

# fix here #
manual_params = {
    # 'data_setting': 'facefew',
    'data_setting': 'align',
    
    # 'data_setting': 'small',s
    # 'lora_rank' :  16
}

data_setting = 'align' 

final_exp_names = []
# for ul_exp_name in ul_exp_names:

rng = np.random.RandomState(123)
scripts = []
for learn_concept in unseen_concepts:
    
    for seed in seeds:

        concept = learn_concept
        unlearned_concept = rng.choice(seen_concepts)


        train_method = 'stereo'
        ul_exp_name = f'{train_method}.{unlearned_concept}_sd1.4'
        
        
        base_exp_name = '_'.join(ul_exp_name.split('_')[1:])
        exp_name = base_exp_name
        

        # print(f"Base Experiment Name: {base_exp_name}")
        
        #fix edit here : they are using this to reconstruct the base_exp as well
        lr_lora, lr_ti = extract_lrs(base_exp_name)
        lora_rank = extract_learning_lora_rank(base_exp_name)
        
        # todo: better use 're'
        for re_lr_lora, re_lr_ti, re_lr_te, re_lora_rank in combos:
            
            # print(f'lora_rank: {lora_rank}, lr: {lr_lora}, ti_lr: {lr_ti}')
            # pretrained_path = f"data_root/logs/{ul_exp_name}/LoRA_fusion_model"
            pretrained_path = 'CompVis/stable-diffusion-v1-4'
            unet_weight_path = f"data_root/logs/stereo/{data_info[unlearned_concept]['Full_name']}/final_reo_unet.pt"
            
            
            # print(f"Concept: {concept}"

            use_ti = 'ti' in exp_name or '-V' in exp_name or apply_use_ti
            # use_pr = 'pr' in exp_name

            if data_setting == 'facefew':
                dataset_name = f'{concept}5F0r{seed}'
                
            elif data_setting == 'align':
                dataset_name = f'{concept}A5V0'
            elif data_setting == 'fewshot':
                
                if 'sceleb' in concept:
                    dataset_name = f'{concept}U3'
                elif concept == 'avp':
                    dataset_name = 'avpS3'
                else:
                    dataset_name = f'{concept}U3'
            elif data_setting == 'small':
                
                if 'sceleb' in concept:
                    dataset_name = f'{concept}N10'
                else:
                    dataset_name = f'{concept}10'
            else:
                
                if 'sceleb' in concept:
                    dataset_name = f'{concept}N50'
                elif concept == 'avp':
                    dataset_name = 'avp20'
                else:
                    dataset_name = f'{concept}50'
                    
            if reV:
                initializer_token = concept2initializer[concept]
            else: 
                initializer_token = ''
                
            if use_ti:
                if 'sceleb' in concept:
                    prompt = 'A photo of a v1,A photo of a v2,A photo of a v3,A photo of a v4,A photo of a v5'
                    placeholder_token = 'v1,v2,v3,v4,v5'
                else:
                    prompt = 'a photo of v1' 
                    placeholder_token = 'v1'
            else:
                prompt = concept2prompt[concept]

            name_tag = ''
            if is_relearn: name_tag += 'uul'
            name_tag = f'{name_tag} {dataset_name}'
            name_tag += f' l{lora_rank}'
            if use_ti: 
                # name_tag += f' ti.{lr_ti}'
                name_tag += f' ti'

            data_root = dataset_name2data_root[dataset_name]
            
            if use_ti:
                dataset_name_for_exp = dataset_name + "-V"
                # if use_ni:
                #     dataset_name_for_exp += ".ni"
            else: dataset_name_for_exp = dataset_name
            
            
            # prior preservation folder
            prior_folder = 'original_realistic_vision'
            if pretrained == 'sd1.5':
                prior_folder = 'original_pretrained_sd1.5'
            if pretrained == 'sd1.4':
                prior_folder = 'original_pretrained_sd1.4'     
            if pretrained == 'rv':
                prior_folder = 'original_realistic_vision'
            elif pretrained == 'ch':
                prior_folder = 'original_chilloutmix'

            
            # renaming to check
            # re_exp_name = f'c.l{lora_rank}.kv_{dataset_name_for_exp}'
            # if use_pr:
            #     re_exp_name += f'_pr0.50'
            # re_exp_name += '_lr'
            # if lora_rank >0: re_exp_name += f"{str(lr_lora)}"
            # if use_ti:
            #     re_exp_name += f'.ti{str(lr_ti)}'
            # re_exp_name += '_f0.5_b1g4'
            
            # print(re_exp_name)
            # assert re_exp_name in base_exp_name, f"Expected {re_exp_name} in {base_exp_name}"

            # if manual_lora is not None and manual_data != lora_rank:
            
            if use_manual_params:
                lora_rank = re_lora_rank
                eff_data_setting = manual_params['data_setting']

                lr_lora, lr_ti = re_lr_lora, re_lr_ti
                
                if eff_data_setting == 'facefew':
                    dataset_name = f'{concept}5F0r{seed}'
                    
                elif eff_data_setting == 'align':
                    dataset_name = f'{concept}A5V0'
                
                elif eff_data_setting == 'fewshot':
                    
                    if 'sceleb' in concept:
                        dataset_name = f'{concept}U3'
                    elif concept == 'avp':
                        dataset_name = 'avpS3'
                    else:
                        dataset_name = f'{concept}U3'
                elif eff_data_setting == 'small':
                    
                    if 'sceleb' in concept:
                        dataset_name = f'{concept}N10'
                    else:
                        dataset_name = f'{concept}10'
                else:
                    
                    if 'sceleb' in concept:
                        dataset_name = f'{concept}N50'
                    elif concept == 'avp':
                        dataset_name = 'avp20'
                    else:
                        dataset_name = f'{concept}50'            
                        
                data_root = dataset_name2data_root[dataset_name]
            
            
            
            
            if reV:
                if use_te:
                    relearn_exp_name = f"rlct{lora_rank}.reV.{dataset_name}"
                else:
                    relearn_exp_name = f"rlc{lora_rank}.reV.{dataset_name}"
            else:
                if use_te:
                    relearn_exp_name = f"rlct{lora_rank}.{dataset_name}"
                else:
                    relearn_exp_name = f"rlc{lora_rank}.{dataset_name}"
            
            
            if lr_scheduler == 'linear':
                relearn_exp_name += f".ln"
            relearn_exp_name += f".lr{re_lr_lora}.ti{re_lr_ti}"
                
                
            if use_pr:
                relearn_exp_name += f".pr1.00"
                if apply_negative_prompt:
                    relearn_exp_name += ".neg"
            
            relearn_exp_name += f".b{batch_size}g{gradient_accumulation_step}"
            
            if seed != 0:
                relearn_exp_name += f".r{seed}"

            final_exp_name = f"{relearn_exp_name}_{ul_exp_name}"
            
            
            script = f"""
            accelerate launch train_dreambooth_lora.py \\
            --pretrained_model_name_or_path="{pretrained_path}"  \\
            --load_unet_weight_path="{unet_weight_path}" \\
            --instance_data_dir="{data_root}" \\
            --output_dir="data_root/logs/{final_exp_name}" \\
            --validation_prompt="{prompt}" --instance_prompt="{prompt}" \\
            --train_batch_size={batch_size} --gradient_accumulation_steps={gradient_accumulation_step} \\
            --lora_rank {lora_rank} --target_lora_modules to_k to_v --target_lora_layers cross \\
            --max_train_steps=1000  --validation_steps=50  --checkpointing_steps=50  --lr_scheduler "{lr_scheduler}"  --seed {seed} \\
            --run_note '{name_tag}' \\"""
                
            script += f"""
            --cfg_scale 6.0 \\"""
        
        
            if apply_negative_prompt:
                script += f"""
            --negative_prompt "longbody, lowres, bad anatomy, bad hands, missing fingers, extra digit, fewer digits, cropped, worst quality, low quality." \\"""
                
                
            if use_pr:
                if pretrained == 'sd1.4':
                    cfg_pr = 7.5
                elif pretrained == 'ch':
                    cfg_pr = 6.0
                script += f"""
            --with_prior_preservation --prior_loss_weight=1.0 --num_class_images 200 \\
            --class_prompt="a photo of a person" --class_data_dir="data_root/generated/model/{prior_folder}/a photo of a person_neg/{cfg_pr:.2f}" \\"""
                
            # Conditional learning rate + TI options
            if use_ti:
                
                if lora_rank <= 0:
                    script += f"""
            --learning_rate_ti {lr_ti} \\
            --placeholder_token="{placeholder_token}" --initializer_token='{initializer_token}'"""
                else:
                    if use_te:
                        script += f"""
            --learning_rate_lora {lr_lora} --learning_rate_ti {lr_ti} \\
            --train_text_encoder --learning_rate_lora_text_encoder {re_lr_te} \\
            --placeholder_token="{placeholder_token}" --initializer_token='{initializer_token}'"""
                    else:
                        script += f"""
            --learning_rate_lora {lr_lora} --learning_rate_ti {lr_ti} \\
            --placeholder_token="{placeholder_token}" --initializer_token='{initializer_token}'"""
            else:
                script += f"""
            --learning_rate {lr_lora}"""

            print(f"echo 'count:{len(final_exp_names)} '")
            print(script)
            final_exp_names += [final_exp_name]
            
            scripts += [script]
        
print(len(final_exp_names))
print(final_exp_names)


# stereo
# ['rlct4.reV.asanteA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_stereo.edsheeran_sd1.4', 'rlct4.reV.asanteA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_stereo.edsheeran_sd1.4', 'rlct4.reV.asanteA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_stereo.aadam_sd1.4', 'rlct4.reV.reeseA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_stereo.rihanna_sd1.4', 'rlct4.reV.reeseA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_stereo.mrobbie_sd1.4', 'rlct4.reV.reeseA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_stereo.morganf_sd1.4', 'rlct4.reV.nivolaA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_stereo.drake_sd1.4', 'rlct4.reV.nivolaA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_stereo.octavia_sd1.4', 'rlct4.reV.nivolaA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_stereo.aadam_sd1.4', 'rlct4.reV.earleA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_stereo.rihanna_sd1.4', 'rlct4.reV.earleA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_stereo.obama_sd1.4', 'rlct4.reV.earleA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_stereo.rihanna_sd1.4', 'rlct4.reV.leowoodalA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_stereo.octavia_sd1.4', 'rlct4.reV.leowoodalA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_stereo.obama_sd1.4', 'rlct4.reV.leowoodalA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_stereo.obama_sd1.4', 'rlct4.reV.starkeyA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_stereo.octavia_sd1.4', 'rlct4.reV.starkeyA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_stereo.mrobbie_sd1.4', 'rlct4.reV.starkeyA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_stereo.chemsworth_sd1.4', 'rlct4.reV.apierreA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_stereo.obama_sd1.4', 'rlct4.reV.apierreA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_stereo.obama_sd1.4', 'rlct4.reV.apierreA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_stereo.chemsworth_sd1.4', 'rlct4.reV.skyhblackA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_stereo.rihanna_sd1.4', 'rlct4.reV.skyhblackA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_stereo.ahathaway_sd1.4', 'rlct4.reV.skyhblackA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_stereo.mrobbie_sd1.4', 'rlct4.reV.sophiewildeA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_stereo.edsheeran_sd1.4', 'rlct4.reV.sophiewildeA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_stereo.chemsworth_sd1.4', 'rlct4.reV.sophiewildeA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_stereo.ahathaway_sd1.4', 'rlct4.reV.edebiriA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_stereo.edsheeran_sd1.4', 'rlct4.reV.edebiriA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_stereo.chemsworth_sd1.4', 'rlct4.reV.edebiriA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_stereo.mcarey_sd1.4', 'rlct4.reV.mmadisonA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_stereo.obama_sd1.4', 'rlct4.reV.mmadisonA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_stereo.ahathaway_sd1.4', 'rlct4.reV.mmadisonA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_stereo.octavia_sd1.4', 'rlct4.reV.nicoparkerA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_stereo.mrobbie_sd1.4', 'rlct4.reV.nicoparkerA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_stereo.chemsworth_sd1.4', 'rlct4.reV.nicoparkerA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_stereo.aadam_sd1.4']



echo 'count:0 '

            accelerate launch train_dreambooth_lora.py \
            --pretrained_model_name_or_path="CompVis/stable-diffusion-v1-4"  \
            --load_unet_weight_path="data_root/logs/stereo/Ed Sheeran/final_reo_unet.pt" \
            --instance_data_dir="data_root/data/real_data/asante/aligned/asante-5-v0" \
            --output_dir="data_root/logs/rlct4.reV.asanteA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_stereo.edsheeran_sd1.4" \
            --validation_prompt="a photo of v1" --instance_prompt="a photo of v1" \
            --train_batch_size=1 --gradient_accumulation_steps=4 \
            --lora_rank 4 --target_lora_modules to_k to_v --target_lora_layers cross \
            --max_train_steps=1000  --validation_steps=50  --checkpointing_steps=50  --lr_scheduler "linear"  --seed 0 \
            --run_note 'uul asanteA5V0 lNone ti' \
            --cfg_scale 6.0 \
            --negative_prompt "longbody, lowres, bad anatomy, bad hands, missing fingers, extra digit, fewe

['rlct4.reV.asanteA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-x.edsheeran_sd1.4',
 'rlct4.reV.asanteA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_esd-x.edsheeran_sd1.4',
 'rlct4.reV.asanteA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_esd-x.aadam_sd1.4',
 'rlct4.reV.reeseA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-x.rihanna_sd1.4',
 'rlct4.reV.reeseA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_esd-x.mrobbie_sd1.4',
 'rlct4.reV.reeseA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_esd-x.morganf_sd1.4',
 'rlct4.reV.nivolaA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-x.drake_sd1.4',
 'rlct4.reV.nivolaA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_esd-x.octavia_sd1.4',
 'rlct4.reV.nivolaA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_esd-x.aadam_sd1.4',
 'rlct4.reV.earleA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-x.rihanna_sd1.4',
 'rlct4.reV.earleA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_esd-x.obama_sd1.4',
 'rlct4.reV.earleA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_esd-x.rihanna_sd1.4',
 'rlct4.reV.leowoodalA5V0.ln.lr1e-4.ti5e-4.pr1.00.

In [40]:
v = 2
# for i, script in enumerate(scripts[v*5:v*5+5]):
n_device = 3
len_ = int(len(scripts)/ n_device) 

print(f"Total scripts: {len(scripts)}: {len_} per device")
for i in range(v*len_, v*len_+len_):
    print(f"""echo 'count: {i}'""")
    
    script = scripts[i]
    print(script)
    
print(final_exp_names[v*len_:v*len_+len_])
print(f"Total final experiment names: {len(final_exp_names[v*len_:v*len_+len_])}")

Total scripts: 36: 12 per device
echo 'count: 24'

            accelerate launch train_dreambooth_lora.py \
            --pretrained_model_name_or_path="CompVis/stable-diffusion-v1-4"  \
            --load_unet_weight_path="data_root/logs/stereo/Ed Sheeran/final_reo_unet.pt" \
            --instance_data_dir="data_root/data/real_data/sophiewilde/aligned/sophiewilde-5-v0" \
            --output_dir="data_root/logs/rlct4.reV.sophiewildeA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_stereo.edsheeran_sd1.4" \
            --validation_prompt="a photo of v1" --instance_prompt="a photo of v1" \
            --train_batch_size=1 --gradient_accumulation_steps=4 \
            --lora_rank 4 --target_lora_modules to_k to_v --target_lora_layers cross \
            --max_train_steps=1000  --validation_steps=50  --checkpointing_steps=50  --lr_scheduler "linear"  --seed 0 \
            --run_note 'uul sophiewildeA5V0 lNone ti' \
            --cfg_scale 6.0 \
            --negative_prompt "longbody, lowres, bad 

In [ ]:
# UCE
# Remove concepts
for seen_concept in seen_concepts:
    # if seen_concept  in seen_concepts_1: continue
    concept = seen_concept
    target_concept = data_info[seen_concept]['Full_name']
    script = f"""python trainscripts/uce_sd_erase.py --model_id 'CompVis/stable-diffusion-v1-4' --edit_concepts '{target_concept}' --guide_concept 'Person' --preserve_concepts 'Person' --device 'cuda:0' --concept_type 'object' --exp_name '{concept}_uce_sd' --save_dir '../data_root/logs/uce' """
    print(script)



python trainscripts/uce_sd_erase.py --model_id 'CompVis/stable-diffusion-v1-4' --edit_concepts 'Adam Driver' --guide_concept 'Person' --preserve_concepts 'Person' --device 'cuda:0' --concept_type 'object' --exp_name 'adriver_uce_sd' --save_dir '../data_root/logs/uce' 
python trainscripts/uce_sd_erase.py --model_id 'CompVis/stable-diffusion-v1-4' --edit_concepts 'Andrew Garfield' --guide_concept 'Person' --preserve_concepts 'Person' --device 'cuda:0' --concept_type 'object' --exp_name 'agarfield_uce_sd' --save_dir '../data_root/logs/uce' 
python trainscripts/uce_sd_erase.py --model_id 'CompVis/stable-diffusion-v1-4' --edit_concepts 'Angelina Jolie' --guide_concept 'Person' --preserve_concepts 'Person' --device 'cuda:0' --concept_type 'object' --exp_name 'ajolie_uce_sd' --save_dir '../data_root/logs/uce' 
python trainscripts/uce_sd_erase.py --model_id 'CompVis/stable-diffusion-v1-4' --edit_concepts 'Amber Heard' --guide_concept 'Person' --preserve_concepts 'Person' --device 'cuda:0' --co

In [61]:
# UCE: relearning 
# decoding unlearning - with same hyperparameter

# Implement text encoder relearning

# ul_exp_names = ['ul1.prg1e-4d5e-4.lr1e-4.n8.G.sceleb5g0.person.s50_c.l16.kv_sceleb5g0N50-V_pr0.50_lr5e-5.ti5e-4_f0.5_b4g4.s10000']

# ul_exp_names = ['ul1.prg1e-4d8e+3.lr1e-4.n8.G.mcarey.person.s50_sd14', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.mcarey.person.s50.r1_sd14', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.mcarey.person.s50.r2_sd14', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.octavia.person.s50_sd14', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.octavia.person.s50.r1_sd14', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.octavia.person.s50.r2_sd14', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.oprah.person.s50_sd14', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.oprah.person.s50.r1_sd14', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.oprah.person.s50.r2_sd14', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.morganf.person.s50_sd14', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.morganf.person.s50.r1_sd14', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.morganf.person.s50.r2_sd14', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.drake.person.s50_sd14', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.drake.person.s50.r1_sd14', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.drake.person.s50.r2_sd14', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.idris.person.s50_sd14', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.idris.person.s50.r1_sd14', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.idris.person.s50.r2_sd14']


# ul_exp_names = ['esd-Barrack_Obama-from-Barrack_Obama-esdx', 'esd-Barrack_Obama-from-Barrack_Obama-esdu', 'esd-Barrack_Obama-from-Barrack_Obama-esdall', 'esd-Rihanna-from-Rihanna-esdx', 'esd-Rihanna-from-Rihanna-esdu', 'esd-Rihanna-from-Rihanna-esdall', 'esd-Ed_Sheeran-from-Ed_Sheeran-esdx', 'esd-Ed_Sheeran-from-Ed_Sheeran-esdu', 'esd-Ed_Sheeran-from-Ed_Sheeran-esdall', 'esd-Margot_Robbie-from-Margot_Robbie-esdx', 'esd-Margot_Robbie-from-Margot_Robbie-esdu', 'esd-Margot_Robbie-from-Margot_Robbie-esdall', 'esd-Chris_Hemsworth-from-Chris_Hemsworth-esdx', 'esd-Chris_Hemsworth-from-Chris_Hemsworth-esdu', 'esd-Chris_Hemsworth-from-Chris_Hemsworth-esdall', 'esd-Chris_Evans-from-Chris_Evans-esdx', 'esd-Chris_Evans-from-Chris_Evans-esdu', 'esd-Chris_Evans-from-Chris_Evans-esdall', 'esd-Adam_Driver-from-Adam_Driver-esdx', 'esd-Adam_Driver-from-Adam_Driver-esdu', 'esd-Adam_Driver-from-Adam_Driver-esdall', 'esd-Andrew_Garfield-from-Andrew_Garfield-esdx', 'esd-Andrew_Garfield-from-Andrew_Garfield-esdu', 'esd-Andrew_Garfield-from-Andrew_Garfield-esdall', 'esd-Anne_Adam-from-Anne_Adam-esdx', 'esd-Anne_Adam-from-Anne_Adam-esdu', 'esd-Anne_Adam-from-Anne_Adam-esdall', 'esd-Anne_Hathaway-from-Anne_Hathaway-esdx', 'esd-Anne_Hathaway-from-Anne_Hathaway-esdu', 'esd-Anne_Hathaway-from-Anne_Hathaway-esdall', 'esd-Angelina_Jolie-from-Angelina_Jolie-esdx', 'esd-Angelina_Jolie-from-Angelina_Jolie-esdu', 'esd-Angelina_Jolie-from-Angelina_Jolie-esdall', 'esd-Amber_Heard-from-Amber_Heard-esdx', 'esd-Amber_Heard-from-Amber_Heard-esdu', 'esd-Amber_Heard-from-Amber_Heard-esdall', 'esd-Mariah_Carey-from-Mariah_Carey-esdx', 'esd-Mariah_Carey-from-Mariah_Carey-esdu', 'esd-Mariah_Carey-from-Mariah_Carey-esdall', 'esd-Octavia_Spencer-from-Octavia_Spencer-esdx', 'esd-Octavia_Spencer-from-Octavia_Spencer-esdu', 'esd-Octavia_Spencer-from-Octavia_Spencer-esdall', 'esd-Oprah_Winfrey-from-Oprah_Winfrey-esdx', 'esd-Oprah_Winfrey-from-Oprah_Winfrey-esdu', 'esd-Oprah_Winfrey-from-Oprah_Winfrey-esdall', 'esd-Morgan_Freeman-from-Morgan_Freeman-esdx', 'esd-Morgan_Freeman-from-Morgan_Freeman-esdu', 'esd-Morgan_Freeman-from-Morgan_Freeman-esdall', 'esd-Drake-from-Drake-esdx', 'esd-Drake-from-Drake-esdu', 'esd-Drake-from-Drake-esdall', 'esd-Idris_Elba-from-Idris_Elba-esdx', 'esd-Idris_Elba-from-Idris_Elba-esdu', 'esd-Idris_Elba-from-Idris_Elba-esdall']

# ul_exp_names = [ul_exp_name for ul_exp_name in ul_exp_names if 'esdu' in ul_exp_name][12:] # filter out the esd ones
# # ul_exp_names = ['esd-Barrack_Obama-from-Barrack_Obama-esdx', 'esd-Barrack_Obama-from-Barrack_Obama-esdu', 'esd-Barrack_Obama-from-Barrack_Obama-esdall']

ul_exp_names = list(seen_concepts)


learn_concept = "" # 'moodeng' # 'crybaby' # 'avp' # 'chiquita' # 'reese' # 'gout' # 'jooli' # 'honer' # 'sceleb5g0'
pretrained = 'sd1.4'



#seed = 0

# seeds = [0,1,2] # along

# hacked 
# seeds = [999]*len(ul_exp_names)
seeds = [2]
# seeds = [3,4]
# seeds = [0,1,2,3,4]

use_te = True
batch_size = 1
gradient_accumulation_step = 4
# re
# lr_lora_grid = ["1e-4", "5e-5", "1e-5"]
max_train_steps = 3000
lr_lora_grid = ["1e-4"]
# lr_lora_grid = [ "1e-5"]
lr_ti_grid = ["5e-4"]   # only used if use_ti
# lr_ti_grid   = ["5e-3","5e-2"]   # only used if use_ti
lr_te_grid = ["1e-5"] 
lora_ranks = [4] # [1,2,4,8,16,32,64]
# # Create all combinations of lr_lora, lr_ti, and seed
combos = list(itertools.product(lr_lora_grid,
                                lr_ti_grid,
                                lr_te_grid if use_te else [None],
                                lora_ranks))

use_pr = True

use_te = True
apply_use_ti = True

lr_scheduler = 'linear' # 'linear' # 'cosine' # 'cosine_with_restarts'
apply_negative_prompt = True

use_manual_params = True
reV = True
is_relearn = True 

# fix here #
manual_params = {
    # 'data_setting': 'facefew',
    'data_setting': 'align',
    
    # 'data_setting': 'small',s
    # 'lora_rank' :  16
}

data_setting = 'align' 

final_exp_names = []
# for ul_exp_name in ul_exp_names:
for ul_exp_name in ul_exp_names:
    
    if not learn_concept:
        for c in seen_concepts:
            if c in data_info and c in ul_exp_name:
                concept = c
                break
    else:
        concept = learn_concept

    train_method = 'uce'


    pretrained_unet_name = ul_exp_name 
    ul_exp_name = f'{train_method}.{concept}_sd1.4'
    
    
    base_exp_name = '_'.join(ul_exp_name.split('_')[1:])
    exp_name = base_exp_name

        
    for seed in seeds:

        if seed == 999:
            if '.r1' in ul_exp_name:
                seed = 1
            elif '.r2' in ul_exp_name:
                seed = 2    
            else:
                seed = 0
        
        # print(f"Base Experiment Name: {base_exp_name}")
        
        #fix edit here : they are using this to reconstruct the base_exp as well
        lr_lora, lr_ti = extract_lrs(base_exp_name)
        lora_rank = extract_learning_lora_rank(base_exp_name)
        
        # todo: better use 're'
        for re_lr_lora, re_lr_ti, re_lr_te, re_lora_rank in combos:
            
            # print(f'lora_rank: {lora_rank}, lr: {lr_lora}, ti_lr: {lr_ti}')
            # pretrained_path = f"data_root/logs/{ul_exp_name}/LoRA_fusion_model"
            pretrained_path = 'CompVis/stable-diffusion-v1-4'
            unet_weight_path = f"data_root/logs/uce/{concept}_uce_sd.safetensors"
            # print(f"Concept: {concept}"

            use_ti = 'ti' in exp_name or '-V' in exp_name or apply_use_ti
            # use_pr = 'pr' in exp_name

            if data_setting == 'facefew':
                dataset_name = f'{concept}5F0r{seed}'
                
            elif data_setting == 'align':
                dataset_name = f'{concept}A5V0'
            elif data_setting == 'fewshot':
                
                if 'sceleb' in concept:
                    dataset_name = f'{concept}U3'
                elif concept == 'avp':
                    dataset_name = 'avpS3'
                else:
                    dataset_name = f'{concept}U3'
            elif data_setting == 'small':
                
                if 'sceleb' in concept:
                    dataset_name = f'{concept}N10'
                else:
                    dataset_name = f'{concept}10'
            else:
                
                if 'sceleb' in concept:
                    dataset_name = f'{concept}N50'
                elif concept == 'avp':
                    dataset_name = 'avp20'
                else:
                    dataset_name = f'{concept}50'
                    
            if reV:
                initializer_token = concept2initializer[concept]
            else: 
                initializer_token = ''
                
            if use_ti:
                if 'sceleb' in concept:
                    prompt = 'A photo of a v1,A photo of a v2,A photo of a v3,A photo of a v4,A photo of a v5'
                    placeholder_token = 'v1,v2,v3,v4,v5'
                else:
                    prompt = 'a photo of v1' 
                    placeholder_token = 'v1'
            else:
                prompt = concept2prompt[concept]

            name_tag = ''
            if is_relearn: name_tag += 'uul'
            name_tag = f'{name_tag} {dataset_name}'
            name_tag += f' l{lora_rank}'
            if use_ti: 
                # name_tag += f' ti.{lr_ti}'
                name_tag += f' ti'

            data_root = dataset_name2data_root[dataset_name]
            
            if use_ti:
                dataset_name_for_exp = dataset_name + "-V"
                # if use_ni:
                #     dataset_name_for_exp += ".ni"
            else: dataset_name_for_exp = dataset_name
            
            
            # prior preservation folder
            prior_folder = 'original_realistic_vision'
            if pretrained == 'sd1.5':
                prior_folder = 'original_pretrained_sd1.5'
            if pretrained == 'sd1.4':
                prior_folder = 'original_pretrained_sd1.4'     
            if pretrained == 'rv':
                prior_folder = 'original_realistic_vision'
            elif pretrained == 'ch':
                prior_folder = 'original_chilloutmix'

            
            # renaming to check
            # re_exp_name = f'c.l{lora_rank}.kv_{dataset_name_for_exp}'
            # if use_pr:
            #     re_exp_name += f'_pr0.50'
            # re_exp_name += '_lr'
            # if lora_rank >0: re_exp_name += f"{str(lr_lora)}"
            # if use_ti:
            #     re_exp_name += f'.ti{str(lr_ti)}'
            # re_exp_name += '_f0.5_b1g4'
            
            # print(re_exp_name)
            # assert re_exp_name in base_exp_name, f"Expected {re_exp_name} in {base_exp_name}"

            # if manual_lora is not None and manual_data != lora_rank:
            
            if use_manual_params:
                lora_rank = re_lora_rank
                eff_data_setting = manual_params['data_setting']

                lr_lora, lr_ti = re_lr_lora, re_lr_ti
                
                if eff_data_setting == 'facefew':
                    dataset_name = f'{concept}5F0r{seed}'
                    
                elif eff_data_setting == 'align':
                    dataset_name = f'{concept}A5V0'
                
                elif eff_data_setting == 'fewshot':
                    
                    if 'sceleb' in concept:
                        dataset_name = f'{concept}U3'
                    elif concept == 'avp':
                        dataset_name = 'avpS3'
                    else:
                        dataset_name = f'{concept}U3'
                elif eff_data_setting == 'small':
                    
                    if 'sceleb' in concept:
                        dataset_name = f'{concept}N10'
                    else:
                        dataset_name = f'{concept}10'
                else:
                    
                    if 'sceleb' in concept:
                        dataset_name = f'{concept}N50'
                    elif concept == 'avp':
                        dataset_name = 'avp20'
                    else:
                        dataset_name = f'{concept}50'            
                        
                data_root = dataset_name2data_root[dataset_name]
            
            
            
            
            if reV:
                if use_te:
                    relearn_exp_name = f"rlct{lora_rank}.reV.{dataset_name}"
                else:
                    relearn_exp_name = f"rlc{lora_rank}.reV.{dataset_name}"
            else:
                if use_te:
                    relearn_exp_name = f"rlct{lora_rank}.{dataset_name}"
                else:
                    relearn_exp_name = f"rlc{lora_rank}.{dataset_name}"
            
            
            if lr_scheduler == 'linear':
                relearn_exp_name += f".ln"
            relearn_exp_name += f".lr{re_lr_lora}.ti{re_lr_ti}"
                
                
            if use_pr:
                relearn_exp_name += f".pr1.00"
                if apply_negative_prompt:
                    relearn_exp_name += ".neg"
            
            relearn_exp_name += f".b{batch_size}g{gradient_accumulation_step}"
            
            if max_train_steps != 1000:
                relearn_exp_name += f".s{max_train_steps}"

            if seed != 0:
                relearn_exp_name += f".r{seed}"

            final_exp_name = f"{relearn_exp_name}_{ul_exp_name}"
            
            
            script = f"""
            accelerate launch train_dreambooth_lora.py \\
            --pretrained_model_name_or_path="{pretrained_path}"  \\
            --load_unet_weight_path="{unet_weight_path}" \\
            --instance_data_dir="{data_root}" \\
            --output_dir="data_root/logs/{final_exp_name}" \\
            --validation_prompt="{prompt}" --instance_prompt="{prompt}" \\
            --train_batch_size={batch_size} --gradient_accumulation_steps={gradient_accumulation_step} \\
            --lora_rank {lora_rank} --target_lora_modules to_k to_v --target_lora_layers cross \\
            --max_train_steps={max_train_steps}  --validation_steps=50  --checkpointing_steps=50  --lr_scheduler "{lr_scheduler}"  --seed {seed} \\
            --run_note '{name_tag}' \\"""
                
            script += f"""
            --cfg_scale 6.0 \\"""
        
        
            if apply_negative_prompt:
                script += f"""
            --negative_prompt "longbody, lowres, bad anatomy, bad hands, missing fingers, extra digit, fewer digits, cropped, worst quality, low quality." \\"""
                
                
            if use_pr:
                if pretrained == 'sd1.4':
                    cfg_pr = 7.5
                elif pretrained == 'ch':
                    cfg_pr = 6.0
                script += f"""
            --with_prior_preservation --prior_loss_weight=1.0 --num_class_images 200 \\
            --class_prompt="a photo of a person" --class_data_dir="data_root/generated/model/{prior_folder}/a photo of a person_neg/{cfg_pr:.2f}" \\"""
                
            # Conditional learning rate + TI options
            if use_ti:
                
                if lora_rank <= 0:
                    script += f"""
            --learning_rate_ti {lr_ti} \\
            --placeholder_token="{placeholder_token}" --initializer_token='{initializer_token}'"""
                else:
                    if use_te:
                        script += f"""
            --learning_rate_lora {lr_lora} --learning_rate_ti {lr_ti} \\
            --train_text_encoder --learning_rate_lora_text_encoder {re_lr_te} \\
            --placeholder_token="{placeholder_token}" --initializer_token='{initializer_token}'"""
                    else:
                        script += f"""
            --learning_rate_lora {lr_lora} --learning_rate_ti {lr_ti} \\
            --placeholder_token="{placeholder_token}" --initializer_token='{initializer_token}'"""
            else:
                script += f"""
            --learning_rate {lr_lora}"""

            print(script)
            final_exp_names += [final_exp_name]
        
print(len(final_exp_names))
print(final_exp_names)

        


            accelerate launch train_dreambooth_lora.py \
            --pretrained_model_name_or_path="CompVis/stable-diffusion-v1-4"  \
            --load_unet_weight_path="data_root/logs/uce/obama_uce_sd.safetensors" \
            --instance_data_dir="data_root/data/real_data/obama/aligned/obama-5-v0" \
            --output_dir="data_root/logs/rlct4.reV.obamaA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000.r2_uce.obama_sd1.4" \
            --validation_prompt="a photo of v1" --instance_prompt="a photo of v1" \
            --train_batch_size=1 --gradient_accumulation_steps=4 \
            --lora_rank 4 --target_lora_modules to_k to_v --target_lora_layers cross \
            --max_train_steps=3000  --validation_steps=50  --checkpointing_steps=50  --lr_scheduler "linear"  --seed 2 \
            --run_note 'uul obamaA5V0 lNone ti' \
            --cfg_scale 6.0 \
            --negative_prompt "longbody, lowres, bad anatomy, bad hands, missing fingers, extra digit, fewer digits, cropped, worst 

In [64]:
# UCE: Generation
# exp_names = ['rlct4.reV.asanteA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000_uce.edsheeran_sd1.4', 'rlct4.reV.reeseA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000_uce.edsheeran_sd1.4', 'rlct4.reV.nivolaA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000_uce.aadam_sd1.4', 'rlct4.reV.earleA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000_uce.rihanna_sd1.4', 'rlct4.reV.leowoodalA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000_uce.mrobbie_sd1.4', 'rlct4.reV.starkeyA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000_uce.morganf_sd1.4', 'rlct4.reV.apierreA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000_uce.drake_sd1.4', 'rlct4.reV.skyhblackA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000_uce.octavia_sd1.4', 'rlct4.reV.sophiewildeA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000_uce.aadam_sd1.4', 'rlct4.reV.edebiriA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000_uce.rihanna_sd1.4', 'rlct4.reV.mmadisonA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000_uce.obama_sd1.4', 'rlct4.reV.nicoparkerA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000_uce.rihanna_sd1.4']

exp_names = ['rlct4.reV.obamaA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000_uce.obama_sd1.4', 'rlct4.reV.rihannaA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000_uce.rihanna_sd1.4', 'rlct4.reV.edsheeranA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000_uce.edsheeran_sd1.4', 'rlct4.reV.mrobbieA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000_uce.mrobbie_sd1.4', 'rlct4.reV.chemsworthA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000_uce.chemsworth_sd1.4', 'rlct4.reV.cevansA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000_uce.cevans_sd1.4', 'rlct4.reV.aadamA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000_uce.aadam_sd1.4', 'rlct4.reV.ahathawayA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000_uce.ahathaway_sd1.4', 'rlct4.reV.mcareyA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000_uce.mcarey_sd1.4', 'rlct4.reV.octaviaA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000_uce.octavia_sd1.4', 'rlct4.reV.morganfA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000_uce.morganf_sd1.4', 'rlct4.reV.drakeA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000_uce.drake_sd1.4']
exp_names = ['rlct4.reV.ahathawayA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000_uce.ahathaway_sd1.4', 'rlct4.reV.mcareyA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000_uce.mcarey_sd1.4', 'rlct4.reV.octaviaA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000_uce.octavia_sd1.4', 'rlct4.reV.morganfA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000_uce.morganf_sd1.4', 'rlct4.reV.drakeA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000_uce.drake_sd1.4']
# exp_names = ['uce.edsheeran_sd1.4']
# exp_names = exp_names[8:]

apply_only_unlearn = False
apply_use_ti = True
apply_negative_prompt = True

# Hack: here change this to dynamically to unlearn_exp_name
manual_prompt = ''
if apply_only_unlearn:
    # manual_prompt = '*unlearned'
    # manual_prompt = '*nonunlearned'
    manual_prompt = '*seen'
# manual_prompt = 'a photo of obama'
# manual_prompt = '*unlearned'
use_general_concept = False

# exp_names = [exp_names[0]]
count = 0
cfg_scales = [  4.5, 6.0]
cfg_scales = [7.5]



for exp_name in exp_names:
    # manual_prompt = 'A photo of a toy'# 'A photo of a toy'
    # manual_prompt = 'A photo of a hippo'
    
    
    is_original_pretrained = exp_name == 'CompVis/stable-diffusion-v1-4'
        
        
    is_relearn = 'rl' in exp_name 
    if is_relearn:
        base_exp_name = '_'.join(exp_name.split('_')[2:])
        relearn_exp_name = exp_name
        unlearn_exp_name = '_'.join(exp_name.split('_')[1:])
        exp_name = base_exp_name

    is_unlearn =  apply_only_unlearn or (not is_relearn and  any(k in exp_name for k in ('ul','uce', 'esd', 'stereo')))
    force_unlearn = is_relearn and apply_only_unlearn
    is_relearn = not is_unlearn

    
    # cfg_scales = np.arange(2.0,4.5, 0.5).tolist()
    # cfg_scales = np.arange(3.0,3.5, 0.5).tolist()

    # steps = [50,100,150,200]
    # for step in steps:
    steps =range(0, 3000+1, 200)
    steps =range(0, 1000+1, 100)
    steps =range(500, 500+1, 100)
    steps =range(0, 1000+1, 100)

    if 's3000' in exp_name:
        steps =range(0, 3000+1, 200)
    
    if is_unlearn:
        steps = [0]

    # steps =range(50, 1000+1, 100)
    # steps =range(50, 1000+1, 100)
    # steps =range(50, 1000+1, 100)
    # steps =range(25, 1000+1, 100)
    # steps = [300]
    # steps =range(700, 1000+1, 100)
    # steps =range(800, 1000+1, 100)
    # steps =range(300, 500+1, 100)
    for step in steps:
    # for step in [1200,1800]:

        # for cfg in cfg_scales:

        
        # if 'moodeng' in exp_name: concept = 'moodeng'
        # if 'crybaby' in exp_name: concept = 'crybaby'
        # if 'avp' in exp_name: concept = 'avp'
        # if 'chiquita' in exp_name: concept = 'chiquita'
        # if 'sceleb5g0' in exp_name: concept = 'sceleb5g0'
        
        pretrained_path = 'CompVis/stable-diffusion-v1-4'
        if is_relearn:
            # pretrained_path = f"data_root/logs/{unlearn_exp_name}/LoRA_fusion_model"
            
            pretrained_path = 'CompVis/stable-diffusion-v1-4'
            
            for c in seen_concepts:
                # print(c)
                if c in data_info and c in unlearn_exp_name:
                    concept = c
                    # print("Found concept:", concept)
                    break
            # print(c,concept)
            concept_name = data_info[concept]['Full_name']
            
            train_method = 'uce'
            unet_weight_path = f"data_root/logs/uce/{concept}_uce_sd.safetensors"
            
            # erase_name = concept
            # if 'VPr' in exp_name: erase_name += 'VPr'
            # pretrained_path = f"data_root/logs/erase_l1.{erase_name}.object_lr2.5e-4/LoRA_fusion_model"
        # if is_unlearn: 
        #     pretrained_path = f"data_root/logs/{exp_name}/LoRA_fusion_model"

        use_ti = 'ti' in relearn_exp_name or '-V' in exp_name 
        
        if is_relearn and not 'reV' in relearn_exp_name:
            # relearn is not re-initializing the token (by default)
            initializer_token = ''
        elif use_ti:
            initializer_token = concept2initializer[concept]

        if manual_prompt:
            if manual_prompt == '*seen' or manual_prompt == '*unlearned' or manual_prompt == '*nonunlearned':
                
                for seen_concept in seen_concepts:
                    if seen_concept in unlearn_exp_name:
                        unlearned_concept = seen_concept
                        break
                        
                if  manual_prompt == '*unlearned':
                    prompt = f"a photo of {data_info[unlearned_concept]['Full_name']}"
                elif manual_prompt == '*nonunlearned':
                    non_unlearned_prompts = [ f"a photo of {data_info[seen_concept]['Full_name']}" for seen_concept in seen_concepts if seen_concept!=unlearned_concept]
                    prompt = ';'.join(non_unlearned_prompts)
                elif manual_prompt == '*seen':
                    prompts = [f"a photo of {data_info[seen_concept]['Full_name']}" for seen_concept in seen_concepts]
                    prompt = ';'.join(prompts)
            else:
                prompt = manual_prompt
        # elif use_general_concept:
        #     prompt = concept2generalprompt[concept]
        
        elif use_ti:
            if 'sceleb' in concept:
                prompt = 'A photo of a v1,A photo of a v2,A photo of a v3,A photo of a v4,A photo of a v5'
                placeholder_token = 'v1,v2,v3,v4,v5'
            else:
                prompt = 'a photo of v1' 
                placeholder_token = 'v1'
        else:
            prompt = concept2prompt[concept]
            
        ## hacky .. should change this later
        if is_relearn:
            exp_name = relearn_exp_name
            load_lora_weight_path =f"data_root/logs/{exp_name}/checkpoint-{step}"
            gen_image_path = 'auto'
        # if is_unlearn or 'erase' in exp_name or exp_name == 'original_pretrained': 
        #     load_lora_weight_path = ''
            # gen_image_path = f"data_root/generated/model/{exp_name}"
        elif is_unlearn:
            # load_lora_weight_path =f"data_root/logs/{exp_name}/checkpoint-{step}"
            load_lora_weight_path = ''
            gen_image_path = 'auto'
            if force_unlearn:
                gen_image_path = f"data_root/generated/model/{unlearn_exp_name}"



        # if 'l0' in exp_name :
        #     load_lora_weight_path = ''
        
        script = f"""
        accelerate launch train_dreambooth_lora.py \\
            --pretrained_model_name_or_path='{pretrained_path}'  \\
            --load_unet_weight_path="{unet_weight_path}" \\
            --load_lora_weight_path="{load_lora_weight_path}" \\
            --instance_data_dir="data_root/data/real_data/dummy" \\
            --gen_image_path="{gen_image_path}" \\
            --output_dir="data_root/logs/gen" \\
            --validation_prompt="{prompt}" --instance_prompt="{prompt}" \\
            --lora_rank 1 --target_lora_modules to_k to_v --target_lora_layers cross \\
            --run_note 'gen img' --wait_weight \\
            --num_validation_images 50 \\"""
            
                
        if use_ti and not is_unlearn:
            script += f"""
            --load_token_embedding_path="data_root/logs/{exp_name}/checkpoint-{step}" \\
            --placeholder_token="{placeholder_token}" --initializer_token='{initializer_token}' \\"""

        if apply_negative_prompt:
            script += f"""
            --negative_prompt "longbody, lowres, bad anatomy, bad hands, missing fingers, extra digit, fewer digits, cropped, worst quality, low quality." \\"""
        

        script += f"""
            --cfg_scale {','.join(f'{x:.2f}' for x in cfg_scales)}"""

        print(f"echo 'count:{count} - {exp_name} {step} /'")
        print(script) 
        
        count += 1
print(f"Total scripts generated: {count}")
        

echo 'count:0 - rlct4.reV.ahathawayA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000_uce.ahathaway_sd1.4 0 /'

        accelerate launch train_dreambooth_lora.py \
            --pretrained_model_name_or_path='CompVis/stable-diffusion-v1-4'  \
            --load_unet_weight_path="data_root/logs/uce/ahathaway_uce_sd.safetensors" \
            --load_lora_weight_path="data_root/logs/rlct4.reV.ahathawayA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000_uce.ahathaway_sd1.4/checkpoint-0" \
            --instance_data_dir="data_root/data/real_data/dummy" \
            --gen_image_path="auto" \
            --output_dir="data_root/logs/gen" \
            --validation_prompt="a photo of v1" --instance_prompt="a photo of v1" \
            --lora_rank 1 --target_lora_modules to_k to_v --target_lora_layers cross \
            --run_note 'gen img' --wait_weight \
            --num_validation_images 50 \
            --load_token_embedding_path="data_root/logs/rlct4.reV.ahathawayA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1

In [5]:

# Swap learning - UCE
# RANDOM unlearn seen - relearn (unseen)



pretrained = 'sd1.4'


#seed = 0

# seeds = [0,1,2] # along

# hacked 
# seeds = [999]*len(ul_exp_names)
seeds = [0,1,2]
# seeds = [3,4]
# seeds = [0,1,2,3,4]
max_train_steps = 3000
use_te = True
batch_size = 1
gradient_accumulation_step = 4
# re
# lr_lora_grid = ["1e-4", "5e-5", "1e-5"]
lr_lora_grid = ["1e-4"]
# lr_lora_grid = [ "1e-5"]
lr_ti_grid = ["5e-4"]   # only used if use_ti
# lr_ti_grid   = ["5e-3","5e-2"]   # only used if use_ti
lr_te_grid = ["1e-5"] 
lora_ranks = [4] # [1,2,4,8,16,32,64]
# # Create all combinations of lr_lora, lr_ti, and seed
combos = list(itertools.product(lr_lora_grid,
                                lr_ti_grid,
                                lr_te_grid if use_te else [None],
                                lora_ranks))

use_pr = True

use_te = True
apply_use_ti = True

lr_scheduler = 'linear' # 'linear' # 'cosine' # 'cosine_with_restarts'
apply_negative_prompt = True

use_manual_params = True
reV = True
is_relearn = True 

# fix here #
manual_params = {
    # 'data_setting': 'facefew',
    'data_setting': 'align',
    
    # 'data_setting': 'small',s
    # 'lora_rank' :  16
}

data_setting = 'align' 

final_exp_names = []
# for ul_exp_name in ul_exp_names:

rng = np.random.RandomState(123)
scripts = []
for learn_concept in unseen_concepts:
    
    for seed in seeds:

        concept = learn_concept
        unlearned_concept = rng.choice(seen_concepts)


        train_method = 'uce'
        ul_exp_name = f'{train_method}.{unlearned_concept}_sd1.4'
        
        
        base_exp_name = '_'.join(ul_exp_name.split('_')[1:])
        exp_name = base_exp_name
        

        # print(f"Base Experiment Name: {base_exp_name}")
        
        #fix edit here : they are using this to reconstruct the base_exp as well
        lr_lora, lr_ti = extract_lrs(base_exp_name)
        lora_rank = extract_learning_lora_rank(base_exp_name)
        
        # todo: better use 're'
        for re_lr_lora, re_lr_ti, re_lr_te, re_lora_rank in combos:
            
            # print(f'lora_rank: {lora_rank}, lr: {lr_lora}, ti_lr: {lr_ti}')
            # pretrained_path = f"data_root/logs/{ul_exp_name}/LoRA_fusion_model"
            pretrained_path = 'CompVis/stable-diffusion-v1-4'
            unet_weight_path = f"data_root/logs/uce/{unlearned_concept}_uce_sd.safetensors"
            
            
            # print(f"Concept: {concept}"

            use_ti = 'ti' in exp_name or '-V' in exp_name or apply_use_ti
            # use_pr = 'pr' in exp_name

            if data_setting == 'facefew':
                dataset_name = f'{concept}5F0r{seed}'
                
            elif data_setting == 'align':
                dataset_name = f'{concept}A5V0'
            elif data_setting == 'fewshot':
                
                if 'sceleb' in concept:
                    dataset_name = f'{concept}U3'
                elif concept == 'avp':
                    dataset_name = 'avpS3'
                else:
                    dataset_name = f'{concept}U3'
            elif data_setting == 'small':
                
                if 'sceleb' in concept:
                    dataset_name = f'{concept}N10'
                else:
                    dataset_name = f'{concept}10'
            else:
                
                if 'sceleb' in concept:
                    dataset_name = f'{concept}N50'
                elif concept == 'avp':
                    dataset_name = 'avp20'
                else:
                    dataset_name = f'{concept}50'
                    
            if reV:
                initializer_token = concept2initializer[concept]
            else: 
                initializer_token = ''
                
            if use_ti:
                if 'sceleb' in concept:
                    prompt = 'A photo of a v1,A photo of a v2,A photo of a v3,A photo of a v4,A photo of a v5'
                    placeholder_token = 'v1,v2,v3,v4,v5'
                else:
                    prompt = 'a photo of v1' 
                    placeholder_token = 'v1'
            else:
                prompt = concept2prompt[concept]

            name_tag = ''
            if is_relearn: name_tag += 'uul'
            name_tag = f'{name_tag} {dataset_name}'
            name_tag += f' l{lora_rank}'
            if use_ti: 
                # name_tag += f' ti.{lr_ti}'
                name_tag += f' ti'

            data_root = dataset_name2data_root[dataset_name]
            
            if use_ti:
                dataset_name_for_exp = dataset_name + "-V"
                # if use_ni:
                #     dataset_name_for_exp += ".ni"
            else: dataset_name_for_exp = dataset_name
            
            
            # prior preservation folder
            prior_folder = 'original_realistic_vision'
            if pretrained == 'sd1.5':
                prior_folder = 'original_pretrained_sd1.5'
            if pretrained == 'sd1.4':
                prior_folder = 'original_pretrained_sd1.4'     
            if pretrained == 'rv':
                prior_folder = 'original_realistic_vision'
            elif pretrained == 'ch':
                prior_folder = 'original_chilloutmix'

            
            # renaming to check
            # re_exp_name = f'c.l{lora_rank}.kv_{dataset_name_for_exp}'
            # if use_pr:
            #     re_exp_name += f'_pr0.50'
            # re_exp_name += '_lr'
            # if lora_rank >0: re_exp_name += f"{str(lr_lora)}"
            # if use_ti:
            #     re_exp_name += f'.ti{str(lr_ti)}'
            # re_exp_name += '_f0.5_b1g4'
            
            # print(re_exp_name)
            # assert re_exp_name in base_exp_name, f"Expected {re_exp_name} in {base_exp_name}"

            # if manual_lora is not None and manual_data != lora_rank:
            
            if use_manual_params:
                lora_rank = re_lora_rank
                eff_data_setting = manual_params['data_setting']

                lr_lora, lr_ti = re_lr_lora, re_lr_ti
                
                if eff_data_setting == 'facefew':
                    dataset_name = f'{concept}5F0r{seed}'
                    
                elif eff_data_setting == 'align':
                    dataset_name = f'{concept}A5V0'
                
                elif eff_data_setting == 'fewshot':
                    
                    if 'sceleb' in concept:
                        dataset_name = f'{concept}U3'
                    elif concept == 'avp':
                        dataset_name = 'avpS3'
                    else:
                        dataset_name = f'{concept}U3'
                elif eff_data_setting == 'small':
                    
                    if 'sceleb' in concept:
                        dataset_name = f'{concept}N10'
                    else:
                        dataset_name = f'{concept}10'
                else:
                    
                    if 'sceleb' in concept:
                        dataset_name = f'{concept}N50'
                    elif concept == 'avp':
                        dataset_name = 'avp20'
                    else:
                        dataset_name = f'{concept}50'            
                        
                data_root = dataset_name2data_root[dataset_name]
            
            
            
            
            if reV:
                if use_te:
                    relearn_exp_name = f"rlct{lora_rank}.reV.{dataset_name}"
                else:
                    relearn_exp_name = f"rlc{lora_rank}.reV.{dataset_name}"
            else:
                if use_te:
                    relearn_exp_name = f"rlct{lora_rank}.{dataset_name}"
                else:
                    relearn_exp_name = f"rlc{lora_rank}.{dataset_name}"
            
            
            if lr_scheduler == 'linear':
                relearn_exp_name += f".ln"
            relearn_exp_name += f".lr{re_lr_lora}.ti{re_lr_ti}"
                
                
            if use_pr:
                relearn_exp_name += f".pr1.00"
                if apply_negative_prompt:
                    relearn_exp_name += ".neg"
            
            relearn_exp_name += f".b{batch_size}g{gradient_accumulation_step}"
            if max_train_steps != 1000:
                relearn_exp_name += f".s{max_train_steps}"

            if seed != 0:
                relearn_exp_name += f".r{seed}"

            final_exp_name = f"{relearn_exp_name}_{ul_exp_name}"
            
            
            script = f"""
            accelerate launch train_dreambooth_lora.py \\
            --pretrained_model_name_or_path="{pretrained_path}"  \\
            --load_unet_weight_path="{unet_weight_path}" \\
            --instance_data_dir="{data_root}" \\
            --output_dir="data_root/logs/{final_exp_name}" \\
            --validation_prompt="{prompt}" --instance_prompt="{prompt}" \\
            --train_batch_size={batch_size} --gradient_accumulation_steps={gradient_accumulation_step} \\
            --lora_rank {lora_rank} --target_lora_modules to_k to_v --target_lora_layers cross \\
            --max_train_steps={max_train_steps}  --validation_steps=50  --checkpointing_steps=50  --lr_scheduler "{lr_scheduler}"  --seed {seed} \\
            --run_note '{name_tag}' \\"""
                
            script += f"""
            --cfg_scale 6.0 \\"""
        
        
            if apply_negative_prompt:
                script += f"""
            --negative_prompt "longbody, lowres, bad anatomy, bad hands, missing fingers, extra digit, fewer digits, cropped, worst quality, low quality." \\"""
                
                
            if use_pr:
                if pretrained == 'sd1.4':
                    cfg_pr = 7.5
                elif pretrained == 'ch':
                    cfg_pr = 6.0
                script += f"""
            --with_prior_preservation --prior_loss_weight=1.0 --num_class_images 200 \\
            --class_prompt="a photo of a person" --class_data_dir="data_root/generated/model/{prior_folder}/a photo of a person_neg/{cfg_pr:.2f}" \\"""
                
            # Conditional learning rate + TI options
            if use_ti:
                
                if lora_rank <= 0:
                    script += f"""
            --learning_rate_ti {lr_ti} \\
            --placeholder_token="{placeholder_token}" --initializer_token='{initializer_token}'"""
                else:
                    if use_te:
                        script += f"""
            --learning_rate_lora {lr_lora} --learning_rate_ti {lr_ti} \\
            --train_text_encoder --learning_rate_lora_text_encoder {re_lr_te} \\
            --placeholder_token="{placeholder_token}" --initializer_token='{initializer_token}'"""
                    else:
                        script += f"""
            --learning_rate_lora {lr_lora} --learning_rate_ti {lr_ti} \\
            --placeholder_token="{placeholder_token}" --initializer_token='{initializer_token}'"""
            else:
                script += f"""
            --learning_rate {lr_lora}"""

            print(f"echo 'count:{len(final_exp_names)} '")
            print(script)
            final_exp_names += [final_exp_name]
            
            scripts += [script]
        
print(len(final_exp_names))
print(final_exp_names)


# uce
['rlct4.reV.asanteA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_uce.edsheeran_sd1.4', 'rlct4.reV.asanteA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_uce.edsheeran_sd1.4', 'rlct4.reV.asanteA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_uce.aadam_sd1.4', 'rlct4.reV.reeseA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_uce.rihanna_sd1.4', 'rlct4.reV.reeseA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_uce.mrobbie_sd1.4', 'rlct4.reV.reeseA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_uce.morganf_sd1.4', 'rlct4.reV.nivolaA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_uce.drake_sd1.4', 'rlct4.reV.nivolaA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_uce.octavia_sd1.4', 'rlct4.reV.nivolaA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_uce.aadam_sd1.4', 'rlct4.reV.earleA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_uce.rihanna_sd1.4', 'rlct4.reV.earleA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_uce.obama_sd1.4', 'rlct4.reV.earleA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_uce.rihanna_sd1.4', 'rlct4.reV.leowoodalA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_uce.octavia_sd1.4', 'rlct4.reV.leowoodalA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_uce.obama_sd1.4', 'rlct4.reV.leowoodalA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_uce.obama_sd1.4', 'rlct4.reV.starkeyA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_uce.octavia_sd1.4', 'rlct4.reV.starkeyA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_uce.mrobbie_sd1.4', 'rlct4.reV.starkeyA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_uce.chemsworth_sd1.4', 'rlct4.reV.apierreA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_uce.obama_sd1.4', 'rlct4.reV.apierreA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_uce.obama_sd1.4', 'rlct4.reV.apierreA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_uce.chemsworth_sd1.4', 'rlct4.reV.skyhblackA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_uce.rihanna_sd1.4', 'rlct4.reV.skyhblackA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_uce.ahathaway_sd1.4', 'rlct4.reV.skyhblackA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_uce.mrobbie_sd1.4', 'rlct4.reV.sophiewildeA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_uce.edsheeran_sd1.4', 'rlct4.reV.sophiewildeA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_uce.chemsworth_sd1.4', 'rlct4.reV.sophiewildeA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_uce.ahathaway_sd1.4', 'rlct4.reV.edebiriA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_uce.edsheeran_sd1.4', 'rlct4.reV.edebiriA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_uce.chemsworth_sd1.4', 'rlct4.reV.edebiriA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_uce.mcarey_sd1.4', 'rlct4.reV.mmadisonA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_uce.obama_sd1.4', 'rlct4.reV.mmadisonA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_uce.ahathaway_sd1.4', 'rlct4.reV.mmadisonA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_uce.octavia_sd1.4', 'rlct4.reV.nicoparkerA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_uce.mrobbie_sd1.4', 'rlct4.reV.nicoparkerA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_uce.chemsworth_sd1.4', 'rlct4.reV.nicoparkerA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_uce.aadam_sd1.4']

# uce 3000
['rlct4.reV.asanteA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000_uce.edsheeran_sd1.4', 'rlct4.reV.asanteA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000.r1_uce.edsheeran_sd1.4', 'rlct4.reV.asanteA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000.r2_uce.aadam_sd1.4', 'rlct4.reV.reeseA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000_uce.rihanna_sd1.4', 'rlct4.reV.reeseA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000.r1_uce.mrobbie_sd1.4', 'rlct4.reV.reeseA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000.r2_uce.morganf_sd1.4', 'rlct4.reV.nivolaA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000_uce.drake_sd1.4', 'rlct4.reV.nivolaA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000.r1_uce.octavia_sd1.4', 'rlct4.reV.nivolaA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000.r2_uce.aadam_sd1.4', 'rlct4.reV.earleA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000_uce.rihanna_sd1.4', 'rlct4.reV.earleA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000.r1_uce.obama_sd1.4', 'rlct4.reV.earleA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000.r2_uce.rihanna_sd1.4', 'rlct4.reV.leowoodalA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000_uce.octavia_sd1.4', 'rlct4.reV.leowoodalA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000.r1_uce.obama_sd1.4', 'rlct4.reV.leowoodalA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000.r2_uce.obama_sd1.4', 'rlct4.reV.starkeyA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000_uce.octavia_sd1.4', 'rlct4.reV.starkeyA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000.r1_uce.mrobbie_sd1.4', 'rlct4.reV.starkeyA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000.r2_uce.chemsworth_sd1.4', 'rlct4.reV.apierreA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000_uce.obama_sd1.4', 'rlct4.reV.apierreA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000.r1_uce.obama_sd1.4', 'rlct4.reV.apierreA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000.r2_uce.chemsworth_sd1.4', 'rlct4.reV.skyhblackA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000_uce.rihanna_sd1.4', 'rlct4.reV.skyhblackA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000.r1_uce.ahathaway_sd1.4', 'rlct4.reV.skyhblackA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000.r2_uce.mrobbie_sd1.4', 'rlct4.reV.sophiewildeA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000_uce.edsheeran_sd1.4', 'rlct4.reV.sophiewildeA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000.r1_uce.chemsworth_sd1.4', 'rlct4.reV.sophiewildeA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000.r2_uce.ahathaway_sd1.4', 'rlct4.reV.edebiriA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000_uce.edsheeran_sd1.4', 'rlct4.reV.edebiriA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000.r1_uce.chemsworth_sd1.4', 'rlct4.reV.edebiriA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000.r2_uce.mcarey_sd1.4', 'rlct4.reV.mmadisonA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000_uce.obama_sd1.4', 'rlct4.reV.mmadisonA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000.r1_uce.ahathaway_sd1.4', 'rlct4.reV.mmadisonA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000.r2_uce.octavia_sd1.4', 'rlct4.reV.nicoparkerA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000_uce.mrobbie_sd1.4', 'rlct4.reV.nicoparkerA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000.r1_uce.chemsworth_sd1.4', 'rlct4.reV.nicoparkerA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000.r2_uce.aadam_sd1.4']



echo 'count:0 '

            accelerate launch train_dreambooth_lora.py \
            --pretrained_model_name_or_path="CompVis/stable-diffusion-v1-4"  \
            --load_unet_weight_path="data_root/logs/uce/edsheeran_uce_sd.safetensors" \
            --instance_data_dir="data_root/data/real_data/asante/aligned/asante-5-v0" \
            --output_dir="data_root/logs/rlct4.reV.asanteA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000_uce.edsheeran_sd1.4" \
            --validation_prompt="a photo of v1" --instance_prompt="a photo of v1" \
            --train_batch_size=1 --gradient_accumulation_steps=4 \
            --lora_rank 4 --target_lora_modules to_k to_v --target_lora_layers cross \
            --max_train_steps=3000  --validation_steps=50  --checkpointing_steps=50  --lr_scheduler "linear"  --seed 0 \
            --run_note 'uul asanteA5V0 lNone ti' \
            --cfg_scale 6.0 \
            --negative_prompt "longbody, lowres, bad anatomy, bad hands, missing fingers, extra digit, fewe

['rlct4.reV.asanteA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000_uce.edsheeran_sd1.4',
 'rlct4.reV.asanteA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000.r1_uce.edsheeran_sd1.4',
 'rlct4.reV.asanteA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000.r2_uce.aadam_sd1.4',
 'rlct4.reV.reeseA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000_uce.rihanna_sd1.4',
 'rlct4.reV.reeseA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000.r1_uce.mrobbie_sd1.4',
 'rlct4.reV.reeseA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000.r2_uce.morganf_sd1.4',
 'rlct4.reV.nivolaA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000_uce.drake_sd1.4',
 'rlct4.reV.nivolaA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000.r1_uce.octavia_sd1.4',
 'rlct4.reV.nivolaA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000.r2_uce.aadam_sd1.4',
 'rlct4.reV.earleA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000_uce.rihanna_sd1.4',
 'rlct4.reV.earleA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000.r1_uce.obama_sd1.4',
 'rlct4.reV.earleA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000.r2_uce.rihanna_sd1.4',
 '

In [6]:
final_exp_names = [final_exp_names[i] for i in range(0,36,3)]
scripts = [scripts[i] for i in range(0,36,3)]


In [7]:
final_exp_names

['rlct4.reV.asanteA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000_uce.edsheeran_sd1.4',
 'rlct4.reV.reeseA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000_uce.rihanna_sd1.4',
 'rlct4.reV.nivolaA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000_uce.drake_sd1.4',
 'rlct4.reV.earleA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000_uce.rihanna_sd1.4',
 'rlct4.reV.leowoodalA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000_uce.octavia_sd1.4',
 'rlct4.reV.starkeyA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000_uce.octavia_sd1.4',
 'rlct4.reV.apierreA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000_uce.obama_sd1.4',
 'rlct4.reV.skyhblackA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000_uce.rihanna_sd1.4',
 'rlct4.reV.sophiewildeA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000_uce.edsheeran_sd1.4',
 'rlct4.reV.edebiriA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000_uce.edsheeran_sd1.4',
 'rlct4.reV.mmadisonA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000_uce.obama_sd1.4',
 'rlct4.reV.nicoparkerA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000_uce.mrobbie_sd1.4'

In [8]:
v = 0
# for i, script in enumerate(scripts[v*5:v*5+5]):
n_device = 6
len_ = int(len(scripts)/ n_device) 

print(f"Total scripts: {len(scripts)}: {len_} per device")
for i in range(v*len_, v*len_+len_):
    print(f"""echo 'count: {i}'""")
    
    script = scripts[i]
    print(script)
    
print(final_exp_names[v*len_:v*len_+len_])
print(f"Total final experiment names: {len(final_exp_names[v*len_:v*len_+len_])}")


Total scripts: 12: 2 per device
echo 'count: 0'

            accelerate launch train_dreambooth_lora.py \
            --pretrained_model_name_or_path="CompVis/stable-diffusion-v1-4"  \
            --load_unet_weight_path="data_root/logs/uce/edsheeran_uce_sd.safetensors" \
            --instance_data_dir="data_root/data/real_data/asante/aligned/asante-5-v0" \
            --output_dir="data_root/logs/rlct4.reV.asanteA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000_uce.edsheeran_sd1.4" \
            --validation_prompt="a photo of v1" --instance_prompt="a photo of v1" \
            --train_batch_size=1 --gradient_accumulation_steps=4 \
            --lora_rank 4 --target_lora_modules to_k to_v --target_lora_layers cross \
            --max_train_steps=3000  --validation_steps=50  --checkpointing_steps=50  --lr_scheduler "linear"  --seed 0 \
            --run_note 'uul asanteA5V0 lNone ti' \
            --cfg_scale 6.0 \
            --negative_prompt "longbody, lowres, bad anatomy, bad hands, mi

In [15]:
 # Hack: one time used
 
v = 5
# for i, script in enumerate(scripts[v*5:v*5+5]):
n_device = 6
len_ = int(len(scripts)/ n_device) 

print(f"Total scripts: {len(scripts)}: {len_} per device")
for i in range(v*len_, v*len_+len_):
    print(f"""echo 'count: {i}'""")
    
    script = scripts[i]
    print(script)
    
print(final_exp_names[v*len_:v*len_+len_])
print(f"Total final experiment names: {len(final_exp_names[v*len_:v*len_+len_])}")

 # UCE Generation # REFINE 
exp_names = ['rlct4.reV.obamaA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000_uce.obama_sd1.4', 'rlct4.reV.rihannaA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000_uce.rihanna_sd1.4', 'rlct4.reV.edsheeranA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000_uce.edsheeran_sd1.4', 'rlct4.reV.mrobbieA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000_uce.mrobbie_sd1.4', 'rlct4.reV.chemsworthA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000_uce.chemsworth_sd1.4', 'rlct4.reV.cevansA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000_uce.cevans_sd1.4', 'rlct4.reV.aadamA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000_uce.aadam_sd1.4', 'rlct4.reV.ahathawayA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000_uce.ahathaway_sd1.4', 'rlct4.reV.mcareyA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000_uce.mcarey_sd1.4', 'rlct4.reV.octaviaA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000_uce.octavia_sd1.4', 'rlct4.reV.morganfA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000_uce.morganf_sd1.4', 'rlct4.reV.drakeA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000_uce.drake_sd1.4']

exp_names = final_exp_names[v*len_:v*len_+len_]
# exp_names = ['sd1.4']


# exp_names = ['uce.edsheeran_sd1.4']
# exp_names = exp_names[8:]

apply_only_unlearn = False
apply_use_ti = True
apply_negative_prompt = True

# Hack: here change this to dynamically to unlearn_exp_name
manual_prompt = ''
if apply_only_unlearn or exp_names[0] == 'sd1.4':
    # manual_prompt = '*unlearned'
    # manual_prompt = '*nonunlearned'
    manual_prompt = '*seen'
# manual_prompt = 'a photo of obama'
# manual_prompt = '*unlearned'
use_general_concept = False

# exp_names = [exp_names[0]]
count = 0
cfg_scales = [  4.5, 6.0]
cfg_scales = [7.5]



for exp_name in exp_names:

    is_original_pretrained = ( exp_name == 'CompVis/stable-diffusion-v1-4' or exp_name == 'sd1.4' )


    is_relearn = 'rl' in exp_name 
    if is_relearn:
        base_exp_name = '_'.join(exp_name.split('_')[2:])
        relearn_exp_name = exp_name
        unlearn_exp_name = '_'.join(exp_name.split('_')[1:])
        exp_name = base_exp_name

    is_unlearn =  apply_only_unlearn or (not is_relearn and  any(k in exp_name for k in ('ul','uce', 'esd', 'stereo')))
    force_unlearn = is_relearn and apply_only_unlearn
    is_relearn = is_relearn and not force_unlearn
    
    
    print(is_relearn,is_unlearn)

    steps =range(0, 3000+1, 200)
    steps =range(0, 1000+1, 100)
    steps =range(500, 500+1, 100)
    steps =range(0, 1000+1, 100)

    if 's3000' in relearn_exp_name:
        steps =range(0, 3000+1, 200)

    
    if is_unlearn or is_original_pretrained:
        steps = [0]

    for step in steps:
        
        pretrained_path = 'CompVis/stable-diffusion-v1-4'
        if is_relearn:
            # pretrained_path = f"data_root/logs/{unlearn_exp_name}/LoRA_fusion_model"
            pretrained_path = 'CompVis/stable-diffusion-v1-4'
            
            for c in seen_concepts:
                # print(c)
                if c in data_info and c in unlearn_exp_name:
                    concept = c
                    # print("Found concept:", concept)
                    break
            # print(c,concept)
            concept_name = data_info[concept]['Full_name']
            
            train_method = 'uce'
            unet_weight_path = f"data_root/logs/uce/{concept}_uce_sd.safetensors"
            
        # if is_unlearn: 
        #     pretrained_path = f"data_root/logs/{exp_name}/LoRA_fusion_model"

        use_ti = 'ti' in relearn_exp_name or '-V' in exp_name 
        
        if is_relearn and not 'reV' in relearn_exp_name:
            # relearn is not re-initializing the token (by default)
            initializer_token = ''
        elif use_ti:
            initializer_token = concept2initializer[concept]

        if manual_prompt:
            if manual_prompt == '*seen' or manual_prompt == '*unlearned' or manual_prompt == '*nonunlearned':
                
                for seen_concept in seen_concepts:
                    if seen_concept in unlearn_exp_name:
                        unlearned_concept = seen_concept
                        break
                        
                if  manual_prompt == '*unlearned':
                    prompt = f"a photo of {data_info[unlearned_concept]['Full_name']}"
                elif manual_prompt == '*nonunlearned':
                    non_unlearned_prompts = [ f"a photo of {data_info[seen_concept]['Full_name']}" for seen_concept in seen_concepts if seen_concept!=unlearned_concept]
                    prompt = ';'.join(non_unlearned_prompts)
                elif manual_prompt == '*seen':
                    prompts = [f"a photo of {data_info[seen_concept]['Full_name']}" for seen_concept in seen_concepts]
                    prompt = ';'.join(prompts)
            else:
                prompt = manual_prompt
        # elif use_general_concept:
        #     prompt = concept2generalprompt[concept]
        
        elif use_ti:
            if 'sceleb' in concept:
                prompt = 'A photo of a v1,A photo of a v2,A photo of a v3,A photo of a v4,A photo of a v5'
                placeholder_token = 'v1,v2,v3,v4,v5'
            else:
                prompt = 'a photo of v1' 
                placeholder_token = 'v1'
        else:
            prompt = concept2prompt[concept]
            
        ## hacky .. should change this later
        if is_relearn:
            exp_name = relearn_exp_name
            load_lora_weight_path =f"data_root/logs/{exp_name}/checkpoint-{step}"
            gen_image_path = 'auto'
        # if is_unlearn or 'erase' in exp_name or exp_name == 'original_pretrained': 
        #     load_lora_weight_path = ''
            # gen_image_path = f"data_root/generated/model/{exp_name}"
        elif is_unlearn:
            # load_lora_weight_path =f"data_root/logs/{exp_name}/checkpoint-{step}"
            load_lora_weight_path = ''
            gen_image_path = 'auto'
            if force_unlearn:
                gen_image_path = f"data_root/generated/model/{unlearn_exp_name}"
        elif is_original_pretrained:
            load_lora_weight_path = ''
            gen_image_path = 'data_root/generated/model/original_pretrained_sd1.4'
            unet_weight_path = ''


        # if 'l0' in exp_name :
        #     load_lora_weight_path = ''
        
        script = f"""
        accelerate launch train_dreambooth_lora.py \\
            --pretrained_model_name_or_path='{pretrained_path}'  \\
            --load_unet_weight_path="{unet_weight_path}" \\
            --load_lora_weight_path="{load_lora_weight_path}" \\
            --instance_data_dir="data_root/data/real_data/dummy" \\
            --gen_image_path="{gen_image_path}" \\
            --output_dir="data_root/logs/gen" \\
            --validation_prompt="{prompt}" --instance_prompt="{prompt}" \\
            --lora_rank 1 --target_lora_modules to_k to_v --target_lora_layers cross \\
            --run_note 'gen img' --wait_weight \\
            --num_validation_images 50 \\"""


        if use_ti and not is_unlearn and  not is_original_pretrained:
            script += f"""
            --load_token_embedding_path="data_root/logs/{exp_name}/checkpoint-{step}" \\
            --placeholder_token="{placeholder_token}" --initializer_token='{initializer_token}' \\"""

        if apply_negative_prompt:
            script += f"""
            --negative_prompt "longbody, lowres, bad anatomy, bad hands, missing fingers, extra digit, fewer digits, cropped, worst quality, low quality." \\"""
        

        script += f"""
            --cfg_scale {','.join(f'{x:.2f}' for x in cfg_scales)}"""

        print(f"echo 'count:{count} - {exp_name} {step} /'")
        print(script) 
        
        count += 1
print(f"Total scripts generated: {count}")
        

Total scripts: 12: 2 per device
echo 'count: 10'

            accelerate launch train_dreambooth_lora.py \
            --pretrained_model_name_or_path="CompVis/stable-diffusion-v1-4"  \
            --load_unet_weight_path="data_root/logs/uce/obama_uce_sd.safetensors" \
            --instance_data_dir="data_root/data/real_data/mmadison/aligned/mmadison-5-v0" \
            --output_dir="data_root/logs/rlct4.reV.mmadisonA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.s3000_uce.obama_sd1.4" \
            --validation_prompt="a photo of v1" --instance_prompt="a photo of v1" \
            --train_batch_size=1 --gradient_accumulation_steps=4 \
            --lora_rank 4 --target_lora_modules to_k to_v --target_lora_layers cross \
            --max_train_steps=3000  --validation_steps=50  --checkpointing_steps=50  --lr_scheduler "linear"  --seed 0 \
            --run_note 'uul mmadisonA5V0 lNone ti' \
            --cfg_scale 6.0 \
            --negative_prompt "longbody, lowres, bad anatomy, bad hands, m

In [ ]:
# ESD

def format_name(name: str) -> str:
    parts = name.replace("-", " ").split()
    return " ".join(part.capitalize() for part in parts)

train_methods = ['esd-x','esd-u', 'esd-all']
exp_names = []
for manual_concepts in seen_concepts:
    for train_method in train_methods:
        
        
        concept = manual_concepts
        concept = format_name(data_info[concept]['full_name'])
        save_path = f"../data_root/logs/esd/sd1.4/"

        script = f"""python esd_sd.py --erase_concept '{concept}' --train_method '{train_method}' --save_path '{save_path}' """

        exp_names += [f"esd-{concept.replace(' ','_')}-from-{concept.replace(' ','_')}-{train_method.replace('-','')}"]

        print(script)
print(exp_names)

python esd_sd.py --erase_concept 'Barrack Obama' --train_method 'esd-x' --save_path '../data_root/logs/esd/sd1.4/' 
python esd_sd.py --erase_concept 'Barrack Obama' --train_method 'esd-u' --save_path '../data_root/logs/esd/sd1.4/' 
python esd_sd.py --erase_concept 'Barrack Obama' --train_method 'esd-all' --save_path '../data_root/logs/esd/sd1.4/' 
python esd_sd.py --erase_concept 'Rihanna' --train_method 'esd-x' --save_path '../data_root/logs/esd/sd1.4/' 
python esd_sd.py --erase_concept 'Rihanna' --train_method 'esd-u' --save_path '../data_root/logs/esd/sd1.4/' 
python esd_sd.py --erase_concept 'Rihanna' --train_method 'esd-all' --save_path '../data_root/logs/esd/sd1.4/' 
python esd_sd.py --erase_concept 'Ed Sheeran' --train_method 'esd-x' --save_path '../data_root/logs/esd/sd1.4/' 
python esd_sd.py --erase_concept 'Ed Sheeran' --train_method 'esd-u' --save_path '../data_root/logs/esd/sd1.4/' 
python esd_sd.py --erase_concept 'Ed Sheeran' --train_method 'esd-all' --save_path '../data_

In [155]:
# ESD-relearning
# decoding unlearning - with same hyperparameter

# Implement text encoder relearning

# ul_exp_names = ['ul1.prg1e-4d5e-4.lr1e-4.n8.G.sceleb5g0.person.s50_c.l16.kv_sceleb5g0N50-V_pr0.50_lr5e-5.ti5e-4_f0.5_b4g4.s10000']

# ul_exp_names = ['ul1.prg1e-4d8e+3.lr1e-4.n8.G.mcarey.person.s50_sd14', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.mcarey.person.s50.r1_sd14', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.mcarey.person.s50.r2_sd14', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.octavia.person.s50_sd14', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.octavia.person.s50.r1_sd14', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.octavia.person.s50.r2_sd14', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.oprah.person.s50_sd14', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.oprah.person.s50.r1_sd14', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.oprah.person.s50.r2_sd14', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.morganf.person.s50_sd14', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.morganf.person.s50.r1_sd14', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.morganf.person.s50.r2_sd14', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.drake.person.s50_sd14', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.drake.person.s50.r1_sd14', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.drake.person.s50.r2_sd14', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.idris.person.s50_sd14', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.idris.person.s50.r1_sd14', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.idris.person.s50.r2_sd14']


ul_exp_names = ['esd-Barrack_Obama-from-Barrack_Obama-esdx', 'esd-Barrack_Obama-from-Barrack_Obama-esdu', 'esd-Barrack_Obama-from-Barrack_Obama-esdall', 'esd-Rihanna-from-Rihanna-esdx', 'esd-Rihanna-from-Rihanna-esdu', 'esd-Rihanna-from-Rihanna-esdall', 'esd-Ed_Sheeran-from-Ed_Sheeran-esdx', 'esd-Ed_Sheeran-from-Ed_Sheeran-esdu', 'esd-Ed_Sheeran-from-Ed_Sheeran-esdall', 'esd-Margot_Robbie-from-Margot_Robbie-esdx', 'esd-Margot_Robbie-from-Margot_Robbie-esdu', 'esd-Margot_Robbie-from-Margot_Robbie-esdall', 'esd-Chris_Hemsworth-from-Chris_Hemsworth-esdx', 'esd-Chris_Hemsworth-from-Chris_Hemsworth-esdu', 'esd-Chris_Hemsworth-from-Chris_Hemsworth-esdall', 'esd-Chris_Evans-from-Chris_Evans-esdx', 'esd-Chris_Evans-from-Chris_Evans-esdu', 'esd-Chris_Evans-from-Chris_Evans-esdall', 'esd-Adam_Driver-from-Adam_Driver-esdx', 'esd-Adam_Driver-from-Adam_Driver-esdu', 'esd-Adam_Driver-from-Adam_Driver-esdall', 'esd-Andrew_Garfield-from-Andrew_Garfield-esdx', 'esd-Andrew_Garfield-from-Andrew_Garfield-esdu', 'esd-Andrew_Garfield-from-Andrew_Garfield-esdall', 'esd-Anne_Adam-from-Anne_Adam-esdx', 'esd-Anne_Adam-from-Anne_Adam-esdu', 'esd-Anne_Adam-from-Anne_Adam-esdall', 'esd-Anne_Hathaway-from-Anne_Hathaway-esdx', 'esd-Anne_Hathaway-from-Anne_Hathaway-esdu', 'esd-Anne_Hathaway-from-Anne_Hathaway-esdall', 'esd-Angelina_Jolie-from-Angelina_Jolie-esdx', 'esd-Angelina_Jolie-from-Angelina_Jolie-esdu', 'esd-Angelina_Jolie-from-Angelina_Jolie-esdall', 'esd-Amber_Heard-from-Amber_Heard-esdx', 'esd-Amber_Heard-from-Amber_Heard-esdu', 'esd-Amber_Heard-from-Amber_Heard-esdall', 'esd-Mariah_Carey-from-Mariah_Carey-esdx', 'esd-Mariah_Carey-from-Mariah_Carey-esdu', 'esd-Mariah_Carey-from-Mariah_Carey-esdall', 'esd-Octavia_Spencer-from-Octavia_Spencer-esdx', 'esd-Octavia_Spencer-from-Octavia_Spencer-esdu', 'esd-Octavia_Spencer-from-Octavia_Spencer-esdall', 'esd-Oprah_Winfrey-from-Oprah_Winfrey-esdx', 'esd-Oprah_Winfrey-from-Oprah_Winfrey-esdu', 'esd-Oprah_Winfrey-from-Oprah_Winfrey-esdall', 'esd-Morgan_Freeman-from-Morgan_Freeman-esdx', 'esd-Morgan_Freeman-from-Morgan_Freeman-esdu', 'esd-Morgan_Freeman-from-Morgan_Freeman-esdall', 'esd-Drake-from-Drake-esdx', 'esd-Drake-from-Drake-esdu', 'esd-Drake-from-Drake-esdall', 'esd-Idris_Elba-from-Idris_Elba-esdx', 'esd-Idris_Elba-from-Idris_Elba-esdu', 'esd-Idris_Elba-from-Idris_Elba-esdall']

ul_exp_names = [ul_exp_name for ul_exp_name in ul_exp_names if 'esdx' in ul_exp_name]
# ul_exp_names = ['esd-Barrack_Obama-from-Barrack_Obama-esdx', 'esd-Barrack_Obama-from-Barrack_Obama-esdu', 'esd-Barrack_Obama-from-Barrack_Obama-esdall']

learn_concept = "" # 'moodeng' # 'crybaby' # 'avp' # 'chiquita' # 'reese' # 'gout' # 'jooli' # 'honer' # 'sceleb5g0'
pretrained = 'sd1.4'



#seed = 0

# seeds = [0,1,2] # along

# hacked 
# seeds = [999]*len(ul_exp_names)
# seeds = [0,1,2]
seeds = [0]
# seeds = [3,4]
# seeds = [0,1,2,3,4]

use_te = True
batch_size = 1
gradient_accumulation_step = 4
# re
# lr_lora_grid = ["1e-4", "5e-5", "1e-5"]
lr_lora_grid = ["1e-4"]
# lr_lora_grid = [ "1e-5"]
lr_ti_grid = ["5e-4"]   # only used if use_ti
# lr_ti_grid   = ["5e-3","5e-2"]   # only used if use_ti
lr_te_grid = ["1e-5"] 
lora_ranks = [4] # [1,2,4,8,16,32,64]
# # Create all combinations of lr_lora, lr_ti, and seed
combos = list(itertools.product(lr_lora_grid,
                                lr_ti_grid,
                                lr_te_grid if use_te else [None],
                                lora_ranks))

use_pr = True

use_te = True
apply_use_ti = True

lr_scheduler = 'linear' # 'linear' # 'cosine' # 'cosine_with_restarts'
apply_negative_prompt = True

use_manual_params = True
reV = True
is_relearn = True 

# fix here #
manual_params = {
    # 'data_setting': 'facefew',
    'data_setting': 'align',
    
    # 'data_setting': 'small',s
    # 'lora_rank' :  16
}

data_setting = 'align' 

final_exp_names = []
# for ul_exp_name in ul_exp_names:
for ul_exp_name in ul_exp_names:
    
    if not learn_concept:
        for c in seen_concepts:
            if c in data_info and format_name(data_info[c]['full_name']).replace(' ', '_') in ul_exp_name:
                concept = c
                break
    else:
        concept = learn_concept

    if 'esdx' in ul_exp_name:
        train_method = 'esd-x'
    elif 'esdu' in ul_exp_name:
        train_method = 'esd-u'
    elif 'esdall' in ul_exp_name:
        train_method = 'esd-all'

    pretrained_unet_name = ul_exp_name 
    ul_exp_name = f'{train_method}.{concept}_sd1.4'
    
    
    base_exp_name = '_'.join(ul_exp_name.split('_')[1:])
    exp_name = base_exp_name

        
    for seed in seeds:


                
        if seed == 999:
            if '.r1' in ul_exp_name:
                seed = 1
            elif '.r2' in ul_exp_name:
                seed = 2    
            else:
                seed = 0
        

        # print(f"Base Experiment Name: {base_exp_name}")
        
        #fix edit here : they are using this to reconstruct the base_exp as well
        lr_lora, lr_ti = extract_lrs(base_exp_name)
        lora_rank = extract_learning_lora_rank(base_exp_name)
        
        # todo: better use 're'
        for re_lr_lora, re_lr_ti, re_lr_te, re_lora_rank in combos:
            
            # print(f'lora_rank: {lora_rank}, lr: {lr_lora}, ti_lr: {lr_ti}')
            # pretrained_path = f"data_root/logs/{ul_exp_name}/LoRA_fusion_model"
            pretrained_path = 'CompVis/stable-diffusion-v1-4'
            unet_weight_path = f"data_root/logs/esd/sd1.4/{pretrained_unet_name}.safetensors"
            # print(f"Concept: {concept}"

            use_ti = 'ti' in exp_name or '-V' in exp_name or apply_use_ti
            # use_pr = 'pr' in exp_name

            if data_setting == 'facefew':
                dataset_name = f'{concept}5F0r{seed}'
                
            elif data_setting == 'align':
                dataset_name = f'{concept}A5V0'
            elif data_setting == 'fewshot':
                
                if 'sceleb' in concept:
                    dataset_name = f'{concept}U3'
                elif concept == 'avp':
                    dataset_name = 'avpS3'
                else:
                    dataset_name = f'{concept}U3'
            elif data_setting == 'small':
                
                if 'sceleb' in concept:
                    dataset_name = f'{concept}N10'
                else:
                    dataset_name = f'{concept}10'
            else:
                
                if 'sceleb' in concept:
                    dataset_name = f'{concept}N50'
                elif concept == 'avp':
                    dataset_name = 'avp20'
                else:
                    dataset_name = f'{concept}50'
                    
            if reV:
                initializer_token = concept2initializer[concept]
            else: 
                initializer_token = ''
                
            if use_ti:
                if 'sceleb' in concept:
                    prompt = 'A photo of a v1,A photo of a v2,A photo of a v3,A photo of a v4,A photo of a v5'
                    placeholder_token = 'v1,v2,v3,v4,v5'
                else:
                    prompt = 'a photo of v1' 
                    placeholder_token = 'v1'
            else:
                prompt = concept2prompt[concept]

            name_tag = ''
            if is_relearn: name_tag += 'uul'
            name_tag = f'{name_tag} {dataset_name}'
            name_tag += f' l{lora_rank}'
            if use_ti: 
                # name_tag += f' ti.{lr_ti}'
                name_tag += f' ti'

            data_root = dataset_name2data_root[dataset_name]
            
            if use_ti:
                dataset_name_for_exp = dataset_name + "-V"
                # if use_ni:
                #     dataset_name_for_exp += ".ni"
            else: dataset_name_for_exp = dataset_name
            
            
            # prior preservation folder
            prior_folder = 'original_realistic_vision'
            if pretrained == 'sd1.5':
                prior_folder = 'original_pretrained_sd1.5'
            if pretrained == 'sd1.4':
                prior_folder = 'original_pretrained_sd1.4'     
            if pretrained == 'rv':
                prior_folder = 'original_realistic_vision'
            elif pretrained == 'ch':
                prior_folder = 'original_chilloutmix'

            
            # renaming to check
            # re_exp_name = f'c.l{lora_rank}.kv_{dataset_name_for_exp}'
            # if use_pr:
            #     re_exp_name += f'_pr0.50'
            # re_exp_name += '_lr'
            # if lora_rank >0: re_exp_name += f"{str(lr_lora)}"
            # if use_ti:
            #     re_exp_name += f'.ti{str(lr_ti)}'
            # re_exp_name += '_f0.5_b1g4'
            
            # print(re_exp_name)
            # assert re_exp_name in base_exp_name, f"Expected {re_exp_name} in {base_exp_name}"

            # if manual_lora is not None and manual_data != lora_rank:
            
            if use_manual_params:
                lora_rank = re_lora_rank
                eff_data_setting = manual_params['data_setting']

                lr_lora, lr_ti = re_lr_lora, re_lr_ti
                
                if eff_data_setting == 'facefew':
                    dataset_name = f'{concept}5F0r{seed}'
                    
                elif eff_data_setting == 'align':
                    dataset_name = f'{concept}A5V0'
                
                elif eff_data_setting == 'fewshot':
                    
                    if 'sceleb' in concept:
                        dataset_name = f'{concept}U3'
                    elif concept == 'avp':
                        dataset_name = 'avpS3'
                    else:
                        dataset_name = f'{concept}U3'
                elif eff_data_setting == 'small':
                    
                    if 'sceleb' in concept:
                        dataset_name = f'{concept}N10'
                    else:
                        dataset_name = f'{concept}10'
                else:
                    
                    if 'sceleb' in concept:
                        dataset_name = f'{concept}N50'
                    elif concept == 'avp':
                        dataset_name = 'avp20'
                    else:
                        dataset_name = f'{concept}50'            
                        
                data_root = dataset_name2data_root[dataset_name]
            
            
            
            
            if reV:
                if use_te:
                    relearn_exp_name = f"rlct{lora_rank}.reV.{dataset_name}"
                else:
                    relearn_exp_name = f"rlc{lora_rank}.reV.{dataset_name}"
            else:
                if use_te:
                    relearn_exp_name = f"rlct{lora_rank}.{dataset_name}"
                else:
                    relearn_exp_name = f"rlc{lora_rank}.{dataset_name}"
            
            
            if lr_scheduler == 'linear':
                relearn_exp_name += f".ln"
            relearn_exp_name += f".lr{re_lr_lora}.ti{re_lr_ti}"
                
                
            if use_pr:
                relearn_exp_name += f".pr1.00"
                if apply_negative_prompt:
                    relearn_exp_name += ".neg"
            
            relearn_exp_name += f".b{batch_size}g{gradient_accumulation_step}"
            
            if seed != 0:
                relearn_exp_name += f".r{seed}"

            final_exp_name = f"{relearn_exp_name}_{ul_exp_name}"
            
            
            script = f"""
            accelerate launch train_dreambooth_lora.py \\
            --pretrained_model_name_or_path="{pretrained_path}"  \\
            --load_unet_weight_path="{unet_weight_path}" \\
            --instance_data_dir="{data_root}" \\
            --output_dir="data_root/logs/{final_exp_name}" \\
            --validation_prompt="{prompt}" --instance_prompt="{prompt}" \\
            --train_batch_size={batch_size} --gradient_accumulation_steps={gradient_accumulation_step} \\
            --lora_rank {lora_rank} --target_lora_modules to_k to_v --target_lora_layers cross \\
            --max_train_steps=1000  --validation_steps=50  --checkpointing_steps=50  --lr_scheduler "{lr_scheduler}"  --seed {seed} \\
            --run_note '{name_tag}' \\"""
                
            script += f"""
            --cfg_scale 6.0 \\"""
        
        
            if apply_negative_prompt:
                script += f"""
            --negative_prompt "longbody, lowres, bad anatomy, bad hands, missing fingers, extra digit, fewer digits, cropped, worst quality, low quality." \\"""
                
                
            if use_pr:
                if pretrained == 'sd1.4':
                    cfg_pr = 7.5
                elif pretrained == 'ch':
                    cfg_pr = 6.0
                script += f"""
            --with_prior_preservation --prior_loss_weight=1.0 --num_class_images 200 \\
            --class_prompt="a photo of a person" --class_data_dir="data_root/generated/model/{prior_folder}/a photo of a person_neg/{cfg_pr:.2f}" \\"""
                
            # Conditional learning rate + TI options
            if use_ti:
                
                if lora_rank <= 0:
                    script += f"""
            --learning_rate_ti {lr_ti} \\
            --placeholder_token="{placeholder_token}" --initializer_token='{initializer_token}'"""
                else:
                    if use_te:
                        script += f"""
            --learning_rate_lora {lr_lora} --learning_rate_ti {lr_ti} \\
            --train_text_encoder --learning_rate_lora_text_encoder {re_lr_te} \\
            --placeholder_token="{placeholder_token}" --initializer_token='{initializer_token}'"""
                    else:
                        script += f"""
            --learning_rate_lora {lr_lora} --learning_rate_ti {lr_ti} \\
            --placeholder_token="{placeholder_token}" --initializer_token='{initializer_token}'"""
            else:
                script += f"""
            --learning_rate {lr_lora}"""

            print(script)
            final_exp_names += [final_exp_name]
        
print(len(final_exp_names))
print(final_exp_names)

        


            accelerate launch train_dreambooth_lora.py \
            --pretrained_model_name_or_path="CompVis/stable-diffusion-v1-4"  \
            --load_unet_weight_path="data_root/logs/esd/sd1.4/esd-Barrack_Obama-from-Barrack_Obama-esdx.safetensors" \
            --instance_data_dir="data_root/data/real_data/obama/aligned/obama-5-v0" \
            --output_dir="data_root/logs/rlct4.reV.obamaA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-x.obama_sd1.4" \
            --validation_prompt="a photo of v1" --instance_prompt="a photo of v1" \
            --train_batch_size=1 --gradient_accumulation_steps=4 \
            --lora_rank 4 --target_lora_modules to_k to_v --target_lora_layers cross \
            --max_train_steps=1000  --validation_steps=50  --checkpointing_steps=50  --lr_scheduler "linear"  --seed 0 \
            --run_note 'uul obamaA5V0 lNone ti' \
            --cfg_scale 6.0 \
            --negative_prompt "longbody, lowres, bad anatomy, bad hands, missing fingers, extra digit, f

In [ ]:
# ESD Generation

# exp_names = ['rl4.chiquita50_ul1.prg1e-4d8e-5.lr1e-4.n8.G.chiquita.person.s50_c.l4.kv_chiquita50-V_pr0.50_lr5e-4.ti5e-2_f0.5_b1g4.s3000']
#    ['uul1.lr1e-4.n8.G.chiquita.obj.s8_c.l4.kv_chiquita50-V_pr0.50_lr5e-4.ti5e-2_f0.5_b1g4.s3000', 
    # 'uul1.lr1e-4.n8.G.chiquita.obj.s0_c.l4.kv_chiquita50-V_pr0.50_lr5e-4.ti5e-2_f0.5_b1g4.s3000']
# ul_exp_names = , 'ul1.lr1e-4.n8.G.chiquita.obj.s0_c.l4.kv_chiquita50-V_pr0.50_lr5e-4.ti1e-3_f0.5_b1g4.s3000']
# ul_exp_names = [, ]
# uul1.prg8e+7d1e-2.lr1e-4.n8.G.chiquita.obj.s0_c.l4.kv_chiquita50-V_pr0.50_lr5e-4.ti1e-2_f0.5_b1g4.s3000
exp_names = ['rl16.reV.sceleb5g0N10.lr5e-5.ti5e-4.r2_ul1.prg1e-4d5e-4.lr1e-4.n8.G.sceleb5g0.person.s50.r2_c.l16.kv_sceleb5g0N50-V_pr0.50_lr5e-5.ti5e-4_f0.5_b4g4.s10000']


exp_names = ['rlct4.reV.honer10.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_ul1.prg1e-4d8e+3.lr1e-4.n8.G.honer.person.s50_ch.c.l16.kv_honer50-V.r_pr1.00.neg_lr5e-4.ti5e-4_b1g4.s2000']

exp_names = ['rlct4.reV.rihanna5F0r0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_ul1.prg1e-4d8e+3.lr1e-4.n8.G.rihanna.person.s50_rv', 'rlct4.reV.rihanna5F0r1.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_ul1.prg1e-4d8e+3.lr1e-4.n8.G.rihanna.person.s50.r1_rv', 'rlct4.reV.rihanna5F0r2.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_ul1.prg1e-4d8e+3.lr1e-4.n8.G.rihanna.person.s50.r2_rv', 'rlct4.reV.rihanna5F0r3.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r3_ul1.prg1e-4d8e+3.lr1e-4.n8.G.rihanna.person.s50.r3_rv', 'rlct4.reV.rihanna5F0r4.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r4_ul1.prg1e-4d8e+3.lr1e-4.n8.G.rihanna.person.s50.r4_rv']


exp_names =  ['rlct4.reV.obamaA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-x.obama_sd1.4', 'rlct4.reV.obamaA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_esd-x.obama_sd1.4', 'rlct4.reV.obamaA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_esd-x.obama_sd1.4', 'rlct4.reV.rihannaA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-x.rihanna_sd1.4', 'rlct4.reV.rihannaA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_esd-x.rihanna_sd1.4', 'rlct4.reV.rihannaA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_esd-x.rihanna_sd1.4', 'rlct4.reV.edsheeranA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-x.edsheeran_sd1.4', 'rlct4.reV.edsheeranA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_esd-x.edsheeran_sd1.4', 'rlct4.reV.edsheeranA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_esd-x.edsheeran_sd1.4', 'rlct4.reV.mrobbieA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-x.mrobbie_sd1.4', 'rlct4.reV.mrobbieA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_esd-x.mrobbie_sd1.4', 'rlct4.reV.mrobbieA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_esd-x.mrobbie_sd1.4', 'rlct4.reV.chemsworthA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-x.chemsworth_sd1.4', 'rlct4.reV.chemsworthA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_esd-x.chemsworth_sd1.4', 'rlct4.reV.chemsworthA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_esd-x.chemsworth_sd1.4', 'rlct4.reV.cevansA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-x.cevans_sd1.4', 'rlct4.reV.cevansA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_esd-x.cevans_sd1.4', 'rlct4.reV.cevansA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_esd-x.cevans_sd1.4']


exp_names = ['rlct4.reV.nicoparkerA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-u.octavia_sd1.4', 'rlct4.reV.nicoparkerA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_esd-u.edsheeran_sd1.4', 'rlct4.reV.nicoparkerA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_esd-u.edsheeran_sd1.4', 'rlct4.reV.nicoparkerA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-u.adriver_sd1.4', 'rlct4.reV.nicoparkerA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_esd-u.idris_sd1.4', 'rlct4.reV.nicoparkerA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_esd-u.ajolie_sd1.4', 'rlct4.reV.nicoparkerA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-u.rihanna_sd1.4', 'rlct4.reV.nicoparkerA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_esd-u.obama_sd1.4', 'rlct4.reV.nicoparkerA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_esd-u.idris_sd1.4', 'rlct4.reV.nicoparkerA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-u.morganf_sd1.4', 'rlct4.reV.nicoparkerA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_esd-u.ahathaway_sd1.4', 'rlct4.reV.nicoparkerA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_esd-u.obama_sd1.4', 'rlct4.reV.nicoparkerA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-u.oprah_sd1.4', 'rlct4.reV.nicoparkerA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_esd-u.obama_sd1.4', 'rlct4.reV.nicoparkerA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_esd-u.morganf_sd1.4', 'rlct4.reV.nicoparkerA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-u.oprah_sd1.4', 'rlct4.reV.nicoparkerA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_esd-u.chemsworth_sd1.4', 'rlct4.reV.nicoparkerA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_esd-u.obama_sd1.4', 'rlct4.reV.nicoparkerA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-u.drake_sd1.4', 'rlct4.reV.nicoparkerA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_esd-u.chemsworth_sd1.4', 'rlct4.reV.nicoparkerA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_esd-u.idris_sd1.4', 'rlct4.reV.nicoparkerA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-u.mrobbie_sd1.4', 'rlct4.reV.nicoparkerA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_esd-u.edsheeran_sd1.4', 'rlct4.reV.nicoparkerA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_esd-u.agarfield_sd1.4', 'rlct4.reV.nicoparkerA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-u.edsheeran_sd1.4', 'rlct4.reV.nicoparkerA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_esd-u.morganf_sd1.4', 'rlct4.reV.nicoparkerA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_esd-u.drake_sd1.4', 'rlct4.reV.nicoparkerA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-u.agarfield_sd1.4', 'rlct4.reV.nicoparkerA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_esd-u.ahathaway_sd1.4', 'rlct4.reV.nicoparkerA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_esd-u.mrobbie_sd1.4', 'rlct4.reV.nicoparkerA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-u.adriver_sd1.4', 'rlct4.reV.nicoparkerA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_esd-u.rihanna_sd1.4', 'rlct4.reV.nicoparkerA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_esd-u.edsheeran_sd1.4', 'rlct4.reV.nicoparkerA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-u.rihanna_sd1.4', 'rlct4.reV.nicoparkerA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_esd-u.mcarey_sd1.4', 'rlct4.reV.nicoparkerA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_esd-u.aadam_sd1.4']

exp_names = ['rlct4.reV.asanteA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-u.edsheeran_sd1.4', 'rlct4.reV.asanteA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_esd-u.edsheeran_sd1.4', 'rlct4.reV.asanteA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_esd-u.aadam_sd1.4', 'rlct4.reV.reeseA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-u.rihanna_sd1.4', 'rlct4.reV.reeseA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_esd-u.mrobbie_sd1.4', 'rlct4.reV.reeseA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_esd-u.morganf_sd1.4', 'rlct4.reV.nivolaA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-u.drake_sd1.4', 'rlct4.reV.nivolaA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_esd-u.octavia_sd1.4', 'rlct4.reV.nivolaA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_esd-u.aadam_sd1.4', 'rlct4.reV.earleA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-u.rihanna_sd1.4', 'rlct4.reV.earleA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_esd-u.obama_sd1.4', 'rlct4.reV.earleA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_esd-u.rihanna_sd1.4', 'rlct4.reV.leowoodalA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-u.octavia_sd1.4', 'rlct4.reV.leowoodalA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_esd-u.obama_sd1.4', 'rlct4.reV.leowoodalA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_esd-u.obama_sd1.4', 'rlct4.reV.starkeyA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-u.octavia_sd1.4', 'rlct4.reV.starkeyA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_esd-u.mrobbie_sd1.4', 'rlct4.reV.starkeyA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_esd-u.chemsworth_sd1.4', 'rlct4.reV.apierreA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-u.obama_sd1.4', 'rlct4.reV.apierreA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_esd-u.obama_sd1.4', 'rlct4.reV.apierreA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_esd-u.chemsworth_sd1.4', 'rlct4.reV.skyhblackA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-u.rihanna_sd1.4', 'rlct4.reV.skyhblackA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_esd-u.ahathaway_sd1.4', 'rlct4.reV.skyhblackA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_esd-u.mrobbie_sd1.4', 'rlct4.reV.sophiewildeA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-u.edsheeran_sd1.4', 'rlct4.reV.sophiewildeA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_esd-u.chemsworth_sd1.4', 'rlct4.reV.sophiewildeA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_esd-u.ahathaway_sd1.4', 'rlct4.reV.edebiriA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-u.edsheeran_sd1.4', 'rlct4.reV.edebiriA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_esd-u.chemsworth_sd1.4', 'rlct4.reV.edebiriA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_esd-u.mcarey_sd1.4', 'rlct4.reV.mmadisonA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-u.obama_sd1.4', 'rlct4.reV.mmadisonA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_esd-u.ahathaway_sd1.4', 'rlct4.reV.mmadisonA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_esd-u.octavia_sd1.4', 'rlct4.reV.nicoparkerA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-u.mrobbie_sd1.4', 'rlct4.reV.nicoparkerA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_esd-u.chemsworth_sd1.4', 'rlct4.reV.nicoparkerA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_esd-u.aadam_sd1.4']



v = 0
exp_names = exp_names[v*5:v*5+5]





# exp_names = [exp_names[3]]


apply_use_ti = True
apply_negative_prompt = True

# exp_names = [exp_names[0]]
count = 0

cfg_scales = [  4.5, 6.0]
cfg_scales = [7.5]

for exp_name in exp_names:
    # manual_prompt = 'A photo of a toy'# 'A photo of a toy'
    # manual_prompt = 'A photo of a hippo'
    
    
    is_relearn = 'uul' in exp_name or 'rl' in exp_name
    if is_relearn:
        base_exp_name = '_'.join(exp_name.split('_')[2:])
        relearn_exp_name = exp_name
        unlearn_exp_name = '_'.join(exp_name.split('_')[1:])
        exp_name = base_exp_name

    
    manual_prompt = ''
    use_general_concept = False
    # cfg_scales = np.arange(2.0,4.5, 0.5).tolist()
    # cfg_scales = np.arange(3.0,3.5, 0.5).tolist()

    # steps = [50,100,150,200]
    # for step in steps:
    steps =range(0, 3000+1, 200)
    steps =range(0, 1000+1, 100)
    steps =range(500, 500+1, 100)
    steps =range(0, 1000+1, 100)
    # steps =range(50, 1000+1, 100)
    # steps =range(50, 1000+1, 100)
    # steps =range(50, 1000+1, 100)
    # steps =range(25, 1000+1, 100)
    # steps = [300]
    # steps =range(700, 1000+1, 100)
    # steps =range(800, 1000+1, 100)
    # steps =range(300, 500+1, 100)
    for step in steps:
    # for step in [1200,1800]:

        # for cfg in cfg_scales:
        is_original_pretrained = exp_name == 'CompVis/stable-diffusion-v1-4'
        is_unlearn = 'ul' in exp_name and not is_relearn
        if 'moodeng' in exp_name: concept = 'moodeng'
        if 'crybaby' in exp_name: concept = 'crybaby'
        if 'avp' in exp_name: concept = 'avp'
        if 'chiquita' in exp_name: concept = 'chiquita'
        if 'sceleb5g0' in exp_name: concept = 'sceleb5g0'
        
        pretrained_path = 'CompVis/stable-diffusion-v1-4'
        if is_relearn:
            # pretrained_path = f"data_root/logs/{unlearn_exp_name}/LoRA_fusion_model"
            
            pretrained_path = 'CompVis/stable-diffusion-v1-4'
            
            for c in seen_concepts:
                # print(c)
                if c in data_info and c in unlearn_exp_name:
                    concept = c
                    # print("Found concept:", concept)
                    break
            # print(c,concept)
            concept_name = format_name(data_info[concept]['full_name']).replace(' ', '_')
            if 'esd-x' in unlearn_exp_name:
               train_method = 'esdx'
            elif 'esd-u' in unlearn_exp_name:
                train_method = 'esdu' 
            elif 'esd-all' in unlearn_exp_name:
                train_method = 'esdall'
            else:
                print(f"Error: Unrecognized training method in {exp_name}")
                assert False
            pretrained_unet_name = f"esd-{concept_name}-from-{concept_name}-{train_method}"
            unet_weight_path = f"data_root/logs/esd/sd1.4/{pretrained_unet_name}.safetensors"
             
            
            
            # erase_name = concept
            # if 'VPr' in exp_name: erase_name += 'VPr'
            # pretrained_path = f"data_root/logs/erase_l1.{erase_name}.object_lr2.5e-4/LoRA_fusion_model"
        if is_unlearn: 
            pretrained_path = f"data_root/logs/{exp_name}/LoRA_fusion_model"

        use_ti = 'ti' in relearn_exp_name or '-V' in exp_name 
        
        if is_relearn and not 'reV' in relearn_exp_name:
            # relearn is not re-initializing the token (by default)
            initializer_token = ''
        elif use_ti:
            initializer_token = concept2initializer[concept]

        if manual_prompt:
            prompt = manual_prompt
        elif use_general_concept:
            prompt = concept2generalprompt[concept]
        
        if use_ti:
            if 'sceleb' in concept:
                prompt = 'A photo of a v1,A photo of a v2,A photo of a v3,A photo of a v4,A photo of a v5'
                placeholder_token = 'v1,v2,v3,v4,v5'
            else:
                prompt = 'a photo of v1' 
                placeholder_token = 'v1'
        else:
            prompt = concept2prompt[concept]
            
        ## hacky .. should change this later
        if is_relearn:
            exp_name = relearn_exp_name
        if is_unlearn or 'erase' in exp_name or exp_name == 'original_pretrained': 
            load_lora_weight_path = ''
            gen_image_path = f"data_root/generated/model/{exp_name}"
        else:
            load_lora_weight_path =f"data_root/logs/{exp_name}/checkpoint-{step}"
            gen_image_path = 'auto'
            
        # if 'l0' in exp_name :
        #     load_lora_weight_path = ''
        
        script = f"""
        accelerate launch train_dreambooth_lora.py \\
            --pretrained_model_name_or_path='{pretrained_path}'  \\
            --load_unet_weight_path="{unet_weight_path}" \\
            --load_lora_weight_path="{load_lora_weight_path}" \\
            --instance_data_dir="data_root/data/real_data/dummy" \\
            --gen_image_path="{gen_image_path}" \\
            --output_dir="data_root/logs/gen" \\
            --validation_prompt="{prompt}" --instance_prompt="{prompt}" \\
            --lora_rank 1 --target_lora_modules to_k to_v --target_lora_layers cross \\
            --run_note 'gen img' --wait_weight \\
            --num_validation_images 50 \\"""
            
                
        if use_ti and not is_unlearn:
            script += f"""
            --load_token_embedding_path="data_root/logs/{exp_name}/checkpoint-{step}" \\
            --placeholder_token="{placeholder_token}" --initializer_token='{initializer_token}' \\"""

        if apply_negative_prompt:
            script += f"""
            --negative_prompt "longbody, lowres, bad anatomy, bad hands, missing fingers, extra digit, fewer digits, cropped, worst quality, low quality." \\"""
        

        script += f"""
            --cfg_scale {','.join(f'{x:.2f}' for x in cfg_scales)}"""

        print(f"echo 'count:{count} - {exp_name} {step} /'")
        print(script) 
        
        count += 1
print(f"Total scripts generated: {count}")
        

echo 'count:0 - rlct4.reV.asanteA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-u.edsheeran_sd1.4 0 /'

        accelerate launch train_dreambooth_lora.py \
            --pretrained_model_name_or_path='CompVis/stable-diffusion-v1-4'  \
            --load_unet_weight_path="data_root/logs/esd/sd1.4/esd-Ed_Sheeran-from-Ed_Sheeran-esdu.safetensors" \
            --load_lora_weight_path="data_root/logs/rlct4.reV.asanteA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-u.edsheeran_sd1.4/checkpoint-0" \
            --instance_data_dir="data_root/data/real_data/dummy" \
            --gen_image_path="auto" \
            --output_dir="data_root/logs/gen" \
            --validation_prompt="a photo of v1" --instance_prompt="a photo of v1" \
            --lora_rank 1 --target_lora_modules to_k to_v --target_lora_layers cross \
            --run_note 'gen img' --wait_weight \
            --num_validation_images 50 \
            --load_token_embedding_path="data_root/logs/rlct4.reV.asanteA5V0.ln.lr1e-4.ti5e-4.pr1.0

In [ ]:

# Swap learning - ESD
# RANDOM unlearn seen - relearn (unseen)


ul_exp_names = ['esd-Barrack_Obama-from-Barrack_Obama-esdx', 'esd-Barrack_Obama-from-Barrack_Obama-esdu', 'esd-Barrack_Obama-from-Barrack_Obama-esdall', 'esd-Rihanna-from-Rihanna-esdx', 'esd-Rihanna-from-Rihanna-esdu', 'esd-Rihanna-from-Rihanna-esdall', 'esd-Ed_Sheeran-from-Ed_Sheeran-esdx', 'esd-Ed_Sheeran-from-Ed_Sheeran-esdu', 'esd-Ed_Sheeran-from-Ed_Sheeran-esdall', 'esd-Margot_Robbie-from-Margot_Robbie-esdx', 'esd-Margot_Robbie-from-Margot_Robbie-esdu', 'esd-Margot_Robbie-from-Margot_Robbie-esdall', 'esd-Chris_Hemsworth-from-Chris_Hemsworth-esdx', 'esd-Chris_Hemsworth-from-Chris_Hemsworth-esdu', 'esd-Chris_Hemsworth-from-Chris_Hemsworth-esdall', 'esd-Chris_Evans-from-Chris_Evans-esdx', 'esd-Chris_Evans-from-Chris_Evans-esdu', 'esd-Chris_Evans-from-Chris_Evans-esdall', 'esd-Adam_Driver-from-Adam_Driver-esdx', 'esd-Adam_Driver-from-Adam_Driver-esdu', 'esd-Adam_Driver-from-Adam_Driver-esdall', 'esd-Andrew_Garfield-from-Andrew_Garfield-esdx', 'esd-Andrew_Garfield-from-Andrew_Garfield-esdu', 'esd-Andrew_Garfield-from-Andrew_Garfield-esdall', 'esd-Anne_Adam-from-Anne_Adam-esdx', 'esd-Anne_Adam-from-Anne_Adam-esdu', 'esd-Anne_Adam-from-Anne_Adam-esdall', 'esd-Anne_Hathaway-from-Anne_Hathaway-esdx', 'esd-Anne_Hathaway-from-Anne_Hathaway-esdu', 'esd-Anne_Hathaway-from-Anne_Hathaway-esdall', 'esd-Angelina_Jolie-from-Angelina_Jolie-esdx', 'esd-Angelina_Jolie-from-Angelina_Jolie-esdu', 'esd-Angelina_Jolie-from-Angelina_Jolie-esdall', 'esd-Amber_Heard-from-Amber_Heard-esdx', 'esd-Amber_Heard-from-Amber_Heard-esdu', 'esd-Amber_Heard-from-Amber_Heard-esdall', 'esd-Mariah_Carey-from-Mariah_Carey-esdx', 'esd-Mariah_Carey-from-Mariah_Carey-esdu', 'esd-Mariah_Carey-from-Mariah_Carey-esdall', 'esd-Octavia_Spencer-from-Octavia_Spencer-esdx', 'esd-Octavia_Spencer-from-Octavia_Spencer-esdu', 'esd-Octavia_Spencer-from-Octavia_Spencer-esdall', 'esd-Oprah_Winfrey-from-Oprah_Winfrey-esdx', 'esd-Oprah_Winfrey-from-Oprah_Winfrey-esdu', 'esd-Oprah_Winfrey-from-Oprah_Winfrey-esdall', 'esd-Morgan_Freeman-from-Morgan_Freeman-esdx', 'esd-Morgan_Freeman-from-Morgan_Freeman-esdu', 'esd-Morgan_Freeman-from-Morgan_Freeman-esdall', 'esd-Drake-from-Drake-esdx', 'esd-Drake-from-Drake-esdu', 'esd-Drake-from-Drake-esdall', 'esd-Idris_Elba-from-Idris_Elba-esdx', 'esd-Idris_Elba-from-Idris_Elba-esdu', 'esd-Idris_Elba-from-Idris_Elba-esdall']

ul_exp_names_ = []
for ul_exp_name in ul_exp_names:
    for c in seen_concepts:
        if c in data_info and format_name(data_info[c]['full_name']).replace(' ', '_') in ul_exp_name:
            ul_exp_names_.append(ul_exp_name)
            break
ul_exp_names = ul_exp_names_ 
                
scripts = []

ul_exp_names = [ul_exp_name for ul_exp_name in ul_exp_names if 'esdx' in ul_exp_name] # filter out the esd-u ones
# ul_exp_names = ['esd-Barrack_Obama-from-Barrack_Obama-esdx', 'esd-Barrack_Obama-from-Barrack_Obama-esdu', 'esd-Barrack_Obama-from-Barrack_Obama-esdall']

pretrained = 'sd1.4'


#seed = 0

# seeds = [0,1,2] # along

# hacked 
# seeds = [999]*len(ul_exp_names)
seeds = [0,1,2]
# seeds = [3,4]
# seeds = [0,1,2,3,4]

use_te = True
batch_size = 1
gradient_accumulation_step = 4
# re
# lr_lora_grid = ["1e-4", "5e-5", "1e-5"]
lr_lora_grid = ["1e-4"]
# lr_lora_grid = [ "1e-5"]
lr_ti_grid = ["5e-4"]   # only used if use_ti
# lr_ti_grid   = ["5e-3","5e-2"]   # only used if use_ti
lr_te_grid = ["1e-5"] 
lora_ranks = [4] # [1,2,4,8,16,32,64]
# # Create all combinations of lr_lora, lr_ti, and seed
combos = list(itertools.product(lr_lora_grid,
                                lr_ti_grid,
                                lr_te_grid if use_te else [None],
                                lora_ranks))

use_pr = True

use_te = True
apply_use_ti = True

lr_scheduler = 'linear' # 'linear' # 'cosine' # 'cosine_with_restarts'
apply_negative_prompt = True

use_manual_params = True
reV = True
is_relearn = True 

# fix here #
manual_params = {
    # 'data_setting': 'facefew',
    'data_setting': 'align',
    
    # 'data_setting': 'small',s
    # 'lora_rank' :  16
}

data_setting = 'align' 

final_exp_names = []
# for ul_exp_name in ul_exp_names:

rng = np.random.RandomState(123)

for learn_concept in unseen_concepts:
    
    for seed in seeds:
        
        ul_exp_name = rng.choice(ul_exp_names)


        
        concept = learn_concept
        # extract unlearned concept
        for c in seen_concepts:
            if c in data_info and format_name(data_info[c]['full_name']).replace(' ', '_') in ul_exp_name:
                if not learn_concept:
                    concept = c
                unlearned_concept = c
                break

        if 'esdx' in ul_exp_name:
            train_method = 'esd-x'
        elif 'esdu' in ul_exp_name:
            train_method = 'esd-u'
        elif 'esdall' in ul_exp_name:
            train_method = 'esd-all'

        pretrained_unet_name = ul_exp_name 
        ul_exp_name = f'{train_method}.{unlearned_concept}_sd1.4'
        
        
        base_exp_name = '_'.join(ul_exp_name.split('_')[1:])
        exp_name = base_exp_name

        

        

        # print(f"Base Experiment Name: {base_exp_name}")
        
        #fix edit here : they are using this to reconstruct the base_exp as well
        lr_lora, lr_ti = extract_lrs(base_exp_name)
        lora_rank = extract_learning_lora_rank(base_exp_name)
        
        # todo: better use 're'
        for re_lr_lora, re_lr_ti, re_lr_te, re_lora_rank in combos:
            
            # print(f'lora_rank: {lora_rank}, lr: {lr_lora}, ti_lr: {lr_ti}')
            # pretrained_path = f"data_root/logs/{ul_exp_name}/LoRA_fusion_model"
            pretrained_path = 'CompVis/stable-diffusion-v1-4'
            unet_weight_path = f"data_root/logs/esd/sd1.4/{pretrained_unet_name}.safetensors"
            # print(f"Concept: {concept}"

            use_ti = 'ti' in exp_name or '-V' in exp_name or apply_use_ti
            # use_pr = 'pr' in exp_name

            if data_setting == 'facefew':
                dataset_name = f'{concept}5F0r{seed}'
                
            elif data_setting == 'align':
                dataset_name = f'{concept}A5V0'
            elif data_setting == 'fewshot':
                
                if 'sceleb' in concept:
                    dataset_name = f'{concept}U3'
                elif concept == 'avp':
                    dataset_name = 'avpS3'
                else:
                    dataset_name = f'{concept}U3'
            elif data_setting == 'small':
                
                if 'sceleb' in concept:
                    dataset_name = f'{concept}N10'
                else:
                    dataset_name = f'{concept}10'
            else:
                
                if 'sceleb' in concept:
                    dataset_name = f'{concept}N50'
                elif concept == 'avp':
                    dataset_name = 'avp20'
                else:
                    dataset_name = f'{concept}50'
                    
            if reV:
                initializer_token = concept2initializer[concept]
            else: 
                initializer_token = ''
                
            if use_ti:
                if 'sceleb' in concept:
                    prompt = 'A photo of a v1,A photo of a v2,A photo of a v3,A photo of a v4,A photo of a v5'
                    placeholder_token = 'v1,v2,v3,v4,v5'
                else:
                    prompt = 'a photo of v1' 
                    placeholder_token = 'v1'
            else:
                prompt = concept2prompt[concept]

            name_tag = ''
            if is_relearn: name_tag += 'uul'
            name_tag = f'{name_tag} {dataset_name}'
            name_tag += f' l{lora_rank}'
            if use_ti: 
                # name_tag += f' ti.{lr_ti}'
                name_tag += f' ti'

            data_root = dataset_name2data_root[dataset_name]
            
            if use_ti:
                dataset_name_for_exp = dataset_name + "-V"
                # if use_ni:
                #     dataset_name_for_exp += ".ni"
            else: dataset_name_for_exp = dataset_name
            
            
            # prior preservation folder
            prior_folder = 'original_realistic_vision'
            if pretrained == 'sd1.5':
                prior_folder = 'original_pretrained_sd1.5'
            if pretrained == 'sd1.4':
                prior_folder = 'original_pretrained_sd1.4'     
            if pretrained == 'rv':
                prior_folder = 'original_realistic_vision'
            elif pretrained == 'ch':
                prior_folder = 'original_chilloutmix'

            
            # renaming to check
            # re_exp_name = f'c.l{lora_rank}.kv_{dataset_name_for_exp}'
            # if use_pr:
            #     re_exp_name += f'_pr0.50'
            # re_exp_name += '_lr'
            # if lora_rank >0: re_exp_name += f"{str(lr_lora)}"
            # if use_ti:
            #     re_exp_name += f'.ti{str(lr_ti)}'
            # re_exp_name += '_f0.5_b1g4'
            
            # print(re_exp_name)
            # assert re_exp_name in base_exp_name, f"Expected {re_exp_name} in {base_exp_name}"

            # if manual_lora is not None and manual_data != lora_rank:
            
            if use_manual_params:
                lora_rank = re_lora_rank
                eff_data_setting = manual_params['data_setting']

                lr_lora, lr_ti = re_lr_lora, re_lr_ti
                
                if eff_data_setting == 'facefew':
                    dataset_name = f'{concept}5F0r{seed}'
                    
                elif eff_data_setting == 'align':
                    dataset_name = f'{concept}A5V0'
                
                elif eff_data_setting == 'fewshot':
                    
                    if 'sceleb' in concept:
                        dataset_name = f'{concept}U3'
                    elif concept == 'avp':
                        dataset_name = 'avpS3'
                    else:
                        dataset_name = f'{concept}U3'
                elif eff_data_setting == 'small':
                    
                    if 'sceleb' in concept:
                        dataset_name = f'{concept}N10'
                    else:
                        dataset_name = f'{concept}10'
                else:
                    
                    if 'sceleb' in concept:
                        dataset_name = f'{concept}N50'
                    elif concept == 'avp':
                        dataset_name = 'avp20'
                    else:
                        dataset_name = f'{concept}50'            
                        
                data_root = dataset_name2data_root[dataset_name]
            
            
            
            
            if reV:
                if use_te:
                    relearn_exp_name = f"rlct{lora_rank}.reV.{dataset_name}"
                else:
                    relearn_exp_name = f"rlc{lora_rank}.reV.{dataset_name}"
            else:
                if use_te:
                    relearn_exp_name = f"rlct{lora_rank}.{dataset_name}"
                else:
                    relearn_exp_name = f"rlc{lora_rank}.{dataset_name}"
            
            
            if lr_scheduler == 'linear':
                relearn_exp_name += f".ln"
            relearn_exp_name += f".lr{re_lr_lora}.ti{re_lr_ti}"
                
                
            if use_pr:
                relearn_exp_name += f".pr1.00"
                if apply_negative_prompt:
                    relearn_exp_name += ".neg"
            
            relearn_exp_name += f".b{batch_size}g{gradient_accumulation_step}"
            
            if seed != 0:
                relearn_exp_name += f".r{seed}"

            final_exp_name = f"{relearn_exp_name}_{ul_exp_name}"
            
            
            script = f"""
            accelerate launch train_dreambooth_lora.py \\
            --pretrained_model_name_or_path="{pretrained_path}"  \\
            --load_unet_weight_path="{unet_weight_path}" \\
            --instance_data_dir="{data_root}" \\
            --output_dir="data_root/logs/{final_exp_name}" \\
            --validation_prompt="{prompt}" --instance_prompt="{prompt}" \\
            --train_batch_size={batch_size} --gradient_accumulation_steps={gradient_accumulation_step} \\
            --lora_rank {lora_rank} --target_lora_modules to_k to_v --target_lora_layers cross \\
            --max_train_steps=1000  --validation_steps=50  --checkpointing_steps=50  --lr_scheduler "{lr_scheduler}"  --seed {seed} \\
            --run_note '{name_tag}' \\"""
                
            script += f"""
            --cfg_scale 6.0 \\"""
        
        
            if apply_negative_prompt:
                script += f"""
            --negative_prompt "longbody, lowres, bad anatomy, bad hands, missing fingers, extra digit, fewer digits, cropped, worst quality, low quality." \\"""
                
                
            if use_pr:
                if pretrained == 'sd1.4':
                    cfg_pr = 7.5
                elif pretrained == 'ch':
                    cfg_pr = 6.0
                script += f"""
            --with_prior_preservation --prior_loss_weight=1.0 --num_class_images 200 \\
            --class_prompt="a photo of a person" --class_data_dir="data_root/generated/model/{prior_folder}/a photo of a person_neg/{cfg_pr:.2f}" \\"""
                
            # Conditional learning rate + TI options
            if use_ti:
                
                if lora_rank <= 0:
                    script += f"""
            --learning_rate_ti {lr_ti} \\
            --placeholder_token="{placeholder_token}" --initializer_token='{initializer_token}'"""
                else:
                    if use_te:
                        script += f"""
            --learning_rate_lora {lr_lora} --learning_rate_ti {lr_ti} \\
            --train_text_encoder --learning_rate_lora_text_encoder {re_lr_te} \\
            --placeholder_token="{placeholder_token}" --initializer_token='{initializer_token}'"""
                    else:
                        script += f"""
            --learning_rate_lora {lr_lora} --learning_rate_ti {lr_ti} \\
            --placeholder_token="{placeholder_token}" --initializer_token='{initializer_token}'"""
            else:
                script += f"""
            --learning_rate {lr_lora}"""

            print(f"echo 'count:{len(final_exp_names)} '")
            print(script)
            final_exp_names += [final_exp_name]
            
            scripts += [script]
        
print(len(final_exp_names))
print(final_exp_names)


# esd-x
['rlct4.reV.asanteA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-x.edsheeran_sd1.4', 'rlct4.reV.asanteA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_esd-x.edsheeran_sd1.4', 'rlct4.reV.asanteA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_esd-x.aadam_sd1.4', 'rlct4.reV.reeseA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-x.rihanna_sd1.4', 'rlct4.reV.reeseA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_esd-x.mrobbie_sd1.4', 'rlct4.reV.reeseA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_esd-x.morganf_sd1.4', 'rlct4.reV.nivolaA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-x.drake_sd1.4', 'rlct4.reV.nivolaA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_esd-x.octavia_sd1.4', 'rlct4.reV.nivolaA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_esd-x.aadam_sd1.4', 'rlct4.reV.earleA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-x.rihanna_sd1.4', 'rlct4.reV.earleA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_esd-x.obama_sd1.4', 'rlct4.reV.earleA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_esd-x.rihanna_sd1.4', 'rlct4.reV.leowoodalA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-x.octavia_sd1.4', 'rlct4.reV.leowoodalA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_esd-x.obama_sd1.4', 'rlct4.reV.leowoodalA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_esd-x.obama_sd1.4', 'rlct4.reV.starkeyA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-x.octavia_sd1.4', 'rlct4.reV.starkeyA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_esd-x.mrobbie_sd1.4', 'rlct4.reV.starkeyA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_esd-x.chemsworth_sd1.4', 'rlct4.reV.apierreA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-x.obama_sd1.4', 'rlct4.reV.apierreA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_esd-x.obama_sd1.4', 'rlct4.reV.apierreA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_esd-x.chemsworth_sd1.4', 'rlct4.reV.skyhblackA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-x.rihanna_sd1.4', 'rlct4.reV.skyhblackA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_esd-x.ahathaway_sd1.4', 'rlct4.reV.skyhblackA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_esd-x.mrobbie_sd1.4', 'rlct4.reV.sophiewildeA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-x.edsheeran_sd1.4', 'rlct4.reV.sophiewildeA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_esd-x.chemsworth_sd1.4', 'rlct4.reV.sophiewildeA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_esd-x.ahathaway_sd1.4', 'rlct4.reV.edebiriA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-x.edsheeran_sd1.4', 'rlct4.reV.edebiriA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_esd-x.chemsworth_sd1.4', 'rlct4.reV.edebiriA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_esd-x.mcarey_sd1.4', 'rlct4.reV.mmadisonA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-x.obama_sd1.4', 'rlct4.reV.mmadisonA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_esd-x.ahathaway_sd1.4', 'rlct4.reV.mmadisonA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_esd-x.octavia_sd1.4', 'rlct4.reV.nicoparkerA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-x.mrobbie_sd1.4', 'rlct4.reV.nicoparkerA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_esd-x.chemsworth_sd1.4', 'rlct4.reV.nicoparkerA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_esd-x.aadam_sd1.4']


# esd-u
# ['rlct4.reV.asanteA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-u.edsheeran_sd1.4', 'rlct4.reV.asanteA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_esd-u.edsheeran_sd1.4', 'rlct4.reV.asanteA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_esd-u.aadam_sd1.4', 'rlct4.reV.reeseA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-u.rihanna_sd1.4', 'rlct4.reV.reeseA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_esd-u.mrobbie_sd1.4', 'rlct4.reV.reeseA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_esd-u.morganf_sd1.4', 'rlct4.reV.nivolaA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-u.drake_sd1.4', 'rlct4.reV.nivolaA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_esd-u.octavia_sd1.4', 'rlct4.reV.nivolaA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_esd-u.aadam_sd1.4', 'rlct4.reV.earleA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-u.rihanna_sd1.4', 'rlct4.reV.earleA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_esd-u.obama_sd1.4', 'rlct4.reV.earleA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_esd-u.rihanna_sd1.4', 'rlct4.reV.leowoodalA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-u.octavia_sd1.4', 'rlct4.reV.leowoodalA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_esd-u.obama_sd1.4', 'rlct4.reV.leowoodalA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_esd-u.obama_sd1.4', 'rlct4.reV.starkeyA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-u.octavia_sd1.4', 'rlct4.reV.starkeyA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_esd-u.mrobbie_sd1.4', 'rlct4.reV.starkeyA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_esd-u.chemsworth_sd1.4', 'rlct4.reV.apierreA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-u.obama_sd1.4', 'rlct4.reV.apierreA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_esd-u.obama_sd1.4', 'rlct4.reV.apierreA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_esd-u.chemsworth_sd1.4', 'rlct4.reV.skyhblackA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-u.rihanna_sd1.4', 'rlct4.reV.skyhblackA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_esd-u.ahathaway_sd1.4', 'rlct4.reV.skyhblackA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_esd-u.mrobbie_sd1.4', 'rlct4.reV.sophiewildeA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-u.edsheeran_sd1.4', 'rlct4.reV.sophiewildeA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_esd-u.chemsworth_sd1.4', 'rlct4.reV.sophiewildeA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_esd-u.ahathaway_sd1.4', 'rlct4.reV.edebiriA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-u.edsheeran_sd1.4', 'rlct4.reV.edebiriA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_esd-u.chemsworth_sd1.4', 'rlct4.reV.edebiriA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_esd-u.mcarey_sd1.4', 'rlct4.reV.mmadisonA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-u.obama_sd1.4', 'rlct4.reV.mmadisonA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_esd-u.ahathaway_sd1.4', 'rlct4.reV.mmadisonA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_esd-u.octavia_sd1.4', 'rlct4.reV.nicoparkerA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-u.mrobbie_sd1.4', 'rlct4.reV.nicoparkerA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_esd-u.chemsworth_sd1.4', 'rlct4.reV.nicoparkerA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_esd-u.aadam_sd1.4']


echo 'count:0 '

            accelerate launch train_dreambooth_lora.py \
            --pretrained_model_name_or_path="CompVis/stable-diffusion-v1-4"  \
            --load_unet_weight_path="data_root/logs/esd/sd1.4/esd-Ed_Sheeran-from-Ed_Sheeran-esdx.safetensors" \
            --instance_data_dir="data_root/data/real_data/asante/aligned/asante-5-v0" \
            --output_dir="data_root/logs/rlct4.reV.asanteA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-x.edsheeran_sd1.4" \
            --validation_prompt="a photo of v1" --instance_prompt="a photo of v1" \
            --train_batch_size=1 --gradient_accumulation_steps=4 \
            --lora_rank 4 --target_lora_modules to_k to_v --target_lora_layers cross \
            --max_train_steps=1000  --validation_steps=50  --checkpointing_steps=50  --lr_scheduler "linear"  --seed 0 \
            --run_note 'uul asanteA5V0 lNone ti' \
            --cfg_scale 6.0 \
            --negative_prompt "longbody, lowres, bad anatomy, bad hands, missing finge

In [ ]:
v = 3
# for i, script in enumerate(scripts[v*5:v*5+5]):
n_device = 4
len_ = int(len(scripts)/ n_device)

print(f"Total scripts: {len(scripts)}: {len_} per device")
for i in range(v*len_, v*len_+len_):
    print(f"""echo 'count: {i}'""")
    
    script = scripts[i]
    print(script)

Total scripts: 36: 9 per device
echo 'count: 27'

            accelerate launch train_dreambooth_lora.py \
            --pretrained_model_name_or_path="CompVis/stable-diffusion-v1-4"  \
            --load_unet_weight_path="data_root/logs/esd/sd1.4/esd-Ed_Sheeran-from-Ed_Sheeran-esdx.safetensors" \
            --instance_data_dir="data_root/data/real_data/edebiri/aligned/edebiri-5-v0" \
            --output_dir="data_root/logs/rlct4.reV.edebiriA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-x.edsheeran_sd1.4" \
            --validation_prompt="a photo of v1" --instance_prompt="a photo of v1" \
            --train_batch_size=1 --gradient_accumulation_steps=4 \
            --lora_rank 4 --target_lora_modules to_k to_v --target_lora_layers cross \
            --max_train_steps=1000  --validation_steps=50  --checkpointing_steps=50  --lr_scheduler "linear"  --seed 0 \
            --run_note 'uul edebiriA5V0 lNone ti' \
            --cfg_scale 6.0 \
            --negative_prompt "longbody, lowres, 

In [ ]:
# ESD-Generation

# exp_names = ['rl4.chiquita50_ul1.prg1e-4d8e-5.lr1e-4.n8.G.chiquita.person.s50_c.l4.kv_chiquita50-V_pr0.50_lr5e-4.ti5e-2_f0.5_b1g4.s3000']
#    ['uul1.lr1e-4.n8.G.chiquita.obj.s8_c.l4.kv_chiquita50-V_pr0.50_lr5e-4.ti5e-2_f0.5_b1g4.s3000', 
    # 'uul1.lr1e-4.n8.G.chiquita.obj.s0_c.l4.kv_chiquita50-V_pr0.50_lr5e-4.ti5e-2_f0.5_b1g4.s3000']
# ul_exp_names = , 'ul1.lr1e-4.n8.G.chiquita.obj.s0_c.l4.kv_chiquita50-V_pr0.50_lr5e-4.ti1e-3_f0.5_b1g4.s3000']
# ul_exp_names = [, ]
# uul1.prg8e+7d1e-2.lr1e-4.n8.G.chiquita.obj.s0_c.l4.kv_chiquita50-V_pr0.50_lr5e-4.ti1e-2_f0.5_b1g4.s3000
exp_names = ['rl16.reV.sceleb5g0N10.lr5e-5.ti5e-4.r2_ul1.prg1e-4d5e-4.lr1e-4.n8.G.sceleb5g0.person.s50.r2_c.l16.kv_sceleb5g0N50-V_pr0.50_lr5e-5.ti5e-4_f0.5_b4g4.s10000']


exp_names = ['rlct4.reV.honer10.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_ul1.prg1e-4d8e+3.lr1e-4.n8.G.honer.person.s50_ch.c.l16.kv_honer50-V.r_pr1.00.neg_lr5e-4.ti5e-4_b1g4.s2000']

exp_names = ['rlct4.reV.rihanna5F0r0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_ul1.prg1e-4d8e+3.lr1e-4.n8.G.rihanna.person.s50_rv', 'rlct4.reV.rihanna5F0r1.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_ul1.prg1e-4d8e+3.lr1e-4.n8.G.rihanna.person.s50.r1_rv', 'rlct4.reV.rihanna5F0r2.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_ul1.prg1e-4d8e+3.lr1e-4.n8.G.rihanna.person.s50.r2_rv', 'rlct4.reV.rihanna5F0r3.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r3_ul1.prg1e-4d8e+3.lr1e-4.n8.G.rihanna.person.s50.r3_rv', 'rlct4.reV.rihanna5F0r4.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r4_ul1.prg1e-4d8e+3.lr1e-4.n8.G.rihanna.person.s50.r4_rv']


exp_names =  ['rlct4.reV.obamaA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-x.obama_sd1.4', 'rlct4.reV.obamaA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_esd-x.obama_sd1.4', 'rlct4.reV.obamaA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_esd-x.obama_sd1.4', 'rlct4.reV.rihannaA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-x.rihanna_sd1.4', 'rlct4.reV.rihannaA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_esd-x.rihanna_sd1.4', 'rlct4.reV.rihannaA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_esd-x.rihanna_sd1.4', 'rlct4.reV.edsheeranA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-x.edsheeran_sd1.4', 'rlct4.reV.edsheeranA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_esd-x.edsheeran_sd1.4', 'rlct4.reV.edsheeranA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_esd-x.edsheeran_sd1.4', 'rlct4.reV.mrobbieA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-x.mrobbie_sd1.4', 'rlct4.reV.mrobbieA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_esd-x.mrobbie_sd1.4', 'rlct4.reV.mrobbieA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_esd-x.mrobbie_sd1.4', 'rlct4.reV.chemsworthA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-x.chemsworth_sd1.4', 'rlct4.reV.chemsworthA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_esd-x.chemsworth_sd1.4', 'rlct4.reV.chemsworthA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_esd-x.chemsworth_sd1.4', 'rlct4.reV.cevansA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-x.cevans_sd1.4', 'rlct4.reV.cevansA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_esd-x.cevans_sd1.4', 'rlct4.reV.cevansA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_esd-x.cevans_sd1.4']


exp_names = ['rlct4.reV.nicoparkerA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-u.octavia_sd1.4', 'rlct4.reV.nicoparkerA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_esd-u.edsheeran_sd1.4', 'rlct4.reV.nicoparkerA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_esd-u.edsheeran_sd1.4', 'rlct4.reV.nicoparkerA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-u.adriver_sd1.4', 'rlct4.reV.nicoparkerA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_esd-u.idris_sd1.4', 'rlct4.reV.nicoparkerA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_esd-u.ajolie_sd1.4', 'rlct4.reV.nicoparkerA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-u.rihanna_sd1.4', 'rlct4.reV.nicoparkerA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_esd-u.obama_sd1.4', 'rlct4.reV.nicoparkerA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_esd-u.idris_sd1.4', 'rlct4.reV.nicoparkerA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-u.morganf_sd1.4', 'rlct4.reV.nicoparkerA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_esd-u.ahathaway_sd1.4', 'rlct4.reV.nicoparkerA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_esd-u.obama_sd1.4', 'rlct4.reV.nicoparkerA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-u.oprah_sd1.4', 'rlct4.reV.nicoparkerA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_esd-u.obama_sd1.4', 'rlct4.reV.nicoparkerA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_esd-u.morganf_sd1.4', 'rlct4.reV.nicoparkerA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-u.oprah_sd1.4', 'rlct4.reV.nicoparkerA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_esd-u.chemsworth_sd1.4', 'rlct4.reV.nicoparkerA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_esd-u.obama_sd1.4', 'rlct4.reV.nicoparkerA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-u.drake_sd1.4', 'rlct4.reV.nicoparkerA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_esd-u.chemsworth_sd1.4', 'rlct4.reV.nicoparkerA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_esd-u.idris_sd1.4', 'rlct4.reV.nicoparkerA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-u.mrobbie_sd1.4', 'rlct4.reV.nicoparkerA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_esd-u.edsheeran_sd1.4', 'rlct4.reV.nicoparkerA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_esd-u.agarfield_sd1.4', 'rlct4.reV.nicoparkerA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-u.edsheeran_sd1.4', 'rlct4.reV.nicoparkerA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_esd-u.morganf_sd1.4', 'rlct4.reV.nicoparkerA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_esd-u.drake_sd1.4', 'rlct4.reV.nicoparkerA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-u.agarfield_sd1.4', 'rlct4.reV.nicoparkerA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_esd-u.ahathaway_sd1.4', 'rlct4.reV.nicoparkerA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_esd-u.mrobbie_sd1.4', 'rlct4.reV.nicoparkerA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-u.adriver_sd1.4', 'rlct4.reV.nicoparkerA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_esd-u.rihanna_sd1.4', 'rlct4.reV.nicoparkerA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_esd-u.edsheeran_sd1.4', 'rlct4.reV.nicoparkerA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-u.rihanna_sd1.4', 'rlct4.reV.nicoparkerA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_esd-u.mcarey_sd1.4', 'rlct4.reV.nicoparkerA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_esd-u.aadam_sd1.4']

exp_names = ['rlct4.reV.asanteA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-u.edsheeran_sd1.4', 'rlct4.reV.asanteA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_esd-u.edsheeran_sd1.4', 'rlct4.reV.asanteA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_esd-u.aadam_sd1.4', 'rlct4.reV.reeseA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-u.rihanna_sd1.4', 'rlct4.reV.reeseA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_esd-u.mrobbie_sd1.4', 'rlct4.reV.reeseA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_esd-u.morganf_sd1.4', 'rlct4.reV.nivolaA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-u.drake_sd1.4', 'rlct4.reV.nivolaA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_esd-u.octavia_sd1.4', 'rlct4.reV.nivolaA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_esd-u.aadam_sd1.4', 'rlct4.reV.earleA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-u.rihanna_sd1.4', 'rlct4.reV.earleA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_esd-u.obama_sd1.4', 'rlct4.reV.earleA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_esd-u.rihanna_sd1.4', 'rlct4.reV.leowoodalA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-u.octavia_sd1.4', 'rlct4.reV.leowoodalA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_esd-u.obama_sd1.4', 'rlct4.reV.leowoodalA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_esd-u.obama_sd1.4', 'rlct4.reV.starkeyA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-u.octavia_sd1.4', 'rlct4.reV.starkeyA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_esd-u.mrobbie_sd1.4', 'rlct4.reV.starkeyA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_esd-u.chemsworth_sd1.4', 'rlct4.reV.apierreA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-u.obama_sd1.4', 'rlct4.reV.apierreA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_esd-u.obama_sd1.4', 'rlct4.reV.apierreA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_esd-u.chemsworth_sd1.4', 'rlct4.reV.skyhblackA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-u.rihanna_sd1.4', 'rlct4.reV.skyhblackA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_esd-u.ahathaway_sd1.4', 'rlct4.reV.skyhblackA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_esd-u.mrobbie_sd1.4', 'rlct4.reV.sophiewildeA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-u.edsheeran_sd1.4', 'rlct4.reV.sophiewildeA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_esd-u.chemsworth_sd1.4', 'rlct4.reV.sophiewildeA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_esd-u.ahathaway_sd1.4', 'rlct4.reV.edebiriA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-u.edsheeran_sd1.4', 'rlct4.reV.edebiriA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_esd-u.chemsworth_sd1.4', 'rlct4.reV.edebiriA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_esd-u.mcarey_sd1.4', 'rlct4.reV.mmadisonA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-u.obama_sd1.4', 'rlct4.reV.mmadisonA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_esd-u.ahathaway_sd1.4', 'rlct4.reV.mmadisonA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_esd-u.octavia_sd1.4', 'rlct4.reV.nicoparkerA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-u.mrobbie_sd1.4', 'rlct4.reV.nicoparkerA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_esd-u.chemsworth_sd1.4', 'rlct4.reV.nicoparkerA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_esd-u.aadam_sd1.4']

exp_names = ['rlct4.reV.asanteA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-x.edsheeran_sd1.4', 'rlct4.reV.asanteA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_esd-x.edsheeran_sd1.4', 'rlct4.reV.asanteA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_esd-x.aadam_sd1.4', 'rlct4.reV.reeseA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-x.rihanna_sd1.4', 'rlct4.reV.reeseA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_esd-x.mrobbie_sd1.4', 'rlct4.reV.reeseA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_esd-x.morganf_sd1.4', 'rlct4.reV.nivolaA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-x.drake_sd1.4', 'rlct4.reV.nivolaA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_esd-x.octavia_sd1.4', 'rlct4.reV.nivolaA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_esd-x.aadam_sd1.4', 'rlct4.reV.earleA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-x.rihanna_sd1.4', 'rlct4.reV.earleA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_esd-x.obama_sd1.4', 'rlct4.reV.earleA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_esd-x.rihanna_sd1.4', 'rlct4.reV.leowoodalA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-x.octavia_sd1.4', 'rlct4.reV.leowoodalA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_esd-x.obama_sd1.4', 'rlct4.reV.leowoodalA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_esd-x.obama_sd1.4', 'rlct4.reV.starkeyA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-x.octavia_sd1.4', 'rlct4.reV.starkeyA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_esd-x.mrobbie_sd1.4', 'rlct4.reV.starkeyA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_esd-x.chemsworth_sd1.4', 'rlct4.reV.apierreA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-x.obama_sd1.4', 'rlct4.reV.apierreA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_esd-x.obama_sd1.4', 'rlct4.reV.apierreA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_esd-x.chemsworth_sd1.4', 'rlct4.reV.skyhblackA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-x.rihanna_sd1.4', 'rlct4.reV.skyhblackA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_esd-x.ahathaway_sd1.4', 'rlct4.reV.skyhblackA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_esd-x.mrobbie_sd1.4', 'rlct4.reV.sophiewildeA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-x.edsheeran_sd1.4', 'rlct4.reV.sophiewildeA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_esd-x.chemsworth_sd1.4', 'rlct4.reV.sophiewildeA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_esd-x.ahathaway_sd1.4', 'rlct4.reV.edebiriA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-x.edsheeran_sd1.4', 'rlct4.reV.edebiriA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_esd-x.chemsworth_sd1.4', 'rlct4.reV.edebiriA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_esd-x.mcarey_sd1.4', 'rlct4.reV.mmadisonA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-x.obama_sd1.4', 'rlct4.reV.mmadisonA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_esd-x.ahathaway_sd1.4', 'rlct4.reV.mmadisonA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_esd-x.octavia_sd1.4', 'rlct4.reV.nicoparkerA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-x.mrobbie_sd1.4', 'rlct4.reV.nicoparkerA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_esd-x.chemsworth_sd1.4', 'rlct4.reV.nicoparkerA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_esd-x.aadam_sd1.4']


exp_names = exp_names[v*len_:v*len_+len_]









# exp_names = [exp_names[3]]


apply_use_ti = True
apply_negative_prompt = True

# exp_names = [exp_names[0]]
count = 0

cfg_scales = [  4.5, 6.0]
cfg_scales = [7.5]

for exp_name in exp_names:
    # manual_prompt = 'A photo of a toy'# 'A photo of a toy'
    # manual_prompt = 'A photo of a hippo'
    
    
    is_relearn = 'uul' in exp_name or 'rl' in exp_name
    if is_relearn:
        base_exp_name = '_'.join(exp_name.split('_')[2:])
        relearn_exp_name = exp_name
        unlearn_exp_name = '_'.join(exp_name.split('_')[1:])
        exp_name = base_exp_name

    
    manual_prompt = ''
    use_general_concept = False
    # cfg_scales = np.arange(2.0,4.5, 0.5).tolist()
    # cfg_scales = np.arange(3.0,3.5, 0.5).tolist()

    # steps = [50,100,150,200]
    # for step in steps:
    steps =range(0, 3000+1, 200)
    steps =range(0, 1000+1, 100)
    steps =range(500, 500+1, 100)
    steps =range(0, 1000+1, 100)
    # steps =range(50, 1000+1, 100)
    # steps =range(50, 1000+1, 100)
    # steps =range(50, 1000+1, 100)
    # steps =range(25, 1000+1, 100)
    # steps = [300]
    # steps =range(700, 1000+1, 100)
    # steps =range(800, 1000+1, 100)
    # steps =range(300, 500+1, 100)
    for step in steps:
    # for step in [1200,1800]:

        # for cfg in cfg_scales:
        is_original_pretrained = exp_name == 'CompVis/stable-diffusion-v1-4'
        is_unlearn = 'ul' in exp_name and not is_relearn
        if 'moodeng' in exp_name: concept = 'moodeng'
        if 'crybaby' in exp_name: concept = 'crybaby'
        if 'avp' in exp_name: concept = 'avp'
        if 'chiquita' in exp_name: concept = 'chiquita'
        if 'sceleb5g0' in exp_name: concept = 'sceleb5g0'
        
        pretrained_path = 'CompVis/stable-diffusion-v1-4'
        if is_relearn:
            # pretrained_path = f"data_root/logs/{unlearn_exp_name}/LoRA_fusion_model"
            
            pretrained_path = 'CompVis/stable-diffusion-v1-4'
            
            for c in seen_concepts:
                # print(c)
                if c in data_info and c in unlearn_exp_name:
                    concept = c
                    # print("Found concept:", concept)
                    break
            # print(c,concept)
            concept_name = format_name(data_info[concept]['full_name']).replace(' ', '_')
            if 'esd-x' in unlearn_exp_name:
               train_method = 'esdx'
            elif 'esd-u' in unlearn_exp_name:
                train_method = 'esdu' 
            elif 'esd-all' in unlearn_exp_name:
                train_method = 'esdall'
            else:
                print(f"Error: Unrecognized training method in {exp_name}")
                assert False
            pretrained_unet_name = f"esd-{concept_name}-from-{concept_name}-{train_method}"
            unet_weight_path = f"data_root/logs/esd/sd1.4/{pretrained_unet_name}.safetensors"
             
            
            
            # erase_name = concept
            # if 'VPr' in exp_name: erase_name += 'VPr'
            # pretrained_path = f"data_root/logs/erase_l1.{erase_name}.object_lr2.5e-4/LoRA_fusion_model"
        if is_unlearn: 
            pretrained_path = f"data_root/logs/{exp_name}/LoRA_fusion_model"

        use_ti = 'ti' in relearn_exp_name or '-V' in exp_name 
        
        if is_relearn and not 'reV' in relearn_exp_name:
            # relearn is not re-initializing the token (by default)
            initializer_token = ''
        elif use_ti:
            initializer_token = concept2initializer[concept]

        if manual_prompt:
            prompt = manual_prompt
        elif use_general_concept:
            prompt = concept2generalprompt[concept]
        
        if use_ti:
            if 'sceleb' in concept:
                prompt = 'A photo of a v1,A photo of a v2,A photo of a v3,A photo of a v4,A photo of a v5'
                placeholder_token = 'v1,v2,v3,v4,v5'
            else:
                prompt = 'a photo of v1' 
                placeholder_token = 'v1'
        else:
            prompt = concept2prompt[concept]
            
        ## hacky .. should change this later
        if is_relearn:
            exp_name = relearn_exp_name
        if is_unlearn or 'erase' in exp_name or exp_name == 'original_pretrained': 
            load_lora_weight_path = ''
            gen_image_path = f"data_root/generated/model/{exp_name}"
        else:
            load_lora_weight_path =f"data_root/logs/{exp_name}/checkpoint-{step}"
            gen_image_path = 'auto'
            
        # if 'l0' in exp_name :
        #     load_lora_weight_path = ''
        
        script = f"""
        accelerate launch train_dreambooth_lora.py \\
            --pretrained_model_name_or_path='{pretrained_path}'  \\
            --load_unet_weight_path="{unet_weight_path}" \\
            --load_lora_weight_path="{load_lora_weight_path}" \\
            --instance_data_dir="data_root/data/real_data/dummy" \\
            --gen_image_path="{gen_image_path}" \\
            --output_dir="data_root/logs/gen" \\
            --validation_prompt="{prompt}" --instance_prompt="{prompt}" \\
            --lora_rank 1 --target_lora_modules to_k to_v --target_lora_layers cross \\
            --run_note 'gen img' --wait_weight \\
            --num_validation_images 50 \\"""
            
                
        if use_ti and not is_unlearn:
            script += f"""
            --load_token_embedding_path="data_root/logs/{exp_name}/checkpoint-{step}" \\
            --placeholder_token="{placeholder_token}" --initializer_token='{initializer_token}' \\"""

        if apply_negative_prompt:
            script += f"""
            --negative_prompt "longbody, lowres, bad anatomy, bad hands, missing fingers, extra digit, fewer digits, cropped, worst quality, low quality." \\"""
        

        script += f"""
            --cfg_scale {','.join(f'{x:.2f}' for x in cfg_scales)}"""

        print(f"echo 'count:{count} - {exp_name} {step} /'")
        print(script) 
        
        count += 1
print(f"Total scripts generated: {count}")
        

echo 'count:0 - rlct4.reV.edebiriA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-x.edsheeran_sd1.4 0 /'

        accelerate launch train_dreambooth_lora.py \
            --pretrained_model_name_or_path='CompVis/stable-diffusion-v1-4'  \
            --load_unet_weight_path="data_root/logs/esd/sd1.4/esd-Ed_Sheeran-from-Ed_Sheeran-esdx.safetensors" \
            --load_lora_weight_path="data_root/logs/rlct4.reV.edebiriA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_esd-x.edsheeran_sd1.4/checkpoint-0" \
            --instance_data_dir="data_root/data/real_data/dummy" \
            --gen_image_path="auto" \
            --output_dir="data_root/logs/gen" \
            --validation_prompt="a photo of v1" --instance_prompt="a photo of v1" \
            --lora_rank 1 --target_lora_modules to_k to_v --target_lora_layers cross \
            --run_note 'gen img' --wait_weight \
            --num_validation_images 50 \
            --load_token_embedding_path="data_root/logs/rlct4.reV.edebiriA5V0.ln.lr1e-4.ti5e-4.pr

# MACE

In [ ]:

# unlearned MACE

base_exps = [
    # 'rv'
    'sd1.4'

]

# manual_target_concepts = ["obama"] # "obama" # "rihanna" # "mrobbie" # "edsheeran" # "mrobbie" # "obama" # "rihanna" # "edsheeran"
# seed_adds = [0,1,2]
manual_target_concepts = seen_concepts 
seed_adds = [0]

exp_names = []
# domain_preservations = ["8e+3"] # ["8e+2","8e+3","8e-4"] # ["8e+2","8e+3","8e-4"]#  ["8e+3"] # ["8e+3"] # ["8e+4","8e+5"] #  ["8e+2","8e+3","8e-4","8e-5"] # 8.0e+3, 8.0e+4 2e-5 
# general_preservations = ["1e-4"]  # ["1e-0","1e-2","1e-4"] # ,"1e-4"] # preservation sclae for the closed-form
# domain_preservations = ["5e-4"] # for 5 concepts
general_preservations = ["1e-4"]
domain_preservations = ["8e+3"]
learning_rates = ["1e-4"]# ["1e-3", "1e-4", "1e-5"]
num_gen_images = [8] # [50] # [8] 
lora_ranks = [1] # [1]
img_types = ["G"] # ["r","g"]
max_train_steps = [50] # [50,200]
# use_prs = [True] # [True, False]

# seed_adds = [3,4]

# sur_concept = 'object'
base_exp_steps = [0] # we want to see it fit first


for manual_target_concept in manual_target_concepts:
    for base_exp in base_exps:
        for seed_add in seed_adds:
        
            if not manual_target_concept:
                # Try to infer target_concept from base_exp
                possible_concepts = ['moodeng', 'crybaby', 'avp', 'chiquita', 'reese', 'gout', 'jooli', 'honer', 'sceleb5g0']
                for concept in possible_concepts:
                    if concept in base_exp:
                        target_concept = concept
                        # print(f"target_concept is not set, inferred and set to '{target_concept}' from base_exp")
                        break
            else:
                target_concept = manual_target_concept
                # print(f"target_concept is set to '{target_concept}' manually")
            # else:
            #     print("Warning: target_concept is not set and could not be inferred from base_exp.")



            for base_exp_step in base_exp_steps:
                for lr, num_img, lora_rank, img_type, gen_pr, domain_pr, max_train_step in itertools.product(
                    learning_rates, num_gen_images, lora_ranks, img_types, general_preservations,domain_preservations, max_train_steps
                ):
                    
                    
                    if 'sd1.4' in base_exp:
                        pretrained_model_name_or_path = 'CompVis/stable-diffusion-v1-4'
                    elif 'sd1.5' in base_exp:
                        pretrained_model_name_or_path = 'runwayml/stable-diffusion-v1-5'
                    elif 'ch.' in base_exp: 
                        pretrained_model_name_or_path = 'stablediffusionapi/chilloutmix'
                    
                    
                    if base_exp == 'rv':
                        pretrained_model_name_or_path = 'stablediffusionapi/chilloutmix'
                        
                    # print(lr, num_img, lora_rank, img_type, steps)
                    
                    ul_name  = f'ul{lora_rank}.prg{gen_pr}d{domain_pr}.lr{lr}.n{num_img}.{img_type}'
                    
                    base_exp_name_tag = f'{base_exp}.s{base_exp_step}'
                    if base_exp == 'rv':
                        base_exp_name_tag = 'rv'
                    if base_exp == 'sd1.4':
                        base_exp_name_tag = 'sd14'
                    
                    
                    if seed_add > 0:
                        exp_name = f"{ul_name}.{target_concept}.{concept2mapping_concept[target_concept][0]}.s{max_train_step}.r{seed_add}_{base_exp_name_tag}"
                        
                    else:
                        exp_name = f"{ul_name}.{target_concept}.{concept2mapping_concept[target_concept][0]}.s{max_train_step}_{base_exp_name_tag}"
                    final_seed = 2024 + seed_add
                    
                    mapping_concept = f"['{concept2mapping_concept[target_concept][1]}']"
                    config_name =  "erase_default.yaml"
                    if 'sceleb5' in target_concept:
                        config_name = "erase_sceleb_5.yaml"
                    
                    
                    script = f""" python data_preparation.py configs/custom/{config_name} \\
                    exp_name="{exp_name}" \\
                    MACE.pretrained_model_name_or_path="{pretrained_model_name_or_path}" \\
                    MACE.num_gen_images={num_img} MACE.seed={final_seed} \\
                    MACE.multi_concept="[ [ [{erase_target_concept[target_concept]}, object] ] ]" \\
                    MACE.input_data_dir="data_root/generated/mace/{base_exp}/r{seed_add}"
        python training.py configs/custom/{config_name} \\
                    exp_name="{exp_name}" \\
                    MACE.pretrained_model_name_or_path="{pretrained_model_name_or_path}" \\
                    MACE.learning_rate={lr} MACE.max_train_steps={max_train_step} MACE.seed={final_seed} \\
                    MACE.rank={lora_rank} \\
                    MACE.num_gen_images={num_img} \\
                    MACE.domain_preservation_cache_path={concept2domain_preservation_cache_path[target_concept]} MACE.mapping_concept="{mapping_concept}" \\
                    MACE.train_preserve_scale={gen_pr} MACE.preserve_weight={domain_pr} \\
                    MACE.multi_concept="[ [ [{erase_target_concept[target_concept]}, object] ] ]" \\
                    MACE.input_data_dir="data_root/generated/mace/{base_exp}/r{seed_add}" 
                    """


                    
                    print(script)
                    # print(exp_name)
                    exp_names += [exp_name]
print(exp_names)

 python data_preparation.py configs/custom/erase_default.yaml \
                    exp_name="ul1.prg1e-4d8e+3.lr1e-4.n8.G.obama.person.s50_sd14" \
                    MACE.pretrained_model_name_or_path="CompVis/stable-diffusion-v1-4" \
                    MACE.num_gen_images=8 MACE.seed=2024 \
                    MACE.multi_concept="[ [ [barrack-obama, object] ] ]" \
                    MACE.input_data_dir="data_root/generated/mace/sd1.4/r0"
        python training.py configs/custom/erase_default.yaml \
                    exp_name="ul1.prg1e-4d8e+3.lr1e-4.n8.G.obama.person.s50_sd14" \
                    MACE.pretrained_model_name_or_path="CompVis/stable-diffusion-v1-4" \
                    MACE.learning_rate=1e-4 MACE.max_train_steps=50 MACE.seed=2024 \
                    MACE.rank=1 \
                    MACE.num_gen_images=8 \
                    MACE.domain_preservation_cache_path=data_root/cache/mace/cache_cele.pt MACE.mapping_concept="['a person']" \
                    MACE.

In [ ]:
# MACE relearning
# decoding unlearning - with same hyperparameter

# Implement text encoder relearning

# ul_exp_names = ['ul1.prg1e-4d5e-4.lr1e-4.n8.G.sceleb5g0.person.s50_c.l16.kv_sceleb5g0N50-V_pr0.50_lr5e-5.ti5e-4_f0.5_b4g4.s10000']

ul_exp_names = ['ul1.prg1e-4d8e+3.lr1e-4.n8.G.morganf.person.s50_rv', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.morganf.person.s50.r1_rv', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.morganf.person.s50.r2_rv']






concept = "apierre" # 'moodeng' # 'crybaby' # 'avp' # 'chiquita' # 'reese' # 'gout' # 'jooli' # 'honer' # 'sceleb5g0'



#seed = 0

seeds = [0,1,2] # along

# seeds = [3,4]
# seeds = [0,1,2,3,4]

use_te = True
batch_size = 1
gradient_accumulation_step = 4
# re
# lr_lora_grid = ["1e-4", "5e-5", "1e-5"]
lr_lora_grid = ["1e-4"]
# lr_lora_grid = [ "1e-5"]
lr_ti_grid = ["5e-4"]   # only used if use_ti
# lr_ti_grid   = ["5e-3","5e-2"]   # only used if use_ti
lr_te_grid = ["1e-5"] 
lora_ranks = [4] # [1,2,4,8,16,32,64]
# # Create all combinations of lr_lora, lr_ti, and seed
combos = list(itertools.product(lr_lora_grid,
                                lr_ti_grid,
                                lr_te_grid if use_te else [None],
                                lora_ranks))

pretrained = 'ch'
use_pr = True

use_te = True
apply_use_ti = True

lr_scheduler = 'linear' # 'linear' # 'cosine' # 'cosine_with_restarts'
apply_negative_prompt = True

use_manual_params = True
reV = True
is_relearn = True 

# fix here #
manual_params = {
    # 'data_setting': 'facefew',
    'data_setting': 'align',
    
    # 'data_setting': 'small',s
    # 'lora_rank' :  16
}

data_setting = 'align' 

final_exp_names = []
# for ul_exp_name in ul_exp_names:
for ul_exp_name,seed in zip(ul_exp_names,seeds):
    
    base_exp_name = '_'.join(ul_exp_name.split('_')[1:])
    exp_name = base_exp_name
    # print(f"Base Experiment Name: {base_exp_name}")
    
    #fix edit here : they are using this to reconstruct the base_exp as well
    lr_lora, lr_ti = extract_lrs(base_exp_name)
    lora_rank = extract_learning_lora_rank(base_exp_name)
    
    # todo: better use 're'
    for re_lr_lora, re_lr_ti, re_lr_te, re_lora_rank in combos:
        
        # print(f'lora_rank: {lora_rank}, lr: {lr_lora}, ti_lr: {lr_ti}')
        pretrained_path = f"data_root/logs/{ul_exp_name}/LoRA_fusion_model"

        if not concept:
            # if 'moodeng' in ul_exp_name: concept = 'moodeng'
            # if 'crybaby' in ul_exp_name: concept = 'crybaby'
            # if 'avp' in ul_exp_name: concept = 'avp'
            # if 'chiquita' in ul_exp_name: concept = 'chiquita'
            # if 'reese' in ul_exp_name: concept = 'reese'
            # if 'gout' in ul_exp_name: concept = 'gout'
            # if 'jooli' in ul_exp_name: concept = 'jooli'
            # if 'honer' in ul_exp_name: concept = 'honer'
            # if 'obama' in ul_exp_name: concept = 'obama'
            # if 'rihanna' in ul_exp_name: concept = 'rihanna'
            # if 'edsheeran' in ul_exp_name: concept = 'edsheeran'
            # if 'mrobbie' in ul_exp_name: concept = 'mrobbie'
            for c in concept2mapping_concept:
                if c in ul_exp_name:
                    concept = c
                    break

        use_ti = 'ti' in exp_name or '-V' in exp_name or apply_use_ti
        # use_pr = 'pr' in exp_name

        if data_setting == 'facefew':
            dataset_name = f'{concept}5F0r{seed}'
            
        elif data_setting == 'align':
            dataset_name = f'{concept}A5V0'
        elif data_setting == 'fewshot':
            
            if 'sceleb' in concept:
                dataset_name = f'{concept}U3'
            elif concept == 'avp':
                dataset_name = 'avpS3'
            else:
                dataset_name = f'{concept}U3'
        elif data_setting == 'small':
            
            if 'sceleb' in concept:
                dataset_name = f'{concept}N10'
            else:
                dataset_name = f'{concept}10'
        else:
            
            if 'sceleb' in concept:
                dataset_name = f'{concept}N50'
            elif concept == 'avp':
                dataset_name = 'avp20'
            else:
                dataset_name = f'{concept}50'
                
        if reV:
            initializer_token = concept2initializer[concept]
        else: 
            initializer_token = ''
            
        if use_ti:
            if 'sceleb' in concept:
                prompt = 'A photo of a v1,A photo of a v2,A photo of a v3,A photo of a v4,A photo of a v5'
                placeholder_token = 'v1,v2,v3,v4,v5'
            else:
                prompt = 'a photo of v1' 
                placeholder_token = 'v1'
        else:
            prompt = concept2prompt[concept]

        name_tag = ''
        if is_relearn: name_tag += 'uul'
        name_tag = f'{name_tag} {dataset_name}'
        name_tag += f' l{lora_rank}'
        if use_ti: 
            # name_tag += f' ti.{lr_ti}'
            name_tag += f' ti'

        data_root = dataset_name2data_root[dataset_name]
        
        if use_ti:
            dataset_name_for_exp = dataset_name + "-V"
            # if use_ni:
            #     dataset_name_for_exp += ".ni"
        else: dataset_name_for_exp = dataset_name
        
        
        # prior preservation folder
        prior_folder = 'original_realistic_vision'
        if pretrained == 'sd1.5':
            prior_folder = 'original_pretrained_sd1.5'
        if pretrained == 'sd1.4':
            prior_folder = 'original_pretrained_sd1.4'     
        if pretrained == 'rv':
            prior_folder = 'original_realistic_vision'
        elif pretrained == 'ch':
            prior_folder = 'original_chilloutmix'

        
        # renaming to check
        # re_exp_name = f'c.l{lora_rank}.kv_{dataset_name_for_exp}'
        # if use_pr:
        #     re_exp_name += f'_pr0.50'
        # re_exp_name += '_lr'
        # if lora_rank >0: re_exp_name += f"{str(lr_lora)}"
        # if use_ti:
        #     re_exp_name += f'.ti{str(lr_ti)}'
        # re_exp_name += '_f0.5_b1g4'
        
        # print(re_exp_name)
        # assert re_exp_name in base_exp_name, f"Expected {re_exp_name} in {base_exp_name}"

        # if manual_lora is not None and manual_data != lora_rank:
        
        if use_manual_params:
            lora_rank = re_lora_rank
            eff_data_setting = manual_params['data_setting']

            lr_lora, lr_ti = re_lr_lora, re_lr_ti
            
            if eff_data_setting == 'facefew':
                dataset_name = f'{concept}5F0r{seed}'
                
            elif eff_data_setting == 'align':
                dataset_name = f'{concept}A5V0'
            
            elif eff_data_setting == 'fewshot':
                
                if 'sceleb' in concept:
                    dataset_name = f'{concept}U3'
                elif concept == 'avp':
                    dataset_name = 'avpS3'
                else:
                    dataset_name = f'{concept}U3'
            elif eff_data_setting == 'small':
                
                if 'sceleb' in concept:
                    dataset_name = f'{concept}N10'
                else:
                    dataset_name = f'{concept}10'
            else:
                
                if 'sceleb' in concept:
                    dataset_name = f'{concept}N50'
                elif concept == 'avp':
                    dataset_name = 'avp20'
                else:
                    dataset_name = f'{concept}50'            
                    
            data_root = dataset_name2data_root[dataset_name]
        
        
        
        
        if reV:
            if use_te:
                relearn_exp_name = f"rlct{lora_rank}.reV.{dataset_name}"
            else:
                relearn_exp_name = f"rlc{lora_rank}.reV.{dataset_name}"
        else:
            if use_te:
                relearn_exp_name = f"rlct{lora_rank}.{dataset_name}"
            else:
                relearn_exp_name = f"rlc{lora_rank}.{dataset_name}"
         
           
        if lr_scheduler == 'linear':
            relearn_exp_name += f".ln"
        relearn_exp_name += f".lr{re_lr_lora}.ti{re_lr_ti}"
            
            
        if use_pr:
            relearn_exp_name += f".pr1.00"
            if apply_negative_prompt:
                relearn_exp_name += ".neg"
        
        relearn_exp_name += f".b{batch_size}g{gradient_accumulation_step}"
        
        if seed != 0:
            relearn_exp_name += f".r{seed}"

        final_exp_name = f"{relearn_exp_name}_{ul_exp_name}"
        
        
        script = f"""
        accelerate launch train_dreambooth_lora.py \\
        --pretrained_model_name_or_path={pretrained_path}  \\
        --instance_data_dir={data_root} \\
        --output_dir="data_root/logs/{final_exp_name}" \\
        --validation_prompt="{prompt}" --instance_prompt="{prompt}" \\
        --train_batch_size={batch_size} --gradient_accumulation_steps={gradient_accumulation_step} \\
        --lora_rank {lora_rank} --target_lora_modules to_k to_v --target_lora_layers cross \\
        --max_train_steps=1000  --validation_steps=50  --checkpointing_steps=50  --lr_scheduler "{lr_scheduler}"  --seed {seed} \\
        --run_note '{name_tag}' \\"""
            
        script += f"""
        --cfg_scale 6.0 \\"""
    
    
        if apply_negative_prompt:
            script += f"""
        --negative_prompt "longbody, lowres, bad anatomy, bad hands, missing fingers, extra digit, fewer digits, cropped, worst quality, low quality." \\"""
            
            
        if use_pr:
            script += f"""
        --with_prior_preservation --prior_loss_weight=1.0 --num_class_images 200 \\
        --class_prompt="a photo of a person" --class_data_dir="data_root/generated/model/{prior_folder}/a photo of a person_neg/6.00" \\"""
            
        # Conditional learning rate + TI options
        if use_ti:
            
            if lora_rank <= 0:
                script += f"""
        --learning_rate_ti {lr_ti} \\
        --placeholder_token="{placeholder_token}" --initializer_token='{initializer_token}'"""
            else:
                if use_te:
                    script += f"""
        --learning_rate_lora {lr_lora} --learning_rate_ti {lr_ti} \\
        --train_text_encoder --learning_rate_lora_text_encoder {re_lr_te} \\
        --placeholder_token="{placeholder_token}" --initializer_token='{initializer_token}'"""
                else:
                    script += f"""
        --learning_rate_lora {lr_lora} --learning_rate_ti {lr_ti} \\
        --placeholder_token="{placeholder_token}" --initializer_token='{initializer_token}'"""
        else:
            script += f"""
        --learning_rate {lr_lora}"""

        print(script)
        final_exp_names += [final_exp_name]
print(final_exp_names)

        


        accelerate launch train_dreambooth_lora.py \
        --pretrained_model_name_or_path=data_root/logs/ul1.prg1e-4d8e+3.lr1e-4.n8.G.morganf.person.s50_rv/LoRA_fusion_model  \
        --instance_data_dir=data_root/data/real_data/apierre/aligned/apierre-5-v0 \
        --output_dir="data_root/logs/rlct4.reV.apierreA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_ul1.prg1e-4d8e+3.lr1e-4.n8.G.morganf.person.s50_rv" \
        --validation_prompt="a photo of v1" --instance_prompt="a photo of v1" \
        --train_batch_size=1 --gradient_accumulation_steps=4 \
        --lora_rank 4 --target_lora_modules to_k to_v --target_lora_layers cross \
        --max_train_steps=1000  --validation_steps=50  --checkpointing_steps=50  --lr_scheduler "linear"  --seed 0 \
        --run_note 'uul apierreA5V0 lNone ti' \
        --cfg_scale 6.0 \
        --negative_prompt "longbody, lowres, bad anatomy, bad hands, missing fingers, extra digit, fewer digits, cropped, worst quality, low quality." \
        --with_prio

In [24]:
exp_name = 'rlct4.reV.mcareyA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_ul1.prg1e-4d8e+3.lr1e-4.n8.G.mcarey.person.s50_sd14'

unlearn_exp_name = '_'.join(exp_name.split('_')[1:])

unlearn_exp_name

'ul1.prg1e-4d8e+3.lr1e-4.n8.G.mcarey.person.s50_sd14'

In [30]:
# MACE-generation
# exp_names = ['rl4.chiquita50_ul1.prg1e-4d8e-5.lr1e-4.n8.G.chiquita.person.s50_c.l4.kv_chiquita50-V_pr0.50_lr5e-4.ti5e-2_f0.5_b1g4.s3000']
#    ['uul1.lr1e-4.n8.G.chiquita.obj.s8_c.l4.kv_chiquita50-V_pr0.50_lr5e-4.ti5e-2_f0.5_b1g4.s3000', 
    # 'uul1.lr1e-4.n8.G.chiquita.obj.s0_c.l4.kv_chiquita50-V_pr0.50_lr5e-4.ti5e-2_f0.5_b1g4.s3000']
# ul_exp_names = , 'ul1.lr1e-4.n8.G.chiquita.obj.s0_c.l4.kv_chiquita50-V_pr0.50_lr5e-4.ti1e-3_f0.5_b1g4.s3000']
# ul_exp_names = [, ]
# uul1.prg8e+7d1e-2.lr1e-4.n8.G.chiquita.obj.s0_c.l4.kv_chiquita50-V_pr0.50_lr5e-4.ti1e-2_f0.5_b1g4.s3000
exp_names = ['rl16.reV.sceleb5g0N10.lr5e-5.ti5e-4.r2_ul1.prg1e-4d5e-4.lr1e-4.n8.G.sceleb5g0.person.s50.r2_c.l16.kv_sceleb5g0N50-V_pr0.50_lr5e-5.ti5e-4_f0.5_b4g4.s10000']


exp_names = ['rlct4.reV.honer10.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_ul1.prg1e-4d8e+3.lr1e-4.n8.G.honer.person.s50_ch.c.l16.kv_honer50-V.r_pr1.00.neg_lr5e-4.ti5e-4_b1g4.s2000']

exp_names = ['rlct4.reV.rihanna5F0r0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_ul1.prg1e-4d8e+3.lr1e-4.n8.G.rihanna.person.s50_rv', 'rlct4.reV.rihanna5F0r1.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_ul1.prg1e-4d8e+3.lr1e-4.n8.G.rihanna.person.s50.r1_rv', 'rlct4.reV.rihanna5F0r2.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_ul1.prg1e-4d8e+3.lr1e-4.n8.G.rihanna.person.s50.r2_rv', 'rlct4.reV.rihanna5F0r3.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r3_ul1.prg1e-4d8e+3.lr1e-4.n8.G.rihanna.person.s50.r3_rv', 'rlct4.reV.rihanna5F0r4.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r4_ul1.prg1e-4d8e+3.lr1e-4.n8.G.rihanna.person.s50.r4_rv']



exp_names = ['rlct4.reV.mcareyA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_ul1.prg1e-4d8e+3.lr1e-4.n8.G.mcarey.person.s50_sd14', 'rlct4.reV.mcareyA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_ul1.prg1e-4d8e+3.lr1e-4.n8.G.mcarey.person.s50.r1_sd14', 'rlct4.reV.mcareyA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_ul1.prg1e-4d8e+3.lr1e-4.n8.G.mcarey.person.s50.r2_sd14', 'rlct4.reV.octaviaA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_ul1.prg1e-4d8e+3.lr1e-4.n8.G.octavia.person.s50_sd14', 'rlct4.reV.octaviaA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_ul1.prg1e-4d8e+3.lr1e-4.n8.G.octavia.person.s50.r1_sd14', 'rlct4.reV.octaviaA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_ul1.prg1e-4d8e+3.lr1e-4.n8.G.octavia.person.s50.r2_sd14', 'rlct4.reV.oprahA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_ul1.prg1e-4d8e+3.lr1e-4.n8.G.oprah.person.s50_sd14', 'rlct4.reV.oprahA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_ul1.prg1e-4d8e+3.lr1e-4.n8.G.oprah.person.s50.r1_sd14', 'rlct4.reV.oprahA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_ul1.prg1e-4d8e+3.lr1e-4.n8.G.oprah.person.s50.r2_sd14', 'rlct4.reV.morganfA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_ul1.prg1e-4d8e+3.lr1e-4.n8.G.morganf.person.s50_sd14', 'rlct4.reV.morganfA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_ul1.prg1e-4d8e+3.lr1e-4.n8.G.morganf.person.s50.r1_sd14', 'rlct4.reV.morganfA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_ul1.prg1e-4d8e+3.lr1e-4.n8.G.morganf.person.s50.r2_sd14', 'rlct4.reV.drakeA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_ul1.prg1e-4d8e+3.lr1e-4.n8.G.drake.person.s50_sd14', 'rlct4.reV.drakeA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_ul1.prg1e-4d8e+3.lr1e-4.n8.G.drake.person.s50.r1_sd14', 'rlct4.reV.drakeA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_ul1.prg1e-4d8e+3.lr1e-4.n8.G.drake.person.s50.r2_sd14', 'rlct4.reV.idrisA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_ul1.prg1e-4d8e+3.lr1e-4.n8.G.idris.person.s50_sd14', 'rlct4.reV.idrisA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r1_ul1.prg1e-4d8e+3.lr1e-4.n8.G.idris.person.s50.r1_sd14', 'rlct4.reV.idrisA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4.r2_ul1.prg1e-4d8e+3.lr1e-4.n8.G.idris.person.s50.r2_sd14']
# exp_names = ['ul1.prg1e-4d8e+3.lr1e-4.n8.G.obama.person.s50_sd14',]
exp_names = ['ul1.prg1e-4d8e+3.lr1e-4.n8.G.obama.person.s50_sd14', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.rihanna.person.s50_sd14', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.edsheeran.person.s50_sd14', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.mrobbie.person.s50_sd14', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.chemsworth.person.s50_sd14', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.cevans.person.s50_sd14', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.aadam.person.s50_sd14', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.ahathaway.person.s50_sd14', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.mcarey.person.s50_sd14', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.octavia.person.s50_sd14', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.morganf.person.s50_sd14', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.drake.person.s50_sd14']
# exp_names = ['rlct4.reV.obamaA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_ul1.prg1e-4d8e+3.lr1e-4.n8.G.obama.person.s50_sd14', 'rlct4.reV.rihannaA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_ul1.prg1e-4d8e+3.lr1e-4.n8.G.rihanna.person.s50_sd14', 'rlct4.reV.edsheeranA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_ul1.prg1e-4d8e+3.lr1e-4.n8.G.edsheeran.person.s50_sd14', 'rlct4.reV.mrobbieA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_ul1.prg1e-4d8e+3.lr1e-4.n8.G.mrobbie.person.s50_sd14', 'rlct4.reV.chemsworthA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_ul1.prg1e-4d8e+3.lr1e-4.n8.G.chemsworth.person.s50_sd14', 'rlct4.reV.cevansA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_ul1.prg1e-4d8e+3.lr1e-4.n8.G.cevans.person.s50_sd14', 'rlct4.reV.aadamA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_ul1.prg1e-4d8e+3.lr1e-4.n8.G.aadam.person.s50_sd14', 'rlct4.reV.ahathawayA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_ul1.prg1e-4d8e+3.lr1e-4.n8.G.ahathaway.person.s50_sd14', 'rlct4.reV.mcareyA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_ul1.prg1e-4d8e+3.lr1e-4.n8.G.mcarey.person.s50_sd14', 'rlct4.reV.octaviaA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_ul1.prg1e-4d8e+3.lr1e-4.n8.G.octavia.person.s50_sd14', 'rlct4.reV.morganfA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_ul1.prg1e-4d8e+3.lr1e-4.n8.G.morganf.person.s50_sd14', 'rlct4.reV.drakeA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_ul1.prg1e-4d8e+3.lr1e-4.n8.G.drake.person.s50_sd14']


# exp_names = exp_names[3:]

apply_use_ti = True
apply_negative_prompt = True

# exp_names = [exp_names[0]]
count = 0

cfg_scales = [  4.5, 6.0]
cfg_scales = [7.5,6.0]
cfg_scales = [7.5]

for exp_name in exp_names:
    # manual_prompt = 'A photo of a toy'# 'A photo of a toy'
    # manual_prompt = 'A photo of a hippo'
    
    
    is_relearn = 'uul' in exp_name or 'rl' in exp_name
    if is_relearn:
        base_exp_name = '_'.join(exp_name.split('_')[2:])
        relearn_exp_name = exp_name
        unlearn_exp_name = '_'.join(exp_name.split('_')[1:])
        exp_name = base_exp_name

    
    manual_prompt = ''
    use_general_concept = False
    # cfg_scales = np.arange(2.0,4.5, 0.5).tolist()
    # cfg_scales = np.arange(3.0,3.5, 0.5).tolist()

    # steps = [50,100,150,200]
    # for step in steps:
    steps =range(0, 3000+1, 200)
    steps =range(0, 1000+1, 100)
    steps =range(500, 500+1, 100)
    steps =range(0, 1000+1, 100)
    # steps =range(0, 3000+1, 200)
    # steps =range(50, 1000+1, 100)
    # steps =range(50, 1000+1, 100)
    # steps =range(50, 1000+1, 100)
    # steps =range(25, 1000+1, 100)
    # steps = [300]
    # steps =range(700, 1000+1, 100)
    # steps =range(800, 1000+1, 100)
    # steps =range(300, 500+1, 100)
    for step in steps:
    # for step in [1200,1800]:

        # for cfg in cfg_scales:
        is_original_pretrained = exp_name == 'CompVis/stable-diffusion-v1-4'
        is_unlearn = 'ul' in exp_name and not is_relearn
        if 'moodeng' in exp_name: concept = 'moodeng'
        if 'crybaby' in exp_name: concept = 'crybaby'
        if 'avp' in exp_name: concept = 'avp'
        if 'chiquita' in exp_name: concept = 'chiquita'
        if 'sceleb5g0' in exp_name: concept = 'sceleb5g0'
        
        pretrained_path = 'CompVis/stable-diffusion-v1-4'
        if is_relearn:
            pretrained_path = f"data_root/logs/{unlearn_exp_name}/LoRA_fusion_model"
            # erase_name = concept
            # if 'VPr' in exp_name: erase_name += 'VPr'
            # pretrained_path = f"data_root/logs/erase_l1.{erase_name}.object_lr2.5e-4/LoRA_fusion_model"
        if is_unlearn: 
            pretrained_path = f"data_root/logs/{exp_name}/LoRA_fusion_model"

        use_ti = 'ti' in relearn_exp_name or '-V' in exp_name 
        
        if is_relearn and not 'reV' in relearn_exp_name:
            # relearn is not re-initializing the token (by default)
            initializer_token = ''
        elif use_ti:
            initializer_token = concept2initializer[concept]

        if manual_prompt:
            prompt = manual_prompt
        elif use_general_concept:
            prompt = concept2generalprompt[concept]
        
        if use_ti:
            if 'sceleb' in concept:
                prompt = 'A photo of a v1,A photo of a v2,A photo of a v3,A photo of a v4,A photo of a v5'
                placeholder_token = 'v1,v2,v3,v4,v5'
            else:
                prompt = 'a photo of v1' 
                placeholder_token = 'v1'
        else:
            prompt = concept2prompt[concept]
            
        ## hacky .. should change this later
        if is_relearn:
            exp_name = relearn_exp_name
        if is_unlearn or 'erase' in exp_name or exp_name == 'original_pretrained': 
            load_lora_weight_path = ''
            gen_image_path = f"data_root/generated/model/{exp_name}"
        else:
            load_lora_weight_path =f"data_root/logs/{exp_name}/checkpoint-{step}"
            gen_image_path = 'auto'
            
        # if 'l0' in exp_name :
        #     load_lora_weight_path = ''
        
        script = f"""
        accelerate launch train_dreambooth_lora.py \\
            --pretrained_model_name_or_path='{pretrained_path}'  \\
            --instance_data_dir="data_root/data/real_data/dummy" \\
            --load_lora_weight_path="{load_lora_weight_path}" \\
            --gen_image_path="{gen_image_path}" \\
            --output_dir="data_root/logs/gen" \\
            --validation_prompt="{prompt}" --instance_prompt="{prompt}" \\
            --lora_rank 1 --target_lora_modules to_k to_v --target_lora_layers cross \\
            --run_note 'gen img' --wait_weight \\
            --num_validation_images 50 \\"""
            
                
        if use_ti and not is_unlearn:
            script += f"""
            --load_token_embedding_path="data_root/logs/{exp_name}/checkpoint-{step}" \\
            --placeholder_token="{placeholder_token}" --initializer_token='{initializer_token}' \\"""

        if apply_negative_prompt:
            script += f"""
            --negative_prompt "longbody, lowres, bad anatomy, bad hands, missing fingers, extra digit, fewer digits, cropped, worst quality, low quality." \\"""
        

        script += f"""
            --cfg_scale {','.join(f'{x:.2f}' for x in cfg_scales)}"""
    
        print(script) 
        
        count += 1
print(f"Total scripts generated: {count}")
        


        accelerate launch train_dreambooth_lora.py \
            --pretrained_model_name_or_path='data_root/logs/ul1.prg1e-4d8e+3.lr1e-4.n8.G.obama.person.s50_sd14/LoRA_fusion_model'  \
            --instance_data_dir="data_root/data/real_data/dummy" \
            --load_lora_weight_path="" \
            --gen_image_path="data_root/generated/model/ul1.prg1e-4d8e+3.lr1e-4.n8.G.obama.person.s50_sd14" \
            --output_dir="data_root/logs/gen" \
            --validation_prompt="a photo of v1" --instance_prompt="a photo of v1" \
            --lora_rank 1 --target_lora_modules to_k to_v --target_lora_layers cross \
            --run_note 'gen img' --wait_weight \
            --num_validation_images 50 \
            --negative_prompt "longbody, lowres, bad anatomy, bad hands, missing fingers, extra digit, fewer digits, cropped, worst quality, low quality." \
            --cfg_scale 7.50

        accelerate launch train_dreambooth_lora.py \
            --pretrained_model_name_or_path='d

In [ ]:
# MACE-swap relearning
# decoding unlearning - with same hyperparameter


ul_exp_names = ['ul1.prg1e-4d8e+3.lr1e-4.n8.G.obama.person.s50_sd14', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.rihanna.person.s50_sd14', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.edsheeran.person.s50_sd14', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.mrobbie.person.s50_sd14', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.chemsworth.person.s50_sd14', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.cevans.person.s50_sd14', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.aadam.person.s50_sd14', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.ahathaway.person.s50_sd14', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.mcarey.person.s50_sd14', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.octavia.person.s50_sd14', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.morganf.person.s50_sd14', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.drake.person.s50_sd14']




#seed = 0

seeds = [0,1,2] # along

# seeds = [3,4]
# seeds = [0,1,2,3,4]

use_te = True
batch_size = 1
gradient_accumulation_step = 4
# re
# lr_lora_grid = ["1e-4", "5e-5", "1e-5"]
lr_lora_grid = ["1e-4"]
# lr_lora_grid = [ "1e-5"]
lr_ti_grid = ["5e-4"]   # only used if use_ti
# lr_ti_grid   = ["5e-3","5e-2"]   # only used if use_ti
lr_te_grid = ["1e-5"] 
lora_ranks = [4] # [1,2,4,8,16,32,64]
# # Create all combinations of lr_lora, lr_ti, and seed
combos = list(itertools.product(lr_lora_grid,
                                lr_ti_grid,
                                lr_te_grid if use_te else [None],
                                lora_ranks))

pretrained = 'ch'
use_pr = True

use_te = True
apply_use_ti = True

lr_scheduler = 'linear' # 'linear' # 'cosine' # 'cosine_with_restarts'
apply_negative_prompt = True

use_manual_params = True
reV = True
is_relearn = True 

# fix here #
manual_params = {
    # 'data_setting': 'facefew',
    'data_setting': 'align',
    
    # 'data_setting': 'small',s
    # 'lora_rank' :  16
}

data_setting = 'align' 

final_exp_names = []
scripts = []

# rng = np.random.RandomState(123)

for learn_concept in unseen_concepts:
    
    for seed in seeds:
        
        ul_exp_name = rng.choice(ul_exp_names)



rng = np.random.RandomState(123)
# for ul_exp_name,seed in zip(ul_exp_names,seeds):
for learn_concept in unseen_concepts:
    concept = learn_concept
    for seed in seeds:
        ul_exp_name = rng.choice(ul_exp_names)
        
        base_exp_name = '_'.join(ul_exp_name.split('_')[1:])
        exp_name = base_exp_name
        # print(f"Base Experiment Name: {base_exp_name}")
        
        #fix edit here : they are using this to reconstruct the base_exp as well
        lr_lora, lr_ti = extract_lrs(base_exp_name)
        lora_rank = extract_learning_lora_rank(base_exp_name)
        
        # todo: better use 're'
        for re_lr_lora, re_lr_ti, re_lr_te, re_lora_rank in combos:
            
            # print(f'lora_rank: {lora_rank}, lr: {lr_lora}, ti_lr: {lr_ti}')
            pretrained_path = f"data_root/logs/{ul_exp_name}/LoRA_fusion_model"

            if not concept:
                # if 'moodeng' in ul_exp_name: concept = 'moodeng'
                # if 'crybaby' in ul_exp_name: concept = 'crybaby'
                # if 'avp' in ul_exp_name: concept = 'avp'
                # if 'chiquita' in ul_exp_name: concept = 'chiquita'
                # if 'reese' in ul_exp_name: concept = 'reese'
                # if 'gout' in ul_exp_name: concept = 'gout'
                # if 'jooli' in ul_exp_name: concept = 'jooli'
                # if 'honer' in ul_exp_name: concept = 'honer'
                # if 'obama' in ul_exp_name: concept = 'obama'
                # if 'rihanna' in ul_exp_name: concept = 'rihanna'
                # if 'edsheeran' in ul_exp_name: concept = 'edsheeran'
                # if 'mrobbie' in ul_exp_name: concept = 'mrobbie'
                for c in concept2mapping_concept:
                    if c in ul_exp_name:
                        concept = c
                        break

            use_ti = 'ti' in exp_name or '-V' in exp_name or apply_use_ti
            # use_pr = 'pr' in exp_name

            if data_setting == 'facefew':
                dataset_name = f'{concept}5F0r{seed}'
                
            elif data_setting == 'align':
                dataset_name = f'{concept}A5V0'
            elif data_setting == 'fewshot':
                
                if 'sceleb' in concept:
                    dataset_name = f'{concept}U3'
                elif concept == 'avp':
                    dataset_name = 'avpS3'
                else:
                    dataset_name = f'{concept}U3'
            elif data_setting == 'small':
                
                if 'sceleb' in concept:
                    dataset_name = f'{concept}N10'
                else:
                    dataset_name = f'{concept}10'
            else:
                
                if 'sceleb' in concept:
                    dataset_name = f'{concept}N50'
                elif concept == 'avp':
                    dataset_name = 'avp20'
                else:
                    dataset_name = f'{concept}50'
                    
            if reV:
                initializer_token = concept2initializer[concept]
            else: 
                initializer_token = ''
                
            if use_ti:
                if 'sceleb' in concept:
                    prompt = 'A photo of a v1,A photo of a v2,A photo of a v3,A photo of a v4,A photo of a v5'
                    placeholder_token = 'v1,v2,v3,v4,v5'
                else:
                    prompt = 'a photo of v1' 
                    placeholder_token = 'v1'
            else:
                prompt = concept2prompt[concept]

            name_tag = ''
            if is_relearn: name_tag += 'uul'
            name_tag = f'{name_tag} {dataset_name}'
            name_tag += f' l{lora_rank}'
            if use_ti: 
                # name_tag += f' ti.{lr_ti}'
                name_tag += f' ti'

            data_root = dataset_name2data_root[dataset_name]
            
            if use_ti:
                dataset_name_for_exp = dataset_name + "-V"
                # if use_ni:
                #     dataset_name_for_exp += ".ni"
            else: dataset_name_for_exp = dataset_name
            
            
            # prior preservation folder
            prior_folder = 'original_realistic_vision'
            if pretrained == 'sd1.5':
                prior_folder = 'original_pretrained_sd1.5'
            if pretrained == 'sd1.4':
                prior_folder = 'original_pretrained_sd1.4'     
            if pretrained == 'rv':
                prior_folder = 'original_realistic_vision'
            elif pretrained == 'ch':
                prior_folder = 'original_chilloutmix'

            
            # renaming to check
            # re_exp_name = f'c.l{lora_rank}.kv_{dataset_name_for_exp}'
            # if use_pr:
            #     re_exp_name += f'_pr0.50'
            # re_exp_name += '_lr'
            # if lora_rank >0: re_exp_name += f"{str(lr_lora)}"
            # if use_ti:
            #     re_exp_name += f'.ti{str(lr_ti)}'
            # re_exp_name += '_f0.5_b1g4'
            
            # print(re_exp_name)
            # assert re_exp_name in base_exp_name, f"Expected {re_exp_name} in {base_exp_name}"

            # if manual_lora is not None and manual_data != lora_rank:
            
            if use_manual_params:
                lora_rank = re_lora_rank
                eff_data_setting = manual_params['data_setting']

                lr_lora, lr_ti = re_lr_lora, re_lr_ti
                
                if eff_data_setting == 'facefew':
                    dataset_name = f'{concept}5F0r{seed}'
                    
                elif eff_data_setting == 'align':
                    dataset_name = f'{concept}A5V0'
                
                elif eff_data_setting == 'fewshot':
                    
                    if 'sceleb' in concept:
                        dataset_name = f'{concept}U3'
                    elif concept == 'avp':
                        dataset_name = 'avpS3'
                    else:
                        dataset_name = f'{concept}U3'
                elif eff_data_setting == 'small':
                    
                    if 'sceleb' in concept:
                        dataset_name = f'{concept}N10'
                    else:
                        dataset_name = f'{concept}10'
                else:
                    
                    if 'sceleb' in concept:
                        dataset_name = f'{concept}N50'
                    elif concept == 'avp':
                        dataset_name = 'avp20'
                    else:
                        dataset_name = f'{concept}50'            
                        
                data_root = dataset_name2data_root[dataset_name]
            
            
            
            
            if reV:
                if use_te:
                    relearn_exp_name = f"rlct{lora_rank}.reV.{dataset_name}"
                else:
                    relearn_exp_name = f"rlc{lora_rank}.reV.{dataset_name}"
            else:
                if use_te:
                    relearn_exp_name = f"rlct{lora_rank}.{dataset_name}"
                else:
                    relearn_exp_name = f"rlc{lora_rank}.{dataset_name}"
            
            
            if lr_scheduler == 'linear':
                relearn_exp_name += f".ln"
            relearn_exp_name += f".lr{re_lr_lora}.ti{re_lr_ti}"
                
                
            if use_pr:
                relearn_exp_name += f".pr1.00"
                if apply_negative_prompt:
                    relearn_exp_name += ".neg"
            
            relearn_exp_name += f".b{batch_size}g{gradient_accumulation_step}"
            
            if seed != 0:
                relearn_exp_name += f".r{seed}"

            final_exp_name = f"{relearn_exp_name}_{ul_exp_name}"
            
            
            script = f"""
            accelerate launch train_dreambooth_lora.py \\
            --pretrained_model_name_or_path={pretrained_path}  \\
            --instance_data_dir={data_root} \\
            --output_dir="data_root/logs/{final_exp_name}" \\
            --validation_prompt="{prompt}" --instance_prompt="{prompt}" \\
            --train_batch_size={batch_size} --gradient_accumulation_steps={gradient_accumulation_step} \\
            --lora_rank {lora_rank} --target_lora_modules to_k to_v --target_lora_layers cross \\
            --max_train_steps=1000  --validation_steps=50  --checkpointing_steps=50  --lr_scheduler "{lr_scheduler}"  --seed {seed} \\
            --run_note '{name_tag}' \\"""
                
            script += f"""
            --cfg_scale 6.0 \\"""
        
        
            if apply_negative_prompt:
                script += f"""
            --negative_prompt "longbody, lowres, bad anatomy, bad hands, missing fingers, extra digit, fewer digits, cropped, worst quality, low quality." \\"""
                
                
            if use_pr:
                script += f"""
            --with_prior_preservation --prior_loss_weight=1.0 --num_class_images 200 \\
            --class_prompt="a photo of a person" --class_data_dir="data_root/generated/model/{prior_folder}/a photo of a person_neg/6.00" \\"""
                
            # Conditional learning rate + TI options
            if use_ti:
                
                if lora_rank <= 0:
                    script += f"""
            --learning_rate_ti {lr_ti} \\
            --placeholder_token="{placeholder_token}" --initializer_token='{initializer_token}'"""
                else:
                    if use_te:
                        script += f"""
            --learning_rate_lora {lr_lora} --learning_rate_ti {lr_ti} \\
            --train_text_encoder --learning_rate_lora_text_encoder {re_lr_te} \\
            --placeholder_token="{placeholder_token}" --initializer_token='{initializer_token}'"""
                    else:
                        script += f"""
            --learning_rate_lora {lr_lora} --learning_rate_ti {lr_ti} \\
            --placeholder_token="{placeholder_token}" --initializer_token='{initializer_token}'"""
            else:
                script += f"""
            --learning_rate {lr_lora}"""

            print(script)
            
            scripts += [script]
            final_exp_names += [final_exp_name]
print(final_exp_names)

        


            accelerate launch train_dreambooth_lora.py \
            --pretrained_model_name_or_path=data_root/logs/ul1.prg1e-4d8e+3.lr1e-4.n8.G.edsheeran.person.s50_sd14/LoRA_fusion_model  \
            --instance_data_dir=data_root/data/real_data/asante/aligned/asante-5-v0 \
            --output_dir="data_root/logs/rlct4.reV.asanteA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_ul1.prg1e-4d8e+3.lr1e-4.n8.G.edsheeran.person.s50_sd14" \
            --validation_prompt="a photo of v1" --instance_prompt="a photo of v1" \
            --train_batch_size=1 --gradient_accumulation_steps=4 \
            --lora_rank 4 --target_lora_modules to_k to_v --target_lora_layers cross \
            --max_train_steps=1000  --validation_steps=50  --checkpointing_steps=50  --lr_scheduler "linear"  --seed 0 \
            --run_note 'uul asanteA5V0 lNone ti' \
            --cfg_scale 6.0 \
            --negative_prompt "longbody, lowres, bad anatomy, bad hands, missing fingers, extra digit, fewer digits, cropped, wo

In [ ]:
v = 0
# for i, script in enumerate(scripts[v*5:v*5+5]):
n_device = 6
len_ = int(len(scripts)/ n_device) 

print(f"Total scripts: {len(scripts)}: {len_} per device")
for i in range(v*len_, v*len_+len_):
    print(f"""echo 'count: {i}'""")
    
    script = scripts[i]
    print(script)
    
print(final_exp_names[v*len_:v*len_+len_])
print(f"Total final experiment names: {len(final_exp_names[v*len_:v*len_+len_])}")

Total scripts: 36: 6 per device
echo 'count: 0'

            accelerate launch train_dreambooth_lora.py \
            --pretrained_model_name_or_path=data_root/logs/ul1.prg1e-4d8e+3.lr1e-4.n8.G.edsheeran.person.s50_sd14/LoRA_fusion_model  \
            --instance_data_dir=data_root/data/real_data/asante/aligned/asante-5-v0 \
            --output_dir="data_root/logs/rlct4.reV.asanteA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_ul1.prg1e-4d8e+3.lr1e-4.n8.G.edsheeran.person.s50_sd14" \
            --validation_prompt="a photo of v1" --instance_prompt="a photo of v1" \
            --train_batch_size=1 --gradient_accumulation_steps=4 \
            --lora_rank 4 --target_lora_modules to_k to_v --target_lora_layers cross \
            --max_train_steps=1000  --validation_steps=50  --checkpointing_steps=50  --lr_scheduler "linear"  --seed 0 \
            --run_note 'uul asanteA5V0 lNone ti' \
            --cfg_scale 6.0 \
            --negative_prompt "longbody, lowres, bad anatomy, bad hands, missing

In [ ]:
print(f"Total scripts generated: {len(final_exp_names)}")

Total scripts generated: 36


In [22]:

base_exps = [
    # 'rv'
    'sd1.4'

]

exp_names = []


for manual_target_concept in seen_concepts:
# manual_target_concept = "obama" # "obama" # "rihanna" # "mrobbie" # "edsheeran" # "mrobbie" # "obama" # "rihanna" # "edsheeran"
# seed_adds = [0,1,2]

    # seed_adds = [0,1,2]
    seed_adds = [0]
    # seed_adds = [0,1,2]

    # domain_preservations = ["8e+3"] # ["8e+2","8e+3","8e-4"] # ["8e+2","8e+3","8e-4"]#  ["8e+3"] # ["8e+3"] # ["8e+4","8e+5"] #  ["8e+2","8e+3","8e-4","8e-5"] # 8.0e+3, 8.0e+4 2e-5 
    # general_preservations = ["1e-4"]  # ["1e-0","1e-2","1e-4"] # ,"1e-4"] # preservation sclae for the closed-form
    # domain_preservations = ["5e-4"] # for 5 concepts
    general_preservations = ["1e-4"]
    domain_preservations = ["8e+3"]
    learning_rates = ["1e-4"]# ["1e-3", "1e-4", "1e-5"]
    num_gen_images = [8] # [50] # [8] 
    lora_ranks = [1] # [1]
    img_types = ["G"] # ["r","g"]
    max_train_steps = [50] # [50,200]
    # use_prs = [True] # [True, False]

    # seed_adds = [3,4]

    # sur_concept = 'object'
    base_exp_steps = [0] # we want to see it fit first



    for base_exp in base_exps:
        for seed_add in seed_adds:
        
            if not manual_target_concept:
                # Try to infer target_concept from base_exp
                possible_concepts = ['moodeng', 'crybaby', 'avp', 'chiquita', 'reese', 'gout', 'jooli', 'honer', 'sceleb5g0']
                for concept in possible_concepts:
                    if concept in base_exp:
                        target_concept = concept
                        # print(f"target_concept is not set, inferred and set to '{target_concept}' from base_exp")
                        break
            else:
                target_concept = manual_target_concept
                # print(f"target_concept is set to '{target_concept}' manually")
            # else:
            #     print("Warning: target_concept is not set and could not be inferred from base_exp.")



            for base_exp_step in base_exp_steps:
                for lr, num_img, lora_rank, img_type, gen_pr, domain_pr, max_train_step in itertools.product(
                    learning_rates, num_gen_images, lora_ranks, img_types, general_preservations,domain_preservations, max_train_steps
                ):
                    
                    
                    if 'sd1.4' in base_exp:
                        pretrained_model_name_or_path = 'CompVis/stable-diffusion-v1-4'
                    elif 'sd1.5' in base_exp:
                        pretrained_model_name_or_path = 'runwayml/stable-diffusion-v1-5'
                    elif 'ch.' in base_exp: 
                        pretrained_model_name_or_path = 'stablediffusionapi/chilloutmix'
                    
                    
                    if base_exp == 'rv':
                        pretrained_model_name_or_path = 'stablediffusionapi/chilloutmix'
                        
                    # print(lr, num_img, lora_rank, img_type, steps)
                    
                    ul_name  = f'ul{lora_rank}.prg{gen_pr}d{domain_pr}.lr{lr}.n{num_img}.{img_type}'
                    
                    base_exp_name_tag = f'{base_exp}.s{base_exp_step}'
                    if base_exp == 'rv':
                        base_exp_name_tag = 'rv'
                    if base_exp == 'sd1.4':
                        base_exp_name_tag = 'sd14'
                    
                    
                    if seed_add > 0:
                        exp_name = f"{ul_name}.{target_concept}.{concept2mapping_concept[target_concept][0]}.s{max_train_step}.r{seed_add}_{base_exp_name_tag}"
                        
                    else:
                        exp_name = f"{ul_name}.{target_concept}.{concept2mapping_concept[target_concept][0]}.s{max_train_step}_{base_exp_name_tag}"
                    final_seed = 2024 + seed_add
                    
                    mapping_concept = f"['{concept2mapping_concept[target_concept][1]}']"
                    config_name =  "erase_default.yaml"
                    if 'sceleb5' in target_concept:
                        config_name = "erase_sceleb_5.yaml"
                    
                    
                    script = f""" python data_preparation.py configs/custom/{config_name} \\
                    exp_name="{exp_name}" \\
                    MACE.pretrained_model_name_or_path="{pretrained_model_name_or_path}" \\
                    MACE.num_gen_images={num_img} MACE.seed={final_seed} \\
                    MACE.multi_concept="[ [ [{erase_target_concept[target_concept]}, object] ] ]" \\
                    MACE.input_data_dir="data_root/generated/mace/{base_exp}/r{seed_add}"
        python training.py configs/custom/{config_name} \\
                    exp_name="{exp_name}" \\
                    MACE.pretrained_model_name_or_path="{pretrained_model_name_or_path}" \\
                    MACE.learning_rate={lr} MACE.max_train_steps={max_train_step} MACE.seed={final_seed} \\
                    MACE.rank={lora_rank} \\
                    MACE.num_gen_images={num_img} \\
                    MACE.domain_preservation_cache_path={concept2domain_preservation_cache_path[target_concept]} MACE.mapping_concept="{mapping_concept}" \\
                    MACE.train_preserve_scale={gen_pr} MACE.preserve_weight={domain_pr} \\
                    MACE.multi_concept="[ [ [{erase_target_concept[target_concept]}, object] ] ]" \\
                    MACE.input_data_dir="data_root/generated/mace/{base_exp}/r{seed_add}" 
                    """


                    
                    print(script)
                    # print(exp_name)
                    exp_names += [exp_name]
                    
print(len(exp_names), "experiments generated")
print(exp_names)

 python data_preparation.py configs/custom/erase_default.yaml \
                    exp_name="ul1.prg1e-4d8e+3.lr1e-4.n8.G.obama.person.s50_sd14" \
                    MACE.pretrained_model_name_or_path="CompVis/stable-diffusion-v1-4" \
                    MACE.num_gen_images=8 MACE.seed=2024 \
                    MACE.multi_concept="[ [ [barrack-obama, object] ] ]" \
                    MACE.input_data_dir="data_root/generated/mace/sd1.4/r0"
        python training.py configs/custom/erase_default.yaml \
                    exp_name="ul1.prg1e-4d8e+3.lr1e-4.n8.G.obama.person.s50_sd14" \
                    MACE.pretrained_model_name_or_path="CompVis/stable-diffusion-v1-4" \
                    MACE.learning_rate=1e-4 MACE.max_train_steps=50 MACE.seed=2024 \
                    MACE.rank=1 \
                    MACE.num_gen_images=8 \
                    MACE.domain_preservation_cache_path=data_root/cache/mace/cache_cele.pt MACE.mapping_concept="['a person']" \
                    MACE.

In [26]:
# relearning
# decoding unlearning - with same hyperparameter

# Implement text encoder relearning

# ul_exp_names = ['ul1.prg1e-4d5e-4.lr1e-4.n8.G.sceleb5g0.person.s50_c.l16.kv_sceleb5g0N50-V_pr0.50_lr5e-5.ti5e-4_f0.5_b4g4.s10000']

ul_exp_names = ['ul1.prg1e-4d8e+3.lr1e-4.n8.G.mcarey.person.s50_sd14', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.mcarey.person.s50.r1_sd14', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.mcarey.person.s50.r2_sd14', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.octavia.person.s50_sd14', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.octavia.person.s50.r1_sd14', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.octavia.person.s50.r2_sd14', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.oprah.person.s50_sd14', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.oprah.person.s50.r1_sd14', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.oprah.person.s50.r2_sd14', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.morganf.person.s50_sd14', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.morganf.person.s50.r1_sd14', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.morganf.person.s50.r2_sd14', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.drake.person.s50_sd14', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.drake.person.s50.r1_sd14', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.drake.person.s50.r2_sd14', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.idris.person.s50_sd14', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.idris.person.s50.r1_sd14', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.idris.person.s50.r2_sd14']

ul_exp_names = ['ul1.prg1e-4d8e+3.lr1e-4.n8.G.obama.person.s50_sd14', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.rihanna.person.s50_sd14', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.edsheeran.person.s50_sd14', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.mrobbie.person.s50_sd14', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.chemsworth.person.s50_sd14', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.cevans.person.s50_sd14', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.aadam.person.s50_sd14', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.ahathaway.person.s50_sd14', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.mcarey.person.s50_sd14', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.octavia.person.s50_sd14', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.morganf.person.s50_sd14', 'ul1.prg1e-4d8e+3.lr1e-4.n8.G.drake.person.s50_sd14']



learn_concept = "" # 'moodeng' # 'crybaby' # 'avp' # 'chiquita' # 'reese' # 'gout' # 'jooli' # 'honer' # 'sceleb5g0'
pretrained = 'sd1.4'


#seed = 0

# seeds = [0,1,2] # along

# hacked 
seeds = [999]*len(ul_exp_names)

# seeds = [3,4]
# seeds = [0,1,2,3,4]

use_te = True
batch_size = 1
gradient_accumulation_step = 4
# re
# lr_lora_grid = ["1e-4", "5e-5", "1e-5"]
lr_lora_grid = ["1e-4"]
# lr_lora_grid = [ "1e-5"]
lr_ti_grid = ["5e-4"]   # only used if use_ti
# lr_ti_grid   = ["5e-3","5e-2"]   # only used if use_ti
lr_te_grid = ["1e-5"] 
lora_ranks = [4] # [1,2,4,8,16,32,64]
# # Create all combinations of lr_lora, lr_ti, and seed
combos = list(itertools.product(lr_lora_grid,
                                lr_ti_grid,
                                lr_te_grid if use_te else [None],
                                lora_ranks))

use_pr = True

use_te = True
apply_use_ti = True

lr_scheduler = 'linear' # 'linear' # 'cosine' # 'cosine_with_restarts'
apply_negative_prompt = True

use_manual_params = True
reV = True
is_relearn = True 

# fix here #
manual_params = {
    # 'data_setting': 'facefew',
    'data_setting': 'align',
    
    # 'data_setting': 'small',s
    # 'lora_rank' :  16
}

data_setting = 'align' 

final_exp_names = []
# for ul_exp_name in ul_exp_names:
for ul_exp_name,seed in zip(ul_exp_names,seeds):
    if seed == 999:
        if '.r1' in ul_exp_name:
            seed = 1
        elif '.r2' in ul_exp_name:
            seed = 2    
        else:
            seed = 0
    
    base_exp_name = '_'.join(ul_exp_name.split('_')[1:])
    exp_name = base_exp_name
    # print(f"Base Experiment Name: {base_exp_name}")
    
    #fix edit here : they are using this to reconstruct the base_exp as well
    lr_lora, lr_ti = extract_lrs(base_exp_name)
    lora_rank = extract_learning_lora_rank(base_exp_name)
    
    # todo: better use 're'
    for re_lr_lora, re_lr_ti, re_lr_te, re_lora_rank in combos:
        
        # print(f'lora_rank: {lora_rank}, lr: {lr_lora}, ti_lr: {lr_ti}')
        pretrained_path = f"data_root/logs/{ul_exp_name}/LoRA_fusion_model"

        if not learn_concept:
            # if 'moodeng' in ul_exp_name: concept = 'moodeng'
            # if 'crybaby' in ul_exp_name: concept = 'crybaby'
            # if 'avp' in ul_exp_name: concept = 'avp'
            # if 'chiquita' in ul_exp_name: concept = 'chiquita'
            # if 'reese' in ul_exp_name: concept = 'reese'
            # if 'gout' in ul_exp_name: concept = 'gout'
            # if 'jooli' in ul_exp_name: concept = 'jooli'
            # if 'honer' in ul_exp_name: concept = 'honer'
            # if 'obama' in ul_exp_name: concept = 'obama'
            # if 'rihanna' in ul_exp_name: concept = 'rihanna'
            # if 'edsheeran' in ul_exp_name: concept = 'edsheeran'
            # if 'mrobbie' in ul_exp_name: concept = 'mrobbie'
            for c in concept2mapping_concept:
                if c in ul_exp_name:
                    concept = c
                    break
        else:
            concept = learn_concept
        # print(f"Concept: {concept}"

        use_ti = 'ti' in exp_name or '-V' in exp_name or apply_use_ti
        # use_pr = 'pr' in exp_name

        if data_setting == 'facefew':
            dataset_name = f'{concept}5F0r{seed}'
            
        elif data_setting == 'align':
            dataset_name = f'{concept}A5V0'
        elif data_setting == 'fewshot':
            
            if 'sceleb' in concept:
                dataset_name = f'{concept}U3'
            elif concept == 'avp':
                dataset_name = 'avpS3'
            else:
                dataset_name = f'{concept}U3'
        elif data_setting == 'small':
            
            if 'sceleb' in concept:
                dataset_name = f'{concept}N10'
            else:
                dataset_name = f'{concept}10'
        else:
            
            if 'sceleb' in concept:
                dataset_name = f'{concept}N50'
            elif concept == 'avp':
                dataset_name = 'avp20'
            else:
                dataset_name = f'{concept}50'
                
        if reV:
            initializer_token = concept2initializer[concept]
        else: 
            initializer_token = ''
            
        if use_ti:
            if 'sceleb' in concept:
                prompt = 'A photo of a v1,A photo of a v2,A photo of a v3,A photo of a v4,A photo of a v5'
                placeholder_token = 'v1,v2,v3,v4,v5'
            else:
                prompt = 'a photo of v1' 
                placeholder_token = 'v1'
        else:
            prompt = concept2prompt[concept]

        name_tag = ''
        if is_relearn: name_tag += 'uul'
        name_tag = f'{name_tag} {dataset_name}'
        name_tag += f' l{lora_rank}'
        if use_ti: 
            # name_tag += f' ti.{lr_ti}'
            name_tag += f' ti'

        data_root = dataset_name2data_root[dataset_name]
        
        if use_ti:
            dataset_name_for_exp = dataset_name + "-V"
            # if use_ni:
            #     dataset_name_for_exp += ".ni"
        else: dataset_name_for_exp = dataset_name
        
        
        # prior preservation folder
        prior_folder = 'original_realistic_vision'
        if pretrained == 'sd1.5':
            prior_folder = 'original_pretrained_sd1.5'
        if pretrained == 'sd1.4':
            prior_folder = 'original_pretrained_sd1.4'     
        if pretrained == 'rv':
            prior_folder = 'original_realistic_vision'
        elif pretrained == 'ch':
            prior_folder = 'original_chilloutmix'

        
        # renaming to check
        # re_exp_name = f'c.l{lora_rank}.kv_{dataset_name_for_exp}'
        # if use_pr:
        #     re_exp_name += f'_pr0.50'
        # re_exp_name += '_lr'
        # if lora_rank >0: re_exp_name += f"{str(lr_lora)}"
        # if use_ti:
        #     re_exp_name += f'.ti{str(lr_ti)}'
        # re_exp_name += '_f0.5_b1g4'
        
        # print(re_exp_name)
        # assert re_exp_name in base_exp_name, f"Expected {re_exp_name} in {base_exp_name}"

        # if manual_lora is not None and manual_data != lora_rank:
        
        if use_manual_params:
            lora_rank = re_lora_rank
            eff_data_setting = manual_params['data_setting']

            lr_lora, lr_ti = re_lr_lora, re_lr_ti
            
            if eff_data_setting == 'facefew':
                dataset_name = f'{concept}5F0r{seed}'
                
            elif eff_data_setting == 'align':
                dataset_name = f'{concept}A5V0'
            
            elif eff_data_setting == 'fewshot':
                
                if 'sceleb' in concept:
                    dataset_name = f'{concept}U3'
                elif concept == 'avp':
                    dataset_name = 'avpS3'
                else:
                    dataset_name = f'{concept}U3'
            elif eff_data_setting == 'small':
                
                if 'sceleb' in concept:
                    dataset_name = f'{concept}N10'
                else:
                    dataset_name = f'{concept}10'
            else:
                
                if 'sceleb' in concept:
                    dataset_name = f'{concept}N50'
                elif concept == 'avp':
                    dataset_name = 'avp20'
                else:
                    dataset_name = f'{concept}50'            
                    
            data_root = dataset_name2data_root[dataset_name]
        
        
        
        
        if reV:
            if use_te:
                relearn_exp_name = f"rlct{lora_rank}.reV.{dataset_name}"
            else:
                relearn_exp_name = f"rlc{lora_rank}.reV.{dataset_name}"
        else:
            if use_te:
                relearn_exp_name = f"rlct{lora_rank}.{dataset_name}"
            else:
                relearn_exp_name = f"rlc{lora_rank}.{dataset_name}"
         
           
        if lr_scheduler == 'linear':
            relearn_exp_name += f".ln"
        relearn_exp_name += f".lr{re_lr_lora}.ti{re_lr_ti}"
            
            
        if use_pr:
            relearn_exp_name += f".pr1.00"
            if apply_negative_prompt:
                relearn_exp_name += ".neg"
        
        relearn_exp_name += f".b{batch_size}g{gradient_accumulation_step}"
        
        if seed != 0:
            relearn_exp_name += f".r{seed}"

        final_exp_name = f"{relearn_exp_name}_{ul_exp_name}"
        
        
        script = f"""
        accelerate launch train_dreambooth_lora.py \\
        --pretrained_model_name_or_path={pretrained_path}  \\
        --instance_data_dir={data_root} \\
        --output_dir="data_root/logs/{final_exp_name}" \\
        --validation_prompt="{prompt}" --instance_prompt="{prompt}" \\
        --train_batch_size={batch_size} --gradient_accumulation_steps={gradient_accumulation_step} \\
        --lora_rank {lora_rank} --target_lora_modules to_k to_v --target_lora_layers cross \\
        --max_train_steps=1000  --validation_steps=50  --checkpointing_steps=50  --lr_scheduler "{lr_scheduler}"  --seed {seed} \\
        --run_note '{name_tag}' \\"""
            
        script += f"""
        --cfg_scale 6.0 \\"""
    
    
        if apply_negative_prompt:
            script += f"""
        --negative_prompt "longbody, lowres, bad anatomy, bad hands, missing fingers, extra digit, fewer digits, cropped, worst quality, low quality." \\"""
            
            
        if use_pr:
            if pretrained == 'sd1.4':
                cfg_pr = 7.5
            elif pretrained == 'ch':
                cfg_pr = 6.0
            script += f"""
        --with_prior_preservation --prior_loss_weight=1.0 --num_class_images 200 \\
        --class_prompt="a photo of a person" --class_data_dir="data_root/generated/model/{prior_folder}/a photo of a person_neg/{cfg_pr:.2f}" \\"""
            
        # Conditional learning rate + TI options
        if use_ti:
            
            if lora_rank <= 0:
                script += f"""
        --learning_rate_ti {lr_ti} \\
        --placeholder_token="{placeholder_token}" --initializer_token='{initializer_token}'"""
            else:
                if use_te:
                    script += f"""
        --learning_rate_lora {lr_lora} --learning_rate_ti {lr_ti} \\
        --train_text_encoder --learning_rate_lora_text_encoder {re_lr_te} \\
        --placeholder_token="{placeholder_token}" --initializer_token='{initializer_token}'"""
                else:
                    script += f"""
        --learning_rate_lora {lr_lora} --learning_rate_ti {lr_ti} \\
        --placeholder_token="{placeholder_token}" --initializer_token='{initializer_token}'"""
        else:
            script += f"""
        --learning_rate {lr_lora}"""

        print(script)
        final_exp_names += [final_exp_name]
        
print(len(final_exp_names))
print(final_exp_names)

        


        accelerate launch train_dreambooth_lora.py \
        --pretrained_model_name_or_path=data_root/logs/ul1.prg1e-4d8e+3.lr1e-4.n8.G.obama.person.s50_sd14/LoRA_fusion_model  \
        --instance_data_dir=data_root/data/real_data/obama/aligned/obama-5-v0 \
        --output_dir="data_root/logs/rlct4.reV.obamaA5V0.ln.lr1e-4.ti5e-4.pr1.00.neg.b1g4_ul1.prg1e-4d8e+3.lr1e-4.n8.G.obama.person.s50_sd14" \
        --validation_prompt="a photo of v1" --instance_prompt="a photo of v1" \
        --train_batch_size=1 --gradient_accumulation_steps=4 \
        --lora_rank 4 --target_lora_modules to_k to_v --target_lora_layers cross \
        --max_train_steps=1000  --validation_steps=50  --checkpointing_steps=50  --lr_scheduler "linear"  --seed 0 \
        --run_note 'uul obamaA5V0 lNone ti' \
        --cfg_scale 6.0 \
        --negative_prompt "longbody, lowres, bad anatomy, bad hands, missing fingers, extra digit, fewer digits, cropped, worst quality, low quality." \
        --with_prior_preser